# Explicit-cue dependence in text emotion recognition: end-to-end experiment notebook

One notebook runs the whole study. Set `STAGE` in Cell 1 and run all cells top to bottom (Colab or Kaggle, GPU).
Rerun with the same `STAGE` after any disconnect: completed runs are skipped, interrupted runs resume.

**Before the first run**
1. Add two secrets: `HF_TOKEN` (write access) and `HF_USERNAME` (Colab: key icon; Kaggle: Add-ons, Secrets).
2. Deposit `docs/PREREGISTRATION.md` with a timestamp, and deposit Amendment 1 (crowd-enVENT analyses) before any `cev_*` run.
3. Deposit Amendment 2 (enISEAR hypotheses P1 and P2, Cell 9e) with a timestamp before any `eni_*` run and before the `llm_transfer` rerun that adds enISEAR.

**Stage order** (core = needed for the main claims; supp = supplementary, run only if compute allows).
Run the core stages in the order listed; `cev_*` stages come after `shortcut`.

| Stage | Tier | What it runs | Runs |
|---|---|---|---|
| `audit` | core | corpus audit: TF-IDF baselines, "feel" probe, duplicates, explicit shares (CPU, minutes) | 1 |
| `tuning` | core | 5 encoders x 4 learning rates x 2 epoch budgets x {no prompt, template A} x 3 tuning seeds; selects one recipe per encoder | 240 |
| `main` | core | 5 encoders x 7 prompt conditions x 10 seeds on SetFit/emotion, with CRI, calibration, and transfer to 4 corpora | 350 |
| `intra` | core | TweetEval and EmpatheticDialogues, 5 encoders x {none, A} x 5 seeds | 100 |
| `target` | core | target-trained upper bounds on MELD, ISEAR, GoEmotions: 5 encoders x {none, A} x 5 seeds | 150 |
| `shortcut` | core | 2 encoders x 3 conditions x {cue mask, random mask} x {train only, train and test} x 5 seeds | 120 |
| `cev_source` | core | crowd-enVENT analyses: 5 encoders x {none, A} x 5 seeds trained on SetFit/emotion; scored on 2,750 crowd-enVENT texts and their masked copies (emotion names replaced by the corpus creators) | 50 |
| `cev_masked` | core | crowd-enVENT analyses: RoBERTa, DistilRoBERTa x {cue mask, random mask} training x 5 seeds (within-encoder CRI variation) | 20 |
| `cev_target` | core | crowd-enVENT upper bound and control: 5 encoders x 5 seeds trained on 2,025 writer-labelled texts, tested on the 500 reader-annotated texts | 25 |
| `cev_third` | core | third-corpus control for E1 and second source corpus: 5 encoders x 5 seeds trained on GoEmotions (no prompt); scored on the crowd-enVENT original and masked texts and transferred to MELD, ISEAR, ED | 25 |
| `llm_transfer` | core | source-free control: zero-shot GPT-4o (gpt-4o-2024-08-06) on the 4 transfer targets and on crowd-enVENT original and masked texts; target-restricted label lists, strict parser (API, no GPU, about 8,300 calls) | 0 (API) |
| `lr_sweep` | supp | DeBERTa-v3 and ModernBERT (FP32): 4 learning rates x {none, A} x 3 seeds | 48 |
| `convergence` | supp | DeBERTa-v3 and ModernBERT, {none, A} x 20 seeds (needs `lr_sweep`) | 80 |
| `llm` | supp | GPT-4o, Llama-3.1-8B, Ministral-8B x {0, 6 demonstrations} on the full test set | 6 |
| `peft` | supp | DistilRoBERTa LoRA (r = 4, 8, 16, 32) and two soft-prompt variants x 5 seeds | 30 |
| `rebalance` | supp | DistilRoBERTa and ELECTRA x {weighted loss, oversampling} x 5 seeds | 20 |
| `placement` | supp | prompt as suffix or separate segment, 2 encoders x 5 seeds | 20 |
| `scaling` | supp | 32k / 64k / 128k training texts from the leakage-filtered unsplit pool | 27 |
| `prompt_map` | supp | few-shot k in {16, 64, 256, 512} x 4 conditions x 2 heads x 2 encoders x 5 seeds | 320 |
| `prompt_map_full` | supp | verbalizer head with the full training set, 2 encoders x 4 conditions x 5 seeds | 40 |
| `rerun` | superseded | not needed: the `cev_*` stages repeat 70 main/shortcut configurations and provide the run-to-run variation analysis (E5) | (8) |
| `eni_source` | core | enISEAR (v8): the `cev_source` configurations (5 encoders x {none, A} x 5 seeds) retrained on SetFit/emotion; export enISEAR logits | 50 |
| `eni_third` | core | enISEAR (v8): the `cev_third` configurations retrained on GoEmotions; export enISEAR logits | 25 |
| `analysis` | - | aggregation, hypothesis tests, new controls X1-X4 (Cell 9d), reader perspective and enISEAR (Cell 9e), tables (`analysis/tables/*.tex`), figures only | 0 |

Core stages: 1,056 runs before `cev_third` (25 runs) and the enISEAR stages (75 runs). Supplementary: 591 runs.
Only seed-42 models of `main` and `shortcut` are saved (`SAVE_FINAL_MODEL_FOR`), so the enISEAR stages retrain the crowd-enVENT configurations instead of reloading them. On a T4, one SetFit/emotion run took about 8.5 minutes and one GoEmotions run about 1 minute, so `eni_source` needs about 7 GPU-hours and `eni_third` about 0.5.

**enISEAR** (v8: stages `eni_source` and `eni_third`; GPT-4o as task `enisear` of `llm_transfer`)
- The corpus (Troiano, Pado & Klinger 2019) is downloaded and checked against its SHA-256 in Cell 6; it is released under ODC-By 1.0 and contains only the versions the readers saw (emotion word replaced by an ellipsis).
- The release writes the ellipsis as `...` in most descriptions and as the single character `…` in 166 Fear and Anger descriptions (every Joy and Sadness description uses `...`); Cell 6 normalizes every `…` to `...` so that the marker does not differ by class.
- GPT-4o: rerun `STAGE = "llm_transfer"`. Finished chunks are skipped, so only the 572 enISEAR descriptions are sent (3 chunks); the manifest keeps the entries of the earlier tasks.
- Cell 9e computes the reader-perspective analyses for crowd-enVENT (exploratory) and enISEAR (P1, P2) and writes `analysis/tables/perspective.tex`, `analysis/tables/supp_enisear.tex`, and the paper values in `analysis/metrics/perspective_numbers.json`. Measure the time of the first `tuning` runs before committing GPU budget.
The `llm` stage additionally needs the secret `OPENAI_API_KEY` and accepted Llama-3.1 and Ministral licences on the `HF_TOKEN` account.

Stages after `tuning` read the tuned recipes from the Hub automatically; they abort with a clear message if tuning is not complete
(`convergence` also needs `lr_sweep`). Every stage ends with Cells 9b to 11, so the analysis on the Hub is always current.

**crowd-enVENT stages** (`cev_source`, `cev_masked`, `cev_target`)
- The corpus (Troiano, Oberländer & Klinger 2023) is downloaded and checked automatically in Cell 6; it has no explicit licence
  (citation request only), so confirm terms before publication.
- These stages repeat some main/shortcut configurations because finished runs did not export crowd-enVENT outputs. They are used
  only for the crowd-enVENT analyses (E1 to E3 confirmatory; E4, E5 exploratory) and never enter the prompt-effect statistics.
- Run them on the same GPU type as `main`, so the rerun comparison (E5) measures run-to-run variation rather than hardware differences.

**Storage**
- Mid-run checkpoints stay local (`UPLOAD_MIDRUN_CHECKPOINTS = False`): deleted Hub files still count against the storage quota.
- Final models are uploaded only for the runs selected in `SAVE_FINAL_MODEL_FOR`.

**Outputs on the Hub** (`{HF_USERNAME}/emotion_explicit_cue`, private dataset repo): per run `metadata/run_metadata.json`,
`outputs/logits.npz`, `metrics/metrics.json`, `DONE`; shared cue vocabulary, data manifest, and tuned recipes; `analysis/metrics/*.json`
(including `cev_E1` to `cev_E5`), `analysis/metrics/*.csv`, `analysis/tables/*.tex`, and `analysis/figures/*.png`.

## Cell 1 — Configuration

In [25]:
# =====================================================================================
# CELL 1 — CONFIGURATION
# Every tunable parameter of the study lives here. No other cell defines parameters.
# Workflow: set STAGE, run all cells top to bottom; rerun the notebook (same STAGE) after a
# disconnect, and it resumes (completed runs are skipped, interrupted runs resume from latest.json).
# Stage order: audit -> tuning -> lr_sweep -> main -> intra -> target -> shortcut -> llm -> peft -> rebalance ->
# placement -> convergence -> scaling -> prompt_map; "analysis" only re-aggregates what is already on the Hub.
# v8 (reader-perspective revision): eni_source -> eni_third -> llm_transfer (adds enISEAR); deposit Amendment 2 first.
# =====================================================================================
import logging, os, random, sys, uuid, json
from datetime import datetime

# logging.basicConfig(level=logging.INFO, format='[%(asctime)s] [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
# logger = logging.getLogger("emotion_cue")
logger = logging.getLogger("emotion_cue")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(logging.Formatter('[%(asctime)s] [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
logger.addHandler(_handler)
logger.propagate = False

EXPERIMENT_NAME = "emotion_explicit_cue"
# STAGE = "audit"                       # see STAGE_GRIDS below and the notebook introduction for the order
STAGE = "llm_transfer"
MODEL_NAME = "multi_encoder"          # run-directory component; the stage grid defines the actual encoders
RUN_ID = uuid.uuid4().hex[:6]         # identifies this notebook session
SESSION_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- reproducibility ---------------------------------------------------------------
SEED = 42                             # session-level seed (data subsampling, random prefix); each run seeds itself
SEED_LIST = [42, 2024, 7, 100, 999, 1, 2, 3, 4, 5]   # evaluation seeds (main grid uses all ten)
TUNING_SEEDS = [11, 12, 13]           # disjoint from evaluation seeds
CONVERGENCE_SEEDS = list(range(101, 121))
MULTISEED_MODE = True

# ---- training ----------------------------------------------------------------------
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 2e-5                  # default only; tuned per encoder in the tuning stage
NUM_EPOCHS = 5                        # default only; tuned per encoder
MAX_SEQ_LEN = 128
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 2               # epochs without validation-loss improvement
PRECISION = "auto"                    # auto -> bf16 on compute capability >= 8.0, fp16 otherwise; "fp32" forces fp32
FEWSHOT_MIN_STEPS = 300               # few-shot runs never train for fewer optimizer steps than this
CHECKPOINT_EVERY_N_STEPS = 500
UPLOAD_MIDRUN_CHECKPOINTS = "auto"    # "auto": upload mid-run checkpoints only for runs longer than the threshold below
AUTO_UPLOAD_MIN_STEPS = 5000
REQUIRE_GPU = True
LOG_EVERY_N_STEPS = 100

# ---- encoders ------------------------------------------------------------------------
ENCODERS = {
    "distilbert":    {"hf_id": "distilbert-base-uncased"},
    "distilroberta": {"hf_id": "distilroberta-base"},
    "roberta":       {"hf_id": "roberta-base"},
    "albert":        {"hf_id": "albert-base-v2"},
    "electra":       {"hf_id": "google/electra-base-discriminator"},
    "deberta":       {"hf_id": "microsoft/deberta-v3-base", "precision": "fp32", "lr": 1e-5, "epochs": 5},
    "modernbert":    {"hf_id": "answerdotai/ModernBERT-base", "precision": "fp32", "lr": 1e-5, "epochs": 5},
}
MAIN_ENCODERS = ["distilbert", "distilroberta", "roberta", "albert", "electra"]
TUNING_GRID = {"lr": [1e-5, 2e-5, 3e-5, 5e-5], "epochs": [5, 10]}
# RESOLUTION: tuning covers BOTH no prompt and template A and selects by their mean validation accuracy. Tuning on
# the no-prompt condition alone would favour that condition and bias the prompt comparison toward a null result.
TUNING_CONDITIONS = ["none", "A"]
LR_SWEEP_GRID = {"lr": [5e-6, 1e-5, 2e-5, 3e-5], "epochs": [5]}     # DeBERTa-v3 / ModernBERT (FP32)

# ---- parameter-efficient adaptation (v6 settings) ------------------------------------------
PEFT_METHODS = {   # method -> text-prompt condition used with it (as in the v6 experiments)
    "lora_r4": "A", "lora_r8": "A", "lora_r16": "A", "lora_r32": "A", "softprompt_frozen": "none", "softprompt_full": "none"}
LORA_LR = 2e-4
LORA_ALPHA = 16                       # v6 used alpha = 16 for every rank (cells 34 and 35)
LORA_DROPOUT = 0.1
LORA_TARGETS = {"roberta": ["query", "value"], "bert": ["query", "value"], "electra": ["query", "value"], "albert": ["query", "value"],
                "distilbert": ["q_lin", "v_lin"], "deberta-v2": ["query_proj", "value_proj"], "modernbert": ["Wqkv"]}
SOFT_PROMPT_TOKENS = 20
SOFT_PROMPT_LR_FROZEN = 1e-3          # v6: backbone frozen, prompt vectors and head trained
SOFT_PROMPT_LR_FULL = 2e-5            # v6: prompt vectors trained together with the backbone

# ---- class imbalance ---------------------------------------------------------------------------
REBALANCE_METHODS = ["weighted", "oversample"]   # weighted: inverse-frequency class weights; oversample: every class to the majority count

# ---- generative LLM baselines --------------------------------------------------------------------
LLM_MODELS = {"llama31_8b": {"backend": "hf", "hf_id": "meta-llama/Llama-3.1-8B-Instruct"},
              "ministral_8b": {"backend": "hf", "hf_id": "mistralai/Ministral-8B-Instruct-2410"},
              "gpt4o": {"backend": "openai", "api_model": "gpt-4o-2024-08-06"}}   # dated snapshot, recorded per response
LLM_SHOTS = [0, 6]                    # 6 = one demonstration per class from the training split
LLM_DEMO_SEED = 42
LLM_MAX_NEW_TOKENS = 12
LLM_LOAD_4BIT = True                  # NF4 on GPU; ignored on CPU
LLM_CHUNK = 250                       # items per resumable chunk
LLM_SYSTEM = ("You classify the emotion of a text. Answer with exactly one word from this list: "
              "sadness, joy, love, anger, fear, surprise.")

# ---- calibration ---------------------------------------------------------------------------------
CALIBRATION_BINS = 15
TEMPERATURE_BOUNDS = (0.05, 20.0)

# ---- prompts ---------------------------------------------------------------------------
PROMPT_TEMPLATES = {
    "none": "",
    "A": "The emotion of the following sentence is:",
    "B": "Classify the emotion expressed:",
    "C": "This text expresses the following emotion:",
    "D": "What emotion does this text express?",
    "misleading": "The emotion of the following sentence is joy:",
    "random": None,                   # random tokens, same token count as template A, built per tokenizer
}
RANDOM_PREFIX_SEED = 1234
VERBALIZER_CANDIDATES = {"sadness": ["sad", "sadness"], "joy": ["joy", "happy"], "love": ["love"],
                         "anger": ["anger", "angry"], "fear": ["fear", "afraid"], "surprise": ["surprise", "surprised"]}

# ---- explicit-cue analysis (frozen by docs/PREREGISTRATION.md) ---------------------------
CUE_MIN_COUNT = 5
CUE_MIN_PURITY = 0.6
EXPECTED_CUE_VOCAB_SIZE = 358         # value from the pre-registered pilot; Cell 6 aborts if the data changed
INTENSIFIERS = ("so very really a bit little kind of quite too more pretty extremely rather somewhat "
                "completely totally incredibly slightly just like").split()
CONVERGED_VAL_ACC = 0.90
N_BOOTSTRAP = 5000
N_PERMUTATIONS_MAX_EXACT = 16

# ---- data ------------------------------------------------------------------------------
TRANSFER_TARGETS = ["goemotions", "meld", "isear", "ed"]
TARGET_CORPORA = ["meld_target", "isear_target", "goemotions_target"]    # target-trained upper bounds (ED: intra stage)
ISEAR_SPLIT_SEED = 42
CORPUS_OVERRIDES = {"meld_target": {"epochs": 10, "patience": 3}}   # v6 trained the MELD upper bound this way (paper Table A-II)                 # stratified 70/15/15, reproduces the v6 split (3,030 / 649 / 650)
SCALING_SIZES = [32000, 64000, 128000]
SCALING_POOL_SEED = 0
# FEWSHOT_KS = [16, 64, 256, 1024]
FEWSHOT_KS = [16, 64, 256, 512]       # 512 = largest balanced size (surprise has 572 training texts)
ISEAR_HF_REPO = "gsri-18/ISEAR-dataset-complete"
# ISEAR integrity: the converted parquet file is pinned by its SHA-256 in shared/data_manifest.json (Cell 6).

# ---- crowd-enVENT (Troiano, Oberlaender & Klinger 2023, Computational Linguistics 49(1)) -----------------
CEV_URL = "https://www.romanklinger.de/data-sets/crowd-enVent2023.zip"
CEV_TO_SETFIT = {"sadness": 0, "joy": 1, "anger": 3, "fear": 4, "surprise": 5}   # the five classes shared with SetFit
CEV_APPRAISALS = ["suddenness", "familiarity", "predict_event", "pleasantness", "unpleasantness", "goal_relevance",
                  "chance_responsblt", "self_responsblt", "other_responsblt", "predict_conseq", "goal_support", "urgency",
                  "self_control", "other_control", "chance_control", "accept_conseq", "standards", "social_norms",
                  "attention", "not_consider", "effort"]
CEV_SPLIT_SEED = 42
CEV_VAL_FRACTION = 0.1
CEV_STAGES = ["cev_source", "cev_masked"]       # SetFit-trained stages that export crowd-enVENT logits
CEV_EXPORT_STAGES = CEV_STAGES + ["cev_third"]   # stages that export crowd-enVENT logits; cev_third is NOT a SetFit source

# ---- enISEAR (Troiano, Pado & Klinger 2019, ACL; ODC-By 1.0) -- v8, reader-perspective revision ------------------------
# Writers completed "I felt [emotion] when/because ..." for a prompted emotion; five other workers read the description
# with the emotion word omitted and chose, among seven emotions, the one the writer most likely felt. The release contains
# only the versions the readers saw. Evaluation only: stages eni_source, eni_third, task enisear of llm_transfer, Cell 9e.
ENI_URL = "https://www.ims.uni-stuttgart.de/documents/ressourcen/korpora/emotions/deISEARenISEAR.zip"
ENI_SHA256 = "38aa785caaf668652d9278f583990802d7eccf7569d0a206cabb49f9bbb89861"   # archive as downloaded on 2026-10-25
ENI_EMOTIONS = ["Anger", "Disgust", "Fear", "Guilt", "Joy", "Sadness", "Shame"]       # reader vote columns of enISEAR.tsv
ENI_TO_SETFIT = {"Sadness": 0, "Joy": 1, "Anger": 3, "Fear": 4}                       # the four classes shared with SetFit
# RESOLUTION: the omitted emotion word is marked "..." in 407 and "\u2026" (one character) in 166 of the 572 four-class
# descriptions (one has both), and the second form occurs only in Fear and Anger descriptions. A marker that differs by
# class would be an artifact, so every "\u2026" is replaced by "..." (the marker of crowd-enVENT) before any model
# sees the text.
ENI_MARKER = "..."
ENI_EXPORT_STAGES = ["eni_source", "eni_third"]          # trained stages that export enISEAR logits
P_N_BOOT = 2000                                          # bootstrap resamples, reader-perspective analyses (Cell 9e)
P_SEED = 20261025

# ---- storage -----------------------------------------------------------------------------
WORK_DIRS = {"colab": "/content/work", "kaggle": "/kaggle/working/work"}
HF_REPO_PRIVATE = True
SAVE_FINAL_MODEL_FOR = {"stages": ["main", "shortcut"], "seeds": [42], "conditions": ["none", "A", "misleading"]}

# ---- stage grids ---------------------------------------------------------------------------
# Each stage expands to a list of run specifications in Cell 5. "transfer": evaluate on TRANSFER_TARGETS.
STAGE_GRIDS = {
    # core: needed for the main claims | supp: supplementary analyses (run only if compute allows)
    "audit":       {"kind": "audit", "tier": "core"},
    "tuning":      {"kind": "train", "tier": "core", "corpus": ["setfit"], "encoder": MAIN_ENCODERS, "condition": TUNING_CONDITIONS,
                    "seed": TUNING_SEEDS, "lr": TUNING_GRID["lr"], "epochs": TUNING_GRID["epochs"], "transfer": False, "recipe": "grid"},
    "lr_sweep":    {"kind": "train", "tier": "supp", "corpus": ["setfit"], "encoder": ["deberta", "modernbert"], "condition": TUNING_CONDITIONS,
                    "seed": TUNING_SEEDS, "lr": LR_SWEEP_GRID["lr"], "epochs": LR_SWEEP_GRID["epochs"], "transfer": False, "recipe": "grid"},
    "main":        {"kind": "train", "tier": "core", "corpus": ["setfit"], "encoder": MAIN_ENCODERS,
                    "condition": ["none", "A", "B", "C", "D", "random", "misleading"], "seed": SEED_LIST, "transfer": True, "recipe": "tuned"},
    "intra":       {"kind": "train", "tier": "core", "corpus": ["tweeteval", "ed"], "encoder": MAIN_ENCODERS, "condition": ["none", "A"],
                    "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
    "target":      {"kind": "train", "tier": "core", "corpus": TARGET_CORPORA, "encoder": MAIN_ENCODERS, "condition": ["none", "A"],
                    "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
    "shortcut":    {"kind": "train", "tier": "core", "corpus": ["setfit"], "encoder": ["roberta", "distilroberta"], "condition": ["none", "A", "misleading"],
                    "text_transform": ["mask_feel", "mask_random"], "transform_eval": [False, True], "seed": SEED_LIST[:5],
                    "transfer": True, "recipe": "tuned"},
    "llm":         {"kind": "llm", "tier": "supp", "model": list(LLM_MODELS), "shots": LLM_SHOTS},
    "peft":        {"kind": "train", "tier": "supp", "corpus": ["setfit"], "encoder": ["distilroberta"], "peft_method": list(PEFT_METHODS),
                    "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
    "rebalance":   {"kind": "train", "tier": "supp", "corpus": ["setfit"], "encoder": ["distilroberta", "electra"], "condition": ["A"],
                    "rebalance": REBALANCE_METHODS, "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
    "placement":   {"kind": "train", "tier": "supp", "corpus": ["setfit"], "encoder": ["roberta", "distilbert"], "condition": ["A"],
                    "placement": ["suffix", "pair"], "seed": SEED_LIST[:5], "transfer": True, "recipe": "tuned"},
    "convergence": {"kind": "train", "tier": "supp", "corpus": ["setfit"], "encoder": ["deberta", "modernbert"], "condition": ["none", "A"],
                    "seed": CONVERGENCE_SEEDS, "transfer": False, "recipe": "tuned"},
    "scaling":     {"kind": "train", "tier": "supp", "corpus": [f"setfit_large_{n}" for n in SCALING_SIZES], "encoder": ["distilroberta"],
                    "condition": ["none", "A", "misleading"], "seed": SEED_LIST[:3], "transfer": True, "recipe": "tuned", "epochs_override": 3},
    "prompt_map":  {"kind": "train", "tier": "supp", "corpus": ["setfit"], "encoder": ["roberta", "distilroberta"],
                    "condition": ["none", "A", "random", "misleading"], "head": ["cls", "verbalizer"], "fewshot_k": FEWSHOT_KS,
                    "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
    # same configuration and seed trained twice: measures run-to-run variation from GPU nondeterminism (paper RQ3)
    "rerun":       {"kind": "train", "tier": "core", "corpus": ["setfit"], "encoder": ["distilroberta", "roberta", "albert", "electra"],
                    "condition": ["A"], "seed": [42], "repeat": [1, 2], "transfer": False, "recipe": "tuned"},
    # verbalizer head with the full training set (the "Full" row of the data-regime table)
    "prompt_map_full": {"kind": "train", "tier": "supp", "corpus": ["setfit"], "encoder": ["roberta", "distilroberta"],
                    "condition": ["none", "A", "random", "misleading"], "head": ["verbalizer"], "seed": SEED_LIST[:5],
                    "transfer": False, "recipe": "tuned"},
    "analysis":    None,
    # crowd-enVENT analyses: these repeat main/shortcut configurations because finished runs did not export crowd-enVENT
    # logits; used ONLY for the crowd-enVENT analyses and rerun variation, never in the prompt-effect statistics.
    "cev_source":  {"kind": "train", "tier": "core", "corpus": ["setfit"], "encoder": MAIN_ENCODERS, "condition": ["none", "A"],
                    "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
    "cev_masked":  {"kind": "train", "tier": "core", "corpus": ["setfit"], "encoder": ["roberta", "distilroberta"], "condition": ["none"],
                    "text_transform": ["mask_feel", "mask_random"], "transform_eval": [False], "seed": SEED_LIST[:5],
                    "transfer": False, "recipe": "tuned"},
    "cev_target":  {"kind": "train", "tier": "core", "corpus": ["cev_target"], "encoder": MAIN_ENCODERS, "condition": ["none"],
                    "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
    # third-corpus control for E1 and second source corpus for transfer (GoEmotions-trained; exports crowd-enVENT logits)
    "cev_third":   {"kind": "train", "tier": "core", "corpus": ["goemotions_target"], "encoder": MAIN_ENCODERS, "condition": ["none"],
                    "seed": SEED_LIST[:5], "transfer": True, "recipe": "tuned"},
    # zero-shot GPT-4o on the transfer targets, crowd-enVENT, and (v8) enISEAR (Cell 8c; no training)
    "llm_transfer": None,
    # v8: enISEAR. Only seed-42 models were saved (SAVE_FINAL_MODEL_FOR), so the configurations of cev_source and cev_third are
    # retrained with the same recipes and seeds and export enISEAR logits; they never enter the prompt-effect statistics.
    "eni_source":  {"kind": "train", "tier": "core", "corpus": ["setfit"], "encoder": MAIN_ENCODERS, "condition": ["none", "A"],
                    "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
    "eni_third":   {"kind": "train", "tier": "core", "corpus": ["goemotions_target"], "encoder": MAIN_ENCODERS, "condition": ["none"],
                    "seed": SEED_LIST[:5], "transfer": False, "recipe": "tuned"},
}
RECIPE_SELECTION_STAGES = {"tuning": MAIN_ENCODERS, "lr_sweep": ["deberta", "modernbert"]}
VALID_STAGES = list(STAGE_GRIDS)
assert STAGE in VALID_STAGES, f"STAGE must be one of {VALID_STAGES}, got {STAGE!r}"

# ---- reproducibility helper (GodMode standard) --------------------------------------------
import numpy as np
import torch

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
# CONFIG_SNAPSHOT = {k: v for k, v in dict(globals()).items()
#                    if k.isupper() and isinstance(v, (int, float, str, bool, list, dict, type(None)))}
import re
_NOT_CONFIG = {"SECRETS", "API", "HUB_FILES", "DATA", "DATA_FILES", "EXPLICIT", "CUE_VOCAB", "SPECS", "RUN_STATUS",
               "TUNED_RECIPES", "SESSION_METADATA", "ANALYSIS", "METRICS_DF", "FIGURES", "FAILED_RUNS", "LIB_VERSIONS",
               "CONFIG_SNAPSHOT", "HF_REPO_ID", "WORK_DIR", "DEVICE", "ENV", "CEV", "CEV_DATA", "ENI", "PERSPECTIVE", "X_RESULTS"}
_SECRET_RE = re.compile(r"(hf_[A-Za-z0-9]{20,}|sk-[A-Za-z0-9_\-]{20,})")

def _redact(o):
    """Config snapshots never carry credentials, even if a secret ends up in a global by mistake."""
    if isinstance(o, str):
        return _SECRET_RE.sub("<REDACTED>", o)
    if isinstance(o, dict):
        return {k: _redact(v) for k, v in o.items()}
    if isinstance(o, list):
        return [_redact(v) for v in o]
    return o

CONFIG_SNAPSHOT = _redact({k: v for k, v in dict(globals()).items()
                           if k.isupper() and k not in _NOT_CONFIG
                           and isinstance(v, (int, float, str, bool, list, dict, type(None)))})
assert not _SECRET_RE.search(json.dumps(CONFIG_SNAPSHOT, default=str)), "credential found in config snapshot"
logger.info(f"Config ready: experiment={EXPERIMENT_NAME} stage={STAGE} session={SESSION_TIMESTAMP}_{RUN_ID}")
logger.info(f"Seed set to {SEED}")


[2026-10-07 12:07:08] [INFO] Config ready: experiment=emotion_explicit_cue stage=llm_transfer session=20261007_120708_588316
[2026-10-07 12:07:08] [INFO] Seed set to 42


## Cell 2 — Environment detection and secret loading

In [26]:
# =====================================================================================
# CELL 2 — ENVIRONMENT DETECTION AND SECRET LOADING
# Required secrets: HF_TOKEN (write access), HF_USERNAME; OPENAI_API_KEY only for STAGE='llm'.
# Colab: key icon; Kaggle: Add-ons -> Secrets. Llama-3.1 is gated: accept its licence on the Hub with the HF_TOKEN account.
# =====================================================================================
import importlib.util

def detect_environment() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"):
        return "kaggle"
    elif os.environ.get("COLAB_RELEASE_TAG") or importlib.util.find_spec("google.colab") is not None:
        return "colab"
    else:
        raise RuntimeError("Unsupported environment. This notebook must run in Google Colab or Kaggle. "
                           "Local and unknown environments are not supported.")

def load_secrets(env: str) -> dict:
    required_keys = ["HF_TOKEN", "HF_USERNAME"]
    if (STAGE == "llm" and any(m["backend"] == "openai" for m in LLM_MODELS.values())) or STAGE == "llm_transfer":
        required_keys.append("OPENAI_API_KEY")          # only the LLM stage calls the OpenAI API
    secrets = {}
    if env == "colab":
        from google.colab import userdata
        getter = userdata.get
        where = "Colab Secrets (key icon in the left panel)"
    else:
        from kaggle_secrets import UserSecretsClient
        getter = UserSecretsClient().get_secret
        where = "Kaggle -> Add-ons -> Secrets"
    for key in required_keys:
        try:
            val = getter(key)
        except Exception as e:                      # platform raises if the secret does not exist
            raise ValueError(f"Required secret '{key}' is missing or not granted to this notebook. Add it via {where}.") from e
        if not val:
            raise ValueError(f"Required secret '{key}' is empty. Add it via {where}.")
        secrets[key] = val
    logger.info(f"Secrets loaded: {list(secrets.keys())}")      # keys only, never values
    return secrets

def setup_environment() -> tuple:
    env = detect_environment()
    logger.info(f"Environment detected: {env}")
    assert sys.version_info >= (3, 10), f"Python >= 3.10 required, found {sys.version.split()[0]}"
    secrets = load_secrets(env)
    os.environ["HF_TOKEN"] = secrets["HF_TOKEN"]
    if REQUIRE_GPU and not torch.cuda.is_available():
        raise RuntimeError("GPU required but not available. Enable a GPU runtime (REQUIRE_GPU=True in Cell 1).")
    elif not torch.cuda.is_available():
        logger.warning("GPU not available. Running on CPU.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Device: {device}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        logger.info(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB | "
                    f"compute capability {props.major}.{props.minor}")
    work_dir = WORK_DIRS[env]
    os.makedirs(work_dir, exist_ok=True)
    return env, secrets, device, work_dir

ENV, SECRETS, DEVICE, WORK_DIR = setup_environment()


[2026-10-07 12:07:08] [INFO] Environment detected: kaggle
[2026-10-07 12:07:08] [INFO] Secrets loaded: ['HF_TOKEN', 'HF_USERNAME', 'OPENAI_API_KEY']
[2026-10-07 12:07:08] [INFO] Device: cuda
[2026-10-07 12:07:08] [INFO] GPU: Tesla T4 | VRAM: 15.6 GB | compute capability 7.5


## Cell 3 — Dependency installation

In [27]:
# =====================================================================================
# CELL 3 — DEPENDENCY INSTALLATION
# Pinned minimum versions. If an already-imported package gets upgraded, the runtime must restart.
# =====================================================================================
import importlib.metadata, subprocess

REQUIRED = {
    "torch": "2.0.0", "transformers": "4.48.0", "datasets": "2.14.0", "huggingface_hub": "0.20.0",
    "matplotlib": "3.7.0", "numpy": "1.24.0", "scikit-learn": "1.3.0", "accelerate": "0.24.0", "pyyaml": "6.0",
    # study-specific additions
    "pandas": "2.0.0", "pyarrow": "12.0.0", "scipy": "1.10.0", "statsmodels": "0.14.0",
    "sentencepiece": "0.1.99", "protobuf": "3.20.0", "safetensors": "0.4.0", "packaging": "23.0", "peft": "0.10.0",
}
if STAGE == "llm":
    REQUIRED["openai"] = "1.40.0"
    if torch.cuda.is_available() and LLM_LOAD_4BIT:
        REQUIRED["bitsandbytes"] = "0.43.0"
if STAGE == "llm_transfer":
    REQUIRED["openai"] = "1.40.0"
# ASSUMPTION: transformers >= 4.48 is required because ModernBERT (convergence stage) was added in 4.48.
IMPORT_NAMES = {"scikit-learn": "sklearn", "pyyaml": "yaml", "huggingface_hub": "huggingface_hub", "protobuf": "google.protobuf"}

def installed_version(pkg: str):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None

def install_dependencies() -> None:
    from packaging import version as V
    before = {p: installed_version(p) for p in REQUIRED}
    need = [f"{p}>={v}" for p, v in REQUIRED.items() if before[p] is None or V.parse(before[p]) < V.parse(v)]
    if need:
        logger.info(f"Installing/upgrading: {need}")
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as e:
            raise RuntimeError(f"pip install failed: {e.stderr[-1500:]}") from e
    else:
        logger.info("All required packages already satisfy the minimum versions.")

        # peft refuses LoRA while an old torchao (< 0.16) is installed; the notebook never uses torchao, so remove it
    _tao = installed_version("torchao")
    if _tao is not None and V.parse(_tao) < V.parse("0.16.0"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
        logger.info(f"Removed torchao {_tao}: peft needs torchao >= 0.16 or none; this notebook does not use torchao")
    after = {p: installed_version(p) for p in REQUIRED}
    for pkg, min_ver in REQUIRED.items():
        assert after[pkg] is not None, f"{pkg} is not installed after pip install. Aborting."
        assert V.parse(after[pkg]) >= V.parse(min_ver), f"{pkg} version {after[pkg]} is below required {min_ver}. Aborting."
    changed = [p for p in REQUIRED if before[p] is not None and before[p] != after[p]
               and IMPORT_NAMES.get(p, p).split(".")[0] in sys.modules]
    if changed:
        raise RuntimeError(f"Upgraded packages that were already imported: {changed}. "
                           "Restart the runtime (Runtime -> Restart session) and run the notebook again from Cell 1.")
    for pkg in REQUIRED:
        __import__(IMPORT_NAMES.get(pkg, pkg).replace("-", "_"))
    logger.info("Library versions: " + ", ".join(f"{p}={after[p]}" for p in REQUIRED))
    return after

LIB_VERSIONS = install_dependencies()


[2026-10-07 12:07:08] [INFO] All required packages already satisfy the minimum versions.
[2026-10-07 12:07:08] [INFO] Library versions: torch=2.10.0+cu128, transformers=5.0.0, datasets=5.0.0, huggingface_hub=1.11.0, matplotlib=3.10.0, numpy=2.0.2, scikit-learn=1.6.1, accelerate=1.13.0, pyyaml=6.0.3, pandas=2.3.3, pyarrow=24.0.0, scipy=1.16.3, statsmodels=0.14.6, sentencepiece=0.2.1, protobuf=5.29.5, safetensors=0.7.0, packaging=26.1, peft=0.19.1, openai=2.32.0


## Cell 4 — Storage and repository initialization

In [28]:
# =====================================================================================
# CELL 4 — STORAGE AND REPOSITORY INITIALIZATION
# Hub layout (private dataset repo {HF_USERNAME}/{EXPERIMENT_NAME}):
#   {EXPERIMENT_NAME}/{STAGE}/runs/{spec_id}/{checkpoints,logs,metrics,figures,outputs,metadata}/  + DONE
#   {EXPERIMENT_NAME}/{STAGE}/sessions/{timestamp}_{RUN_ID}/...                                     session log/metadata
#   {EXPERIMENT_NAME}/shared/        cue vocabulary, data manifest, tuned recipes
#   {EXPERIMENT_NAME}/analysis/      aggregated metrics, tables, figures
# RESOLUTION: GodMode names a run directory {timestamp}_{run_id}. A timestamped directory cannot be found again
# by a new session, which breaks resumption across Colab disconnects. Runs are therefore stored under a
# deterministic spec_id (hash of the run specification); the timestamped directory is kept for the session.
# RESOLUTION: every run bundles its artifacts into one folder upload (one Hub commit) instead of one commit per
# file, to stay within Hub commit-rate limits over hundreds of runs.
# =====================================================================================
import json, shutil, time
from pathlib import Path
from huggingface_hub import HfApi
import numpy as np
import torch

SUBDIRS = ["checkpoints", "logs", "metrics", "figures", "outputs", "metadata"]
LOCAL_ROOT = Path(WORK_DIR) / "hub_mirror"
STAGE_PREFIX = f"{EXPERIMENT_NAME}/{STAGE}"
SHARED_PREFIX = f"{EXPERIMENT_NAME}/shared"
ANALYSIS_PREFIX = f"{EXPERIMENT_NAME}/analysis"
SESSION_PREFIX = f"{STAGE_PREFIX}/sessions/{SESSION_TIMESTAMP}_{RUN_ID}"

def upload_with_retry(upload_fn, max_retries: int = 3, base_delay: int = 5):
    for attempt in range(max_retries):
        try:
            return upload_fn()
        except Exception as e:
            wait = base_delay * (2 ** attempt)
            logger.warning(f"Upload attempt {attempt + 1}/{max_retries} failed: {e}. Retrying in {wait}s.")
            time.sleep(wait)
    logger.error("All upload attempts failed. Local artifact preserved. Manual upload required.")
    raise RuntimeError("Persistent upload failure. Aborting to preserve state.")

def local_path(path_in_repo: str) -> Path:
    p = LOCAL_ROOT / path_in_repo
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

def hub_upload_file(local: Path, path_in_repo: str) -> None:
    assert Path(local).exists(), f"cannot upload missing file {local}"
    upload_with_retry(lambda: API.upload_file(path_or_fileobj=str(local), path_in_repo=path_in_repo,
                                              repo_id=HF_REPO_ID, repo_type="dataset"))
    HUB_FILES.add(path_in_repo)
    logger.debug(f"uploaded {path_in_repo}")

def hub_upload_folder(local_dir: Path, path_in_repo: str, message: str) -> None:
    assert Path(local_dir).is_dir(), f"cannot upload missing folder {local_dir}"
    upload_with_retry(lambda: API.upload_folder(folder_path=str(local_dir), path_in_repo=path_in_repo, repo_id=HF_REPO_ID,
                                                repo_type="dataset", commit_message=message))
    for f in Path(local_dir).rglob("*"):
        if f.is_file():
            HUB_FILES.add(f"{path_in_repo}/{f.relative_to(local_dir).as_posix()}")

def hub_download(path_in_repo: str):
    """Return a local path for a Hub file, or None if it does not exist on the Hub."""
    if path_in_repo not in HUB_FILES:
        return None
    try:
        p = API.hf_hub_download(repo_id=HF_REPO_ID, filename=path_in_repo, repo_type="dataset", local_dir=str(LOCAL_ROOT))
        return Path(p)
    except Exception as e:
        logger.error(f"Download failed for {path_in_repo}: {e}")
        raise

def write_json(path: Path, obj) -> None:
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.tolist() if isinstance(o, (np.ndarray, torch.Tensor)) else (o.item() if hasattr(o, "item") else (list(o) if isinstance(o, (set, tuple)) else str(o))))
    shutil.move(str(tmp), str(path))            # atomic rename on POSIX

def refresh_hub_listing() -> set:
    try:
        return set(API.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset"))
    except Exception as e:
        logger.error(f"Could not list Hub repository {HF_REPO_ID}: {e}")
        raise

def initialize_storage():
    api = HfApi(token=SECRETS["HF_TOKEN"])
    repo_id = f"{SECRETS['HF_USERNAME']}/{EXPERIMENT_NAME}"
    upload_with_retry(lambda: api.create_repo(repo_id=repo_id, repo_type="dataset", private=HF_REPO_PRIVATE, exist_ok=True))
    logger.info(f"Repository confirmed: {repo_id}")
    return api, repo_id

API, HF_REPO_ID = initialize_storage()
HUB_FILES = refresh_hub_listing()
logger.info(f"Hub repository holds {len(HUB_FILES)} files")

session_dir = LOCAL_ROOT / SESSION_PREFIX
for sub in SUBDIRS:
    (session_dir / sub).mkdir(parents=True, exist_ok=True)
    (session_dir / sub / ".gitkeep").touch()
SESSION_METADATA = {"run_id": RUN_ID, "experiment_name": EXPERIMENT_NAME, "model_name": MODEL_NAME, "stage": STAGE,
                    "timestamp_start": datetime.now().isoformat(), "timestamp_end": None, "status": "running",
                    "config": CONFIG_SNAPSHOT, "library_versions": LIB_VERSIONS, "environment": ENV,
                    "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                    "latest_checkpoint_step": None, "completed": False}
write_json(session_dir / "metadata" / "run_metadata.json", SESSION_METADATA)
hub_upload_folder(session_dir, SESSION_PREFIX, f"session {SESSION_TIMESTAMP}_{RUN_ID} start ({STAGE})")
logger.info(f"Session storage initialized at {HF_REPO_ID}/{SESSION_PREFIX}")

[2026-10-07 12:07:09] [INFO] Repository confirmed: hacky1997/emotion_explicit_cue
[2026-10-07 12:07:21] [INFO] Hub repository holds 17557 files
[2026-10-07 12:07:22] [INFO] Session storage initialized at hacky1997/emotion_explicit_cue/emotion_explicit_cue/llm_transfer/sessions/20261007_120708_588316


## Cell 5 — Checkpoint restoration

In [29]:
# =====================================================================================
# CELL 5 — CHECKPOINT RESTORATION
# Expands the STAGE grid into run specifications and determines, per run, whether it is complete (DONE on the
# Hub), resumable (latest.json on the Hub), or new. Checkpoint files are downloaded lazily in Cell 8.
# =====================================================================================
import hashlib, itertools

TUNED_RECIPES_PATH = f"{SHARED_PREFIX}/tuned_recipes.json"

def load_tuned_recipes():
    p = hub_download(TUNED_RECIPES_PATH)
    if p is None:
        return None
    with open(p) as f:
        return json.load(f)

def _with_id(spec: dict) -> dict:
    spec["spec_id"] = hashlib.sha1(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:12]
    return spec

def expand_stage(stage: str, tuned) -> list:
    grid = STAGE_GRIDS[stage]
    if grid is None:
        return []
    if grid["kind"] == "audit":
        return [_with_id({"stage": stage, "kind": "audit"})]
    axes = {k: v for k, v in grid.items() if isinstance(v, list)}
    keys = list(axes)
    specs = []
    for combo in itertools.product(*[axes[k] for k in keys]):
        s = dict(zip(keys, combo))
        if grid["kind"] == "llm":
            specs.append(_with_id({"stage": stage, "kind": "llm", "model": s["model"], "shots": int(s["shots"]), **LLM_MODELS[s["model"]]}))
            continue
        enc = ENCODERS[s["encoder"]]
        peft_method = s.get("peft_method")
        condition = PEFT_METHODS[peft_method] if peft_method else s["condition"]
        spec = {"stage": stage, "kind": "train", "corpus": s["corpus"], "encoder": s["encoder"], "hf_id": enc["hf_id"],
                "condition": condition, "placement": s.get("placement", "prefix"), "head": s.get("head", "cls"),
                "text_transform": s.get("text_transform"), "transform_eval": bool(s.get("transform_eval", False)),
                "fewshot_k": s.get("fewshot_k"), "peft_method": peft_method, "rebalance": s.get("rebalance"),
                "seed": int(s["seed"]), "transfer": bool(grid["transfer"]), "precision": enc.get("precision", PRECISION)}
        if "repeat" in s:
            spec["repeat"] = int(s["repeat"])            # identical configuration trained again (distinct spec_id)
        if grid["recipe"] == "grid":
            spec["lr"], spec["epochs"] = float(s["lr"]), int(s["epochs"])
        else:  # tuned
            if tuned is None or s["encoder"] not in tuned:
                src = next(st for st, encs in RECIPE_SELECTION_STAGES.items() if s["encoder"] in encs)
                raise RuntimeError(f"Stage '{stage}' needs the tuned recipe for '{s['encoder']}', but {TUNED_RECIPES_PATH} "
                                   f"does not contain it. Run STAGE='{src}' to completion first (Cells 1-11).")
            spec["lr"], spec["epochs"] = float(tuned[s["encoder"]]["lr"]), int(tuned[s["encoder"]]["epochs"])
        if peft_method and peft_method.startswith("lora"):
            spec["lr"] = LORA_LR
        elif peft_method == "softprompt_frozen":
            spec["lr"] = SOFT_PROMPT_LR_FROZEN
        elif peft_method == "softprompt_full":
            spec["lr"] = SOFT_PROMPT_LR_FULL
        if grid.get("epochs_override"):
            spec["epochs"] = int(grid["epochs_override"])
        ov = CORPUS_OVERRIDES.get(spec["corpus"], {})
        spec["epochs"] = int(ov.get("epochs", spec["epochs"]))
        spec["patience"] = int(ov.get("patience", EARLY_STOP_PATIENCE))
        if spec["condition"] == "none" and spec["placement"] != "prefix":
            continue                                    # placement is meaningless without a prompt
        specs.append(_with_id(spec))
    assert len({s["spec_id"] for s in specs}) == len(specs), "duplicate run specifications in the grid"
    return specs

def run_prefix(spec: dict) -> str:
    return f"{EXPERIMENT_NAME}/{spec['stage']}/runs/{spec['spec_id']}"

def restore_checkpoint_status(specs: list) -> dict:
    status = {}
    for s in specs:
        rp = run_prefix(s)
        if f"{rp}/DONE" in HUB_FILES:
            status[s["spec_id"]] = "done"
        elif f"{rp}/checkpoints/latest.json" in HUB_FILES:
            status[s["spec_id"]] = "resumable"
        else:
            status[s["spec_id"]] = "new"
    return status

TUNED_RECIPES = load_tuned_recipes()
SPECS = expand_stage(STAGE, TUNED_RECIPES) if STAGE != "analysis" else []
RUN_STATUS = restore_checkpoint_status(SPECS)
counts = {k: sum(v == k for v in RUN_STATUS.values()) for k in ("done", "resumable", "new")}
logger.info(f"Stage '{STAGE}': {len(SPECS)} runs | done {counts['done']} | resumable {counts['resumable']} | new {counts['new']}")
if TUNED_RECIPES:
    logger.info(f"Tuned recipes: {TUNED_RECIPES}")


[2026-10-07 12:07:23] [INFO] Stage 'llm_transfer': 0 runs | done 0 | resumable 0 | new 0
[2026-10-07 12:07:23] [INFO] Tuned recipes: {'albert': {'lr': 3e-05, 'epochs': 5, 'val_acc_mean': 0.94, 'val_acc_sd': 0.0020976176963403213, 'n_runs': 6, 'selected_in': 'tuning'}, 'distilbert': {'lr': 5e-05, 'epochs': 5, 'val_acc_mean': 0.9391666666666666, 'val_acc_sd': 0.002016597794967222, 'n_runs': 6, 'selected_in': 'tuning'}, 'distilroberta': {'lr': 3e-05, 'epochs': 10, 'val_acc_mean': 0.94075, 'val_acc_sd': 0.004309872387901966, 'n_runs': 6, 'selected_in': 'tuning'}, 'electra': {'lr': 5e-05, 'epochs': 10, 'val_acc_mean': 0.9406666666666667, 'val_acc_sd': 0.002250925735484593, 'n_runs': 6, 'selected_in': 'tuning'}, 'roberta': {'lr': 3e-05, 'epochs': 10, 'val_acc_mean': 0.94, 'val_acc_sd': 0.0028982753492378813, 'n_runs': 6, 'selected_in': 'tuning'}, 'deberta': {'lr': 3e-05, 'epochs': 5, 'val_acc_mean': 0.9209999999999999, 'val_acc_sd': 0.006284902544988291, 'n_runs': 6, 'selected_in': 'lr_sweep

## Cell 6 — Data loading and validation

In [30]:
# =====================================================================================
# CELL 6 — DATA LOADING AND VALIDATION
# All corpora are downloaded from their public sources, checksummed, and validated against the counts
# reported in the paper. The first session writes shared/data_manifest.json; later sessions abort if any
# file checksum differs (a silent dataset change would invalidate comparisons across sessions).
# =====================================================================================
import re, tarfile, urllib.request
from collections import Counter, defaultdict
import pandas as pd

DATA_DIR = Path(WORK_DIR) / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
HF_PARQUET = "https://huggingface.co/datasets/{repo}/resolve/{rev}/{path}"
MELD_CSV = "https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/{split}_sent_emo.csv"
ED_TGZ = "https://dl.fbaipublicfiles.com/parlai/empatheticdialogues/empatheticdialogues.tar.gz"

SETFIT = ["sadness", "joy", "love", "anger", "fear", "surprise"]
SETFIT2ID = {n: i for i, n in enumerate(SETFIT)}
GOEMOTIONS = ["admiration", "amusement", "anger", "annoyance", "approval", "caring", "confusion", "curiosity", "desire",
              "disappointment", "disapproval", "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief", "joy",
              "love", "nervousness", "optimism", "pride", "realization", "relief", "remorse", "sadness", "surprise", "neutral"]
GOEMOTIONS_TO_SETFIT = {n: SETFIT2ID[n] for n in SETFIT}
MELD_TO_SETFIT = {"sadness": 0, "joy": 1, "anger": 3, "fear": 4, "surprise": 5}
ISEAR_TO_SETFIT = {"sadness": 0, "joy": 1, "anger": 3, "fear": 4}
ED_TO_SETFIT = {**{k: 1 for k in ["joyful", "excited", "content", "hopeful", "grateful", "proud", "impressed", "anticipating"]},
                **{k: 0 for k in ["sad", "devastated", "ashamed", "guilty", "lonely", "nostalgic", "disappointed", "sentimental"]},
                **{k: 3 for k in ["angry", "furious", "jealous", "disgusted", "annoyed"]},
                **{k: 4 for k in ["afraid", "apprehensive", "terrified", "anxious"]},
                **{k: 5 for k in ["surprised", "astonished"]}}
PRESENT_CLASSES = {"setfit": list(range(6)), "goemotions": list(range(6)), "meld": [0, 1, 3, 4, 5], "isear": [0, 1, 3, 4], "ed": [0, 1, 3, 4, 5]}
NUM_LABELS = {"setfit": 6, "tweeteval": 4, "ed": 6, "meld_target": 6, "isear_target": 6, "goemotions_target": 6}
# ED and the target-trained corpora keep SetFit label ids (absent classes are never seen); TweetEval has its own 4 labels
EXPECTED = {  # counts verified against the public data and reported in the paper
    "setfit": {"train": 16000, "validation": 2000, "test": 2000, "test_class_counts": [581, 695, 159, 275, 224, 66]},
    "tweeteval": {"train": 3257, "validation": 374, "test": 1421},
    "ed": {"train": 16343, "validation": 2284, "test": 2120, "test_class_counts": {0: 614, 1: 676, 3: 410, 4: 293, 5: 127}},
    "goemotions": 638, "goemotions_train": 5272, "goemotions_validation": 625, "meld": 1286, "meld_train": 5008, "meld_validation": 617,
    "isear": 4329, "isear_rows": 7516, "isear_split": (3030, 649, 650),
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def fetch(url: str, name: str) -> Path:
    p = DATA_DIR / name
    if p.exists() and p.stat().st_size > 0:
        return p
    try:
        tmp = Path(str(p) + ".part")
        urllib.request.urlretrieve(url, tmp)
        shutil.move(str(tmp), str(p))
    except Exception as e:
        raise RuntimeError(f"Download failed for {url}: {e}") from e
    assert p.stat().st_size > 0, f"empty download: {url}"
    return p

def parquet(repo: str, cfg: str, split: str, name: str, rev: str = "refs%2Fconvert%2Fparquet") -> pd.DataFrame:
    return pd.read_parquet(fetch(HF_PARQUET.format(repo=repo, rev=rev, path=f"{cfg}/{split}/0000.parquet"), name))

def load_data() -> dict:
    D, files = {}, {}
    # SetFit/emotion (source) --------------------------------------------------------------
    D["setfit"] = {}
    for split in ["train", "validation", "test"]:
        df = parquet("SetFit/emotion", "default", split, f"setfit_{split}.parquet")
        for i, t in df.groupby("label").label_text.first().items():          # label order check against the data itself
            assert SETFIT[int(i)] == t, f"SetFit label order mismatch: id {i} is '{t}', expected '{SETFIT[int(i)]}'"
        assert df.text.notna().all() and df.label.notna().all(), f"nulls in SetFit {split}"
        D["setfit"][split] = {"text": df.text.tolist(), "label": df.label.astype(int).tolist()}
        files[f"setfit_{split}.parquet"] = sha256_file(DATA_DIR / f"setfit_{split}.parquet")
        assert len(df) == EXPECTED["setfit"][split], f"SetFit {split}: {len(df)} rows"
    c = Counter(D["setfit"]["test"]["label"])
    assert [c[i] for i in range(6)] == EXPECTED["setfit"]["test_class_counts"], f"SetFit test class counts {c}"
    # TweetEval emotion ------------------------------------------------------------------
    D["tweeteval"] = {}
    for split in ["train", "validation", "test"]:
        df = parquet("cardiffnlp/tweet_eval", "emotion", split, f"tweeteval_{split}.parquet")
        assert len(df) == EXPECTED["tweeteval"][split], f"TweetEval {split}: {len(df)} rows"
        assert set(df.label.unique()) <= {0, 1, 2, 3}
        D["tweeteval"][split] = {"text": df.text.tolist(), "label": df.label.astype(int).tolist()}
        files[f"tweeteval_{split}.parquet"] = sha256_file(DATA_DIR / f"tweeteval_{split}.parquet")
    # GoEmotions: single-label test comments whose label is one of the six source classes ---------
    D["goemotions"] = {}
    for split, key in [("test", "goemotions"), ("train", "goemotions_train"), ("validation", "goemotions_validation")]:
        df = parquet("google-research-datasets/go_emotions", "simplified", split, f"goemo_{split}.parquet")
        keep = df[df.labels.map(len) == 1]
        names = keep.labels.map(lambda l: GOEMOTIONS[int(l[0])])
        keep = keep[names.isin(GOEMOTIONS_TO_SETFIT)]
        D["goemotions"][split] = {"text": keep.text.tolist(), "label": [GOEMOTIONS_TO_SETFIT[GOEMOTIONS[int(l[0])]] for l in keep.labels]}
        assert len(keep) == EXPECTED[key], f"GoEmotions strict {split}: {len(keep)}"
        files[f"goemo_{split}.parquet"] = sha256_file(DATA_DIR / f"goemo_{split}.parquet")
    # MELD: text of utterances with one of the five shared labels --------------------------------
    D["meld"] = {}
    for split, fname in [("train", "train"), ("validation", "dev"), ("test", "test")]:
        p = fetch(MELD_CSV.format(split=fname), f"meld_{fname}.csv")
        m = pd.read_csv(p, encoding_errors="replace")
        m = m[m.Emotion.str.lower().isin(MELD_TO_SETFIT)]
        D["meld"][split] = {"text": m.Utterance.astype(str).tolist(), "label": [MELD_TO_SETFIT[e.lower()] for e in m.Emotion]}
        files[f"meld_{fname}.csv"] = sha256_file(p)
    assert len(D["meld"]["test"]["text"]) == EXPECTED["meld"] and len(D["meld"]["train"]["text"]) == EXPECTED["meld_train"]
    assert len(D["meld"]["validation"]["text"]) == EXPECTED["meld_validation"]
    # ISEAR (Hugging Face copy, pinned revision; no licence statement in that repository) ---------
    # WARNING: the gsri-18 repository provides no dataset card or licence; confirm terms of use before release.
    isr = parquet(ISEAR_HF_REPO, "default", "train", "isear_gsri18.parquet")
    assert len(isr) == EXPECTED["isear_rows"], f"ISEAR rows: {len(isr)}"
    isr = isr[isr.emotion.str.lower().str.strip().isin(ISEAR_TO_SETFIT)]
    D["isear"] = {"test": {"text": isr.content.astype(str).tolist(), "label": [ISEAR_TO_SETFIT[e.lower().strip()] for e in isr.emotion]}}
    assert len(isr) == EXPECTED["isear"], f"ISEAR four-class items: {len(isr)}"
    from sklearn.model_selection import train_test_split          # stratified 70/15/15 split for target training
    tr_df, tmp = train_test_split(isr, test_size=0.3, stratify=isr.emotion, random_state=ISEAR_SPLIT_SEED)
    va_df, te_df = train_test_split(tmp, test_size=0.5, stratify=tmp.emotion, random_state=ISEAR_SPLIT_SEED)
    assert (len(tr_df), len(va_df), len(te_df)) == EXPECTED["isear_split"], f"ISEAR split {(len(tr_df), len(va_df), len(te_df))}"
    D["isear_ft"] = {n: {"text": d.content.astype(str).tolist(), "label": [ISEAR_TO_SETFIT[e.lower().strip()] for e in d.emotion]}
                     for n, d in [("train", tr_df), ("validation", va_df), ("test", te_df)]}
    D["isear_ft"]["test"]["pos_in_full"] = isr.index.get_indexer(te_df.index).tolist()   # rows of the 4,329-item transfer set
    assert min(D["isear_ft"]["test"]["pos_in_full"]) >= 0 and len(set(D["isear_ft"]["test"]["pos_in_full"])) == len(te_df)
    files["isear_gsri18.parquet"] = sha256_file(DATA_DIR / "isear_gsri18.parquet")
    # EmpatheticDialogues: situation description, one item per conversation, official splits -------
    p = fetch(ED_TGZ, "empatheticdialogues.tar.gz")
    files["empatheticdialogues.tar.gz"] = sha256_file(p)
    D["ed"] = {}
    with tarfile.open(p) as tf:
        for split, fname in [("train", "train.csv"), ("validation", "valid.csv"), ("test", "test.csv")]:
            member = next(m for m in tf.getmembers() if m.name.endswith(fname))
            lines = tf.extractfile(member).read().decode("utf-8").splitlines()
            head = lines[0].split(",")
            ic, ie, ip = head.index("conv_id"), head.index("context"), head.index("prompt")
            seen, text, lab = set(), [], []
            for line in lines[1:]:
                f = line.split(",")              # ParlAI format: commas inside fields are encoded as _comma_
                if len(f) <= max(ic, ie, ip) or f[ic] in seen:
                    continue
                seen.add(f[ic])
                if f[ie] in ED_TO_SETFIT:
                    text.append(f[ip].replace("_comma_", ",")); lab.append(ED_TO_SETFIT[f[ie]])
            D["ed"][split] = {"text": text, "label": lab}
            assert len(text) == EXPECTED["ed"][split], f"ED {split}: {len(text)} items"
    c = Counter(D["ed"]["test"]["label"])
    assert dict(c) == EXPECTED["ed"]["test_class_counts"], f"ED test class counts {dict(c)}"
    D["meld_target"] = D["meld"]
    D["goemotions_target"] = D["goemotions"]
    D["isear_target"] = D["isear_ft"]
    return D, files

def check_manifest(files: dict) -> None:
    path = f"{SHARED_PREFIX}/data_manifest.json"
    p = hub_download(path)
    if p is None:
        lp = local_path(path); write_json(lp, {"files": files, "created": datetime.now().isoformat()})
        hub_upload_file(lp, path)
        logger.info("Data manifest created on the Hub")
        return
    with open(p) as f:
        old = json.load(f)["files"]
    diff = {k: (old.get(k), v) for k, v in files.items() if k in old and old[k] != v}
    if diff:
        raise RuntimeError(f"Dataset files differ from the manifest written by an earlier session: {list(diff)}. Aborting.")
    new = {k: v for k, v in files.items() if k not in old}
    if new:                                                     # files added by a newer notebook version: append, never overwrite
        lp = local_path(path); write_json(lp, {"files": {**old, **new}, "created": datetime.now().isoformat()})
        hub_upload_file(lp, path); logger.info(f"Data manifest extended with {sorted(new)}")
    logger.info("Data checksums match the manifest")

# ---- lexical cue helpers (pre-registered definitions) ------------------------------------------
FEEL_RE = re.compile(r"\b(?:feel|feels|feeling|felt)\b", re.I)
NEXT_SKIP_RE = re.compile(r"\b(?:feel|feeling|feels|felt)\s+(?:(?:" + "|".join(INTENSIFIERS) + r")\s+)*([a-z]+)", re.I)
WORD_RE = re.compile(r"[a-z']+")
STRIP = ".,!?;:\"()"

def next_word(text: str) -> str:
    m = NEXT_SKIP_RE.search(text.lower())
    return m.group(1) if m else "<none>"

def cue_vocabulary(texts, labels) -> set:
    by = defaultdict(Counter)
    for t, y in zip(texts, labels):
        w = next_word(t)
        if w != "<none>":
            by[w][int(y)] += 1
    return {w for w, c in by.items() if sum(c.values()) >= CUE_MIN_COUNT and c.most_common(1)[0][1] / sum(c.values()) >= CUE_MIN_PURITY}

def is_explicit(text: str) -> bool:
    return any(w in CUE_VOCAB for w in WORD_RE.findall(text.lower()))

def mask_cues(text: str, mask: str):
    words, n = text.split(), 0
    for i, w in enumerate(words):
        if w.lower().strip(STRIP) in CUE_VOCAB:
            words[i], n = mask, n + 1
    return " ".join(words), n

def mask_random_words(text: str, mask: str, n: int, seed: int) -> str:
    words = text.split()
    if len(words) <= n or n <= 0:
        return text
    rng = random.Random(f"{seed}|{text}")
    for i in rng.sample(range(len(words)), n):
        words[i] = mask
    return " ".join(words)

def mask_random_noncue(text: str, mask: str, n: int, seed: int) -> str:
    """CRI control: mask n randomly chosen words that are NOT cue words (so the control never removes a cue)."""
    words = text.split()
    cand = [i for i, w in enumerate(words) if w.lower().strip(STRIP) not in CUE_VOCAB]
    if len(cand) < n or n <= 0:
        return text
    rng = random.Random(f"{seed}|{text}")
    for i in rng.sample(cand, n):
        words[i] = mask
    return " ".join(words)

def mask_feel_phrase(text: str, mask: str) -> str:
    return NEXT_SKIP_RE.sub(lambda m: f"{mask} {mask}", text)

def load_setfit_large(base: dict, n_train: int, seed: int) -> dict:
    p = fetch(HF_PARQUET.format(repo="dair-ai/emotion", rev="refs%2Fconvert%2Fparquet", path="unsplit/train/0000.parquet"),
              "dairai_unsplit.parquet")
    df = pd.read_parquet(p).drop_duplicates("text")
    held_out = set(base["validation"]["text"]) | set(base["test"]["text"])
    df = df[~df.text.isin(held_out)]
    assert n_train <= len(df), f"requested {n_train} > pool {len(df)}"
    rng = np.random.default_rng(seed)
    frac = n_train / len(df)
    s = pd.concat([g.iloc[rng.permutation(len(g))[: int(round(frac * len(g)))]] for _, g in df.groupby("label")])
    assert not set(s.text) & held_out, "leakage: sampled training text occurs in validation or test"
    return {"train": {"text": s.text.tolist(), "label": s.label.astype(int).tolist()}, "validation": base["validation"], "test": base["test"]}

DATA, DATA_FILES = load_data()
check_manifest(DATA_FILES)
CUE_VOCAB = cue_vocabulary(DATA["setfit"]["train"]["text"], DATA["setfit"]["train"]["label"])
assert len(CUE_VOCAB) == EXPECTED_CUE_VOCAB_SIZE, (f"Cue vocabulary has {len(CUE_VOCAB)} words, the pre-registered value is "
                                                   f"{EXPECTED_CUE_VOCAB_SIZE}. The data or the definition changed. Aborting.")
EXPLICIT = {name: np.array([is_explicit(t) for t in split["text"]])
            for name, split in [("setfit_test", DATA["setfit"]["test"]), ("setfit_validation", DATA["setfit"]["validation"]),
                                ("goemotions", DATA["goemotions"]["test"]), ("meld", DATA["meld"]["test"]),
                                ("isear", DATA["isear"]["test"]), ("ed_test", DATA["ed"]["test"]), ("tweeteval_test", DATA["tweeteval"]["test"]),
                                ("isear_target_test", DATA["isear_ft"]["test"])]}
cue_path = local_path(f"{SHARED_PREFIX}/cue_vocabulary.json")
if f"{SHARED_PREFIX}/cue_vocabulary.json" not in HUB_FILES:
    write_json(cue_path, sorted(CUE_VOCAB)); hub_upload_file(cue_path, f"{SHARED_PREFIX}/cue_vocabulary.json")
for corpus in ["setfit", "tweeteval", "ed"]:
    for split, d in DATA[corpus].items():
        lens = np.array([len(t.split()) for t in d["text"]])
        logger.info(f"{corpus}/{split}: n={len(d['text'])} labels={dict(sorted(Counter(d['label']).items()))} "
                    f"words mean={lens.mean():.1f} p95={np.percentile(lens, 95):.0f}")
for name in ["goemotions", "meld", "isear", "isear_ft"]:
    logger.info(f"{name}/test: n={len(DATA[name]['test']['text'])} labels={dict(sorted(Counter(DATA[name]['test']['label']).items()))}")
logger.info(f"Cue vocabulary: {len(CUE_VOCAB)} words | explicit share: " +
            ", ".join(f"{k}={v.mean():.3f}" for k, v in EXPLICIT.items()))

# ---- crowd-enVENT (used only by the cev_source, cev_masked and cev_target stages) ------------------------------
# WARNING: the crowd-enVENT release has no explicit licence (only a citation request); confirm terms before publication.
def load_cev():
    import io, zipfile
    from sklearn.model_selection import train_test_split
    p = fetch(CEV_URL, "crowd-enVent2023.zip")
    with zipfile.ZipFile(p) as zf:
        names = zf.namelist()
        def read(member):
            return pd.read_csv(io.BytesIO(zf.read(next(n for n in names if n.endswith(member)))), sep="\t")
        gen, val = read("crowd-enVent_generation.tsv"), read("crowd-enVent_validation.tsv")
    assert len(gen) == 6600 and len(val) == 6000, f"crowd-enVENT sizes changed: {len(gen)}, {len(val)}"
    g = gen[gen.emotion.isin(CEV_TO_SETFIT)].sort_values("text_id").reset_index(drop=True)
    assert len(g) == 2750 and not g.text_id.duplicated().any(), f"five-class texts: {len(g)}"
    assert g.generated_text.notna().all() and g.hidden_emo_text.notna().all(), "missing text in crowd-enVENT"
    assert not g[CEV_APPRAISALS].isna().any().any(), "missing appraisal ratings"
    orig = g.generated_text.astype(str).str.strip().tolist()
    masked = g.hidden_emo_text.astype(str).str.strip().tolist()
    changed = np.array([a != b for a, b in zip(orig, masked)])
    assert changed.sum() == 863, f"author-masked pairs: {changed.sum()}"
    y = np.array([CEV_TO_SETFIT[e] for e in g.emotion])
    in_reader = g.text_id.isin(set(val.text_id)).to_numpy()
    reader_pos = np.flatnonzero(in_reader)
    assert len(reader_pos) == 500 and changed[reader_pos].sum() == 163, "reader-annotated subset changed"
    vv = val[val.text_id.isin(set(g.text_id[in_reader]))]
    by = {tid: grp.emotion.tolist() for tid, grp in vv.groupby("text_id")}
    assert all(len(v) == 5 for v in by.values()), "expected five readers per text"
    # maj, item_acc, guesses = [], [], []
    maj, item_acc, guesses, cons = [], [], [], []
    for tid, ytrue in zip(g.text_id[in_reader], y[in_reader]):
        c = Counter(by[tid]).most_common()
        cons.append(c[0][1] / len(by[tid]))                 # share of readers choosing the modal emotion (13 classes)
        # -1: top choice tied between labels; -2: majority label outside the five shared classes
        maj.append(-1 if len(c) > 1 and c[0][1] == c[1][1] else CEV_TO_SETFIT.get(c[0][0], -2))
        gl = [CEV_TO_SETFIT.get(e, -2) for e in by[tid]]
        guesses.append(gl); item_acc.append(float(np.mean([x == ytrue for x in gl])))
    maj = np.array(maj)
    assert (maj == -1).sum() == 56, f"reader-majority ties: {(maj == -1).sum()}"
    rest = g[~in_reader]                                   # upper-bound model: texts the readers did not see
    tr, va = train_test_split(rest, test_size=CEV_VAL_FRACTION, stratify=rest.emotion, random_state=CEV_SPLIT_SEED)
    assert (len(tr), len(va)) == (2025, 225), f"upper-bound split {(len(tr), len(va))}"
    split = lambda d: {"text": d.generated_text.astype(str).str.strip().tolist(), "label": [CEV_TO_SETFIT[e] for e in d.emotion]}
    data = {"train": split(tr), "validation": split(va), "test": split(g.iloc[reader_pos])}
    cev = {"text_orig": orig, "text_masked": masked, "label": y, "text_id": g.text_id.to_numpy(), "changed": changed,
           "reader_pos": reader_pos, "reader_majority": maj, "reader_item_acc": np.array(item_acc),
           "reader_guesses": np.array(guesses), "reader_consensus": np.array(cons), "appraisal": g[CEV_APPRAISALS].to_numpy(float)}
    return data, cev, sha256_file(p)

CEV_DATA, CEV, _cev_sha = load_cev()
DATA["cev_target"] = CEV_DATA
NUM_LABELS["cev_target"] = 6                       # SetFit label ids; love is absent
PRESENT_CLASSES["cev"] = list(CEV_TO_SETFIT.values())
check_manifest({"crowd-enVent2023.zip": _cev_sha})
logger.info(f"crowd-enVENT: {len(CEV['label'])} five-class texts, {int(CEV['changed'].sum())} author-masked pairs, "
            f"{len(CEV['reader_pos'])} reader-annotated ({int((CEV['reader_majority'] >= 0).sum())} with a reader majority), "
            f"upper-bound split {len(CEV_DATA['train']['text'])}/{len(CEV_DATA['validation']['text'])}/{len(CEV_DATA['test']['text'])}")

# ---- enISEAR (v8; evaluation only: eni_* stages, llm_transfer task enisear, Cell 9e) ------------------------------------
def load_enisear() -> tuple:
    import io, zipfile
    p = fetch(ENI_URL, "deISEARenISEAR.zip")
    sha = sha256_file(p)
    assert sha == ENI_SHA256, f"enISEAR archive changed: SHA-256 {sha}, expected {ENI_SHA256}"
    with zipfile.ZipFile(p) as zf:
        e = pd.read_csv(io.BytesIO(zf.read("deISEARenISEAR/enISEAR.tsv")), sep="\t")
        ev = pd.read_csv(io.BytesIO(zf.read("deISEARenISEAR/enISEAR_validation.tsv")), sep="\t")
    assert len(e) == 1001 and e.Sentence_id.is_unique, f"enISEAR descriptions: {len(e)}"
    assert set(e.Prior_Emotion) == set(ENI_EMOTIONS) and e.Prior_Emotion.value_counts().eq(143).all(), "143 descriptions per emotion expected"
    assert (e[ENI_EMOTIONS].sum(axis=1) == 5).all(), "every description should have five reader votes"
    raw = {sid: Counter(str(a).strip().capitalize() for a in grp.Annotation) for sid, grp in ev.groupby("Sentence_id")}
    agg = e.set_index("Sentence_id")[ENI_EMOTIONS]
    assert set(raw) == set(agg.index) and all(int(agg.loc[s, k]) == c.get(k, 0) for s, c in raw.items() for k in ENI_EMOTIONS), \
        "aggregated reader votes in enISEAR.tsv differ from enISEAR_validation.tsv"
    e4 = e[e.Prior_Emotion.isin(ENI_TO_SETFIT)].sort_values("Sentence_id").reset_index(drop=True)
    raw_text = e4.Sentence.astype(str).str.strip()
    n_unicode = int(raw_text.str.contains("\u2026", regex=False).sum())
    text = raw_text.str.replace("\u2026", ENI_MARKER, regex=False).tolist()
    y = e4.Prior_Emotion.map(ENI_TO_SETFIT).to_numpy()
    votes = e4[ENI_EMOTIONS].to_numpy(int)
    maj = []
    for row in votes:
        order = np.argsort(-row, kind="stable")
        # -1: top choice tied between labels; -2: majority label outside the four shared classes (as for crowd-enVENT)
        maj.append(-1 if row[order[0]] == row[order[1]] else ENI_TO_SETFIT.get(ENI_EMOTIONS[order[0]], -2))
    maj = np.array(maj)
    item_acc = votes[np.arange(len(e4)), [ENI_EMOTIONS.index(lab) for lab in e4.Prior_Emotion]] / 5.0
    assert len(e4) == 572 and n_unicode == 166 and (maj == -1).sum() == 17 and (maj == -2).sum() == 11, \
        f"enISEAR four-class set changed: n={len(e4)}, marker={n_unicode}, ties={(maj == -1).sum()}, outside={(maj == -2).sum()}"
    eni = {"text": text, "label": y, "sentence_id": e4.Sentence_id.to_numpy(), "reader_majority": maj,
           "reader_item_acc": item_acc, "reader_votes": votes, "n_unicode_marker": n_unicode}
    return eni, sha

ENI, _eni_sha = load_enisear()
PRESENT_CLASSES["eni"] = list(ENI_TO_SETFIT.values())
check_manifest({"deISEARenISEAR.zip": _eni_sha})
_ok = ENI["reader_majority"] >= 0
logger.info(f"enISEAR: {len(ENI['label'])} four-class descriptions ({ENI['n_unicode_marker']} markers normalized to '{ENI_MARKER}'), "
            f"{int(_ok.sum())} with a unique reader majority in the four classes, "
            f"{int((ENI['reader_majority'][_ok] == ENI['label'][_ok]).sum())} of them matching the writer's label")

[2026-10-07 12:07:25] [INFO] Data checksums match the manifest
[2026-10-07 12:07:25] [INFO] setfit/train: n=16000 labels={0: 4666, 1: 5362, 2: 1304, 3: 2159, 4: 1937, 5: 572} words mean=19.2 p95=41
[2026-10-07 12:07:25] [INFO] setfit/validation: n=2000 labels={0: 550, 1: 704, 2: 178, 3: 275, 4: 212, 5: 81} words mean=18.9 p95=40
[2026-10-07 12:07:25] [INFO] setfit/test: n=2000 labels={0: 581, 1: 695, 2: 159, 3: 275, 4: 224, 5: 66} words mean=19.2 p95=41
[2026-10-07 12:07:25] [INFO] tweeteval/train: n=3257 labels={0: 1400, 1: 708, 2: 294, 3: 855} words mean=16.2 p95=26
[2026-10-07 12:07:25] [INFO] tweeteval/validation: n=374 labels={0: 160, 1: 97, 2: 28, 3: 89} words mean=15.7 p95=26
[2026-10-07 12:07:25] [INFO] tweeteval/test: n=1421 labels={0: 558, 1: 358, 2: 123, 3: 382} words mean=15.9 p95=26
[2026-10-07 12:07:25] [INFO] ed/train: n=16343 labels={0: 4714, 1: 5105, 3: 3181, 4: 2339, 5: 1004} words mean=17.5 p95=36
[2026-10-07 12:07:25] [INFO] ed/validation: n=2284 labels={0: 670, 1: 

## Cell 7 — Model and optimizer initialization

In [31]:
# =====================================================================================
# CELL 7 — MODEL AND OPTIMIZER INITIALIZATION
# build_model(spec, n_train) returns tokenizer, model, optimizer, scheduler, and the resolved prompt prefix.
# =====================================================================================
import transformers
import gc, math
from transformers import (AutoModelForMaskedLM, AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

def precision_mode(requested: str) -> str:
    if requested == "fp32" or not torch.cuda.is_available():
        return "fp32"
    major, _ = torch.cuda.get_device_capability(0)
    if requested == "bf16" and major < 8:
        raise RuntimeError("BF16 requested on a GPU without native BF16 support (compute capability < 8.0)")
    return "bf16" if (requested == "bf16" or (requested == "auto" and major >= 8)) else "fp16"

def resolve_prefix(tok, condition: str) -> str:
    """Prompt text prepended (or appended/paired) to every input. 'random' = random vocabulary tokens with the
    same token count as template A under this tokenizer, fixed by RANDOM_PREFIX_SEED."""
    if condition == "random":
        n = len(tok(PROMPT_TEMPLATES["A"], add_special_tokens=False)["input_ids"])
        special = set(tok.all_special_ids)
        rng = random.Random(RANDOM_PREFIX_SEED)
        ids = []
        while len(ids) < n:
            i = rng.randrange(tok.vocab_size)
            piece = tok.convert_ids_to_tokens(i)
            if i not in special and piece is not None and piece.strip("▁Ġ#").isalpha():
                ids.append(i)
        return tok.decode(ids).strip()
    assert condition in PROMPT_TEMPLATES and PROMPT_TEMPLATES[condition] is not None, f"unknown condition {condition}"
    return PROMPT_TEMPLATES[condition]

def encode(tok, texts: list, prefix: str, placement: str, head: str) -> dict:
    """Tokenize with the prompt in the requested placement. Returns lists (padding is done per batch)."""
    texts = [str(t) for t in texts]
    if head == "verbalizer":
        body = [f"{prefix} {t} {tok.mask_token}." if prefix else f"{t} {tok.mask_token}." for t in texts]
        enc = tok(body, truncation=True, max_length=MAX_SEQ_LEN)
        for ids in enc["input_ids"]:                   # truncation must never remove the mask token
            if tok.mask_token_id not in ids:
                ids[-2] = tok.mask_token_id
        return enc
    if prefix and placement == "pair":
        return tok([prefix] * len(texts), texts, truncation="only_second", max_length=MAX_SEQ_LEN)
    if prefix and placement == "suffix":
        return tok([f"{t} {prefix}" for t in texts], truncation=True, max_length=MAX_SEQ_LEN)
    return tok([f"{prefix} {t}" if prefix else t for t in texts], truncation=True, max_length=MAX_SEQ_LEN)

def verbalizer_ids(tok) -> list:
    ids = []
    for name in SETFIT:
        for cand in VERBALIZER_CANDIDATES[name]:
            found = None
            for form in (" " + cand, cand):
                t = tok(form, add_special_tokens=False)["input_ids"]
                if len(t) == 1:
                    found = t[0]; break
            if found is not None:
                ids.append(found); break
        else:
            raise ValueError(f"No single-token verbalizer for '{name}' in {tok.name_or_path}")
    assert len(set(ids)) == len(ids), "verbalizer tokens must be distinct"
    return ids

class VerbalizerModel(torch.nn.Module):
    """PET-style head: class logits = masked-LM logits of one label word per class at the mask position."""
    def __init__(self, mlm, label_ids: list, mask_id: int):
        super().__init__()
        self.mlm, self.mask_id = mlm, mask_id
        self.register_buffer("label_ids", torch.tensor(label_ids, dtype=torch.long))

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        pos = (input_ids == self.mask_id).float().argmax(1)
        assert bool((input_ids[torch.arange(len(pos)), pos] == self.mask_id).all()), "mask token missing in a batch item"
        rows = torch.arange(len(pos), device=input_ids.device)
        head = getattr(self.mlm, "lm_head", None) or getattr(self.mlm, "cls", None)
        if head is not None:
            # The masked-LM head acts on each position independently, so applying it only to the mask position gives
            # exactly the same logits as the full forward pass, without materializing (batch x length x vocabulary).
            hidden = self.mlm.base_model(**kw).last_hidden_state[rows, pos]
            logits = head(hidden.unsqueeze(1)).squeeze(1)
        else:
            # WARNING: architecture without an lm_head/cls attribute; falling back to the full (memory-heavy) forward pass
            logits = self.mlm(**kw).logits[rows, pos]
        return type("Out", (), {"logits": logits[:, self.label_ids]})()

HEAD_PREFIXES = ("classifier", "pre_classifier", "score")

class SoftPromptModel(torch.nn.Module):
    """SOFT_PROMPT_TOKENS trainable vectors placed at position 0 of the input embeddings (v6 design: the
    classification head reads the final hidden state of the first prompt vector). freeze_backbone=True trains only
    the prompt vectors and the classification head."""
    def __init__(self, base, n_tokens: int, freeze_backbone: bool, init_ids: list):
        super().__init__()
        self.base = base
        emb = base.get_input_embeddings()
        self.prompt = torch.nn.Parameter(emb.weight[torch.tensor(init_ids)].detach().clone())
        if freeze_backbone:
            for n, p in base.named_parameters():
                p.requires_grad = n.startswith(HEAD_PREFIXES)
            assert any(p.requires_grad for p in base.parameters()), "no classification-head parameters found to train"

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        e = self.base.get_input_embeddings()(input_ids)
        b, k = e.shape[0], self.prompt.shape[0]
        x = torch.cat([self.prompt.unsqueeze(0).expand(b, -1, -1).to(e.dtype), e], 1)
        m = torch.cat([torch.ones(b, k, dtype=attention_mask.dtype, device=e.device), attention_mask], 1)
        kw = {"inputs_embeds": x, "attention_mask": m}
        if token_type_ids is not None:
            kw["token_type_ids"] = torch.cat([torch.zeros(b, k, dtype=token_type_ids.dtype, device=e.device), token_type_ids], 1)
        return self.base(**kw)

def build_model(spec: dict, n_train: int, num_labels: int):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        tok = AutoTokenizer.from_pretrained(spec["hf_id"])
        if spec["head"] == "verbalizer":
            model = VerbalizerModel(AutoModelForMaskedLM.from_pretrained(spec["hf_id"]), verbalizer_ids(tok), tok.mask_token_id)
        else:
            model = AutoModelForSequenceClassification.from_pretrained(spec["hf_id"], num_labels=num_labels)
        model = model.float()
        assert all(p.dtype == torch.float32 for p in model.parameters()), "model weights are not FP32"
        pm = spec.get("peft_method")
        if pm and pm.startswith("lora"):
            from peft import LoraConfig, TaskType, get_peft_model
            r = int(pm.split("_r")[1])
            mt = model.config.model_type
            assert mt in LORA_TARGETS, f"no LoRA target modules configured for model type '{mt}'"
            model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                                     target_modules=LORA_TARGETS[mt]))
        elif pm in ("softprompt_frozen", "softprompt_full"):
            rng = random.Random(RANDOM_PREFIX_SEED)
            special = set(tok.all_special_ids)
            init = []
            while len(init) < SOFT_PROMPT_TOKENS:                     # initialise from random ordinary vocabulary embeddings
                i = rng.randrange(tok.vocab_size)
                if i not in special:
                    init.append(i)
            model = SoftPromptModel(model, SOFT_PROMPT_TOKENS, pm == "softprompt_frozen", init)
        elif pm is not None:
            raise ValueError(f"unknown peft_method {pm}")
    except Exception as e:
        logger.error(f"Model construction failed for {spec['hf_id']}: {e}")
        raise
    model.to(DEVICE)
    prefix = resolve_prefix(tok, spec["condition"])
    if spec["fewshot_k"]:
        steps_per_epoch = math.ceil(n_train / BATCH_SIZE)
        total_steps = max(FEWSHOT_MIN_STEPS, spec["epochs"] * steps_per_epoch)
    else:
        total_steps = spec["epochs"] * math.ceil(n_train / BATCH_SIZE)
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight", "norm.weight"]
    trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    groups = [{"params": [p for n, p in trainable if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in trainable if any(nd in n for nd in no_decay)], "weight_decay": 0.0}]
    optimizer = torch.optim.AdamW(groups, lr=spec["lr"])
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Model {spec['hf_id']} ({spec['head']}): {n_params:,} parameters, {n_trainable:,} trainable | "
                f"prefix='{prefix}' placement={spec['placement']} | total steps {total_steps}")
    return tok, model, optimizer, scheduler, prefix, total_steps


## Cell 8 — Training loop

In [32]:
# =====================================================================================
# CELL 8 — TRAINING LOOP
# For every pending run of the stage: seed -> data -> model -> train (checkpoints every CHECKPOINT_EVERY_N_STEPS,
# early stopping on validation loss) -> export logits for every evaluation input while the model is in memory
# (validation, test, cue-masked and random-masked validation for CRI, transfer targets) -> optional final model
# with reload gate -> one Hub commit for the run -> DONE marker. A failed run is logged and skipped; the sweep
# continues. Metrics are computed from the exported logits in Cell 9 (no model reload is ever needed there).
# =====================================================================================
import traceback
from contextlib import nullcontext

LARGE_POOLS = {}

def stratified_subset(labels, k: int, seed: int) -> list:
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    idx = []
    for c in sorted(set(labels.tolist())):
        pool = np.flatnonzero(labels == c)
        assert len(pool) >= k, f"class {c} has {len(pool)} items < k={k}"
        idx.extend(rng.choice(pool, size=k, replace=False).tolist())
    return sorted(idx)

def prepare_run_data(spec: dict, mask_token: str) -> dict:
    corpus = spec["corpus"]
    if corpus.startswith("setfit_large_"):
        n = int(corpus.rsplit("_", 1)[1])
        if n not in LARGE_POOLS:
            LARGE_POOLS[n] = load_setfit_large(DATA["setfit"], n, SCALING_POOL_SEED)
        src, num_labels, base = LARGE_POOLS[n], 6, "setfit"
    else:
        src, num_labels, base = DATA[corpus], NUM_LABELS[corpus], corpus
    tr_t, tr_y = list(src["train"]["text"]), list(src["train"]["label"])
    if spec["fewshot_k"]:
        keep = stratified_subset(tr_y, spec["fewshot_k"], spec["seed"])
        tr_t, tr_y = [tr_t[i] for i in keep], [tr_y[i] for i in keep]
    if spec.get("rebalance") == "oversample":                      # every class up to the majority count, with replacement
        rng = np.random.default_rng(spec["seed"])
        y_arr = np.asarray(tr_y); top = np.bincount(y_arr).max(); idx = []
        for c in sorted(set(tr_y)):
            pool = np.flatnonzero(y_arr == c)
            idx.extend(pool.tolist() + rng.choice(pool, size=top - len(pool), replace=True).tolist())
        tr_t, tr_y = [tr_t[i] for i in idx], [tr_y[i] for i in idx]
        assert len(tr_y) == top * len(set(tr_y)), "oversampling size mismatch"
    va_t, te_t = list(src["validation"]["text"]), list(src["test"]["text"])
    if spec["text_transform"]:
        fn = {"mask_feel": lambda t: mask_feel_phrase(t, mask_token),
              "mask_random": lambda t: mask_random_words(t, mask_token, 2, spec["seed"])}[spec["text_transform"]]
        tr_t = [fn(t) for t in tr_t]
        if spec["transform_eval"]:
            va_t, te_t = [fn(t) for t in va_t], [fn(t) for t in te_t]
    assert len(tr_t) == len(tr_y) > 0 and len(va_t) == len(src["validation"]["label"]) and len(te_t) == len(src["test"]["label"])
    return {"train": (tr_t, tr_y), "validation": (va_t, list(src["validation"]["label"])), "test": (te_t, list(src["test"]["label"])),
            "num_labels": num_labels, "base_corpus": base,
            "orig_validation_text": list(src["validation"]["text"])}

def batches(enc: dict, order, labels=None):
    keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
    for i in range(0, len(order), BATCH_SIZE if labels is not None else EVAL_BATCH_SIZE):
        idx = order[i:i + (BATCH_SIZE if labels is not None else EVAL_BATCH_SIZE)]
        yield idx, [{k: enc[k][j] for k in keys} for j in idx]

def autocast_ctx(mode: str):
    if mode == "fp32" or DEVICE.type != "cuda":
        return nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if mode == "bf16" else torch.float16)

def predict_logits(model, tok, enc: dict, mode: str) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad():
        for _, feats in batches(enc, np.arange(len(enc["input_ids"]))):
            b = tok.pad(feats, return_tensors="pt")
            b = {k: v.to(DEVICE) for k, v in b.items()}
            try:
                with autocast_ctx(mode):
                    out.append(model(**b).logits.float().cpu().numpy())
            except torch.cuda.OutOfMemoryError:
                logger.error(f"OOM during inference. Allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB")
                torch.cuda.empty_cache(); raise
    logits = np.concatenate(out)
    assert logits.shape[0] == len(enc["input_ids"]) and np.isfinite(logits).all(), "invalid logits"
    return logits

def save_checkpoint(path: Path, state: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    torch.save(state, tmp)
    shutil.move(str(tmp), str(path))

def train_run(spec: dict, run_dir: Path, rp: str) -> dict:
    set_seed(spec["seed"])
    mode = precision_mode(spec["precision"])
    tok = AutoTokenizer.from_pretrained(spec["hf_id"])
    data = prepare_run_data(spec, tok.mask_token or tok.unk_token)
    tr_t, tr_y = data["train"]
    tok, model, optimizer, scheduler, prefix, total_steps = build_model(spec, len(tr_y), data["num_labels"])
    enc_tr = encode(tok, tr_t, prefix, spec["placement"], spec["head"])
    enc_va = encode(tok, data["validation"][0], prefix, spec["placement"], spec["head"])
    enc_te = encode(tok, data["test"][0], prefix, spec["placement"], spec["head"])
    y_tr = torch.tensor(tr_y)
    y_va = np.asarray(data["validation"][1])
    class_w = None
    if spec.get("rebalance") == "weighted":                        # ASSUMPTION: inverse-frequency weights n / (K * n_c)
        cnt = np.bincount(tr_y, minlength=data["num_labels"]).astype(float)
        present = cnt > 0
        w = np.zeros_like(cnt); w[present] = cnt.sum() / (present.sum() * cnt[present])
        class_w = torch.tensor(w, dtype=torch.float32, device=DEVICE)
        logger.info(f"class weights: {np.round(w, 3).tolist()}")
    if mode == "fp16":
        scaler = torch.amp.GradScaler("cuda") if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler()
    else:
        scaler = None
    upload_mid = UPLOAD_MIDRUN_CHECKPOINTS is True or (UPLOAD_MIDRUN_CHECKPOINTS == "auto" and total_steps > AUTO_UPLOAD_MIN_STEPS)
    few = bool(spec["fewshot_k"])
    st = {"step": 0, "epoch": 0, "batch_in_epoch": 0, "best_val_loss": float("inf"), "bad_epochs": 0, "best_state": None,
          "history": [], "stopped_early": False}

    # ---- resume: local checkpoint first (same session), then the Hub ------------------------------------------
    latest_local = run_dir / "checkpoints" / "latest.json"
    latest_file = latest_local if latest_local.exists() else hub_download(f"{rp}/checkpoints/latest.json")
    if latest_file is not None:
        with open(latest_file) as f:
            latest = json.load(f)
        ck_rel = f"{rp}/{latest['latest_checkpoint']}ckpt.pt"
        ck = local_path(ck_rel) if local_path(ck_rel).exists() else hub_download(ck_rel)
        if ck is not None:
            saved = torch.load(ck, map_location="cpu", weights_only=False)
            assert saved["spec_id"] == spec["spec_id"], "checkpoint belongs to a different run specification"
            model.load_state_dict(saved["model_state_dict"]); optimizer.load_state_dict(saved["optimizer_state_dict"])
            scheduler.load_state_dict(saved["scheduler_state_dict"])
            if scaler is not None and saved.get("scaler_state_dict"):
                scaler.load_state_dict(saved["scaler_state_dict"])
            st = saved["train_state"]
            torch.set_rng_state(saved["rng_torch"]); np.random.set_state(saved["rng_numpy"]); random.setstate(saved["rng_python"])
            if torch.cuda.is_available() and saved.get("rng_cuda") is not None:
                torch.cuda.set_rng_state_all(saved["rng_cuda"])            # GPU dropout masks resume exactly
            logger.info(f"Resumed {spec['spec_id']} from step {st['step']}, epoch {st['epoch']}, batch {st['batch_in_epoch']}")
        else:
            logger.warning(f"latest.json found but checkpoint file missing for {spec['spec_id']}; starting fresh")
    else:
        logger.info(f"No checkpoint for {spec['spec_id']}. Starting fresh.")

    def checkpoint(tag: str):
        rel = f"checkpoints/{tag}/"
        path = run_dir / rel / "ckpt.pt"
        save_checkpoint(path, {"model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                               "scheduler_state_dict": scheduler.state_dict(),
                               "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                               "train_state": st, "spec_id": spec["spec_id"], "run_id": RUN_ID, "config": CONFIG_SNAPSHOT,
                               "timestamp": datetime.now().isoformat(), "rng_torch": torch.get_rng_state(),
                               "rng_numpy": np.random.get_state(), "rng_python": random.getstate(),
                               "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None})
        if upload_mid:
            hub_upload_file(path, f"{rp}/{rel}ckpt.pt")
        latest = {"latest_checkpoint": rel, "step": st["step"], "epoch": st["epoch"], "timestamp": datetime.now().isoformat()}
        write_json(run_dir / "checkpoints" / "latest.json", latest)       # only after the save (and upload) succeeded
        if upload_mid:
            hub_upload_file(run_dir / "checkpoints" / "latest.json", f"{rp}/checkpoints/latest.json")
        for old in (run_dir / "checkpoints").glob("step_*"):              # keep only the newest local checkpoint
            if old.name != tag:
                shutil.rmtree(old, ignore_errors=True)
        if upload_mid:                                                     # and only the newest one on the Hub
            stale = sorted({"/".join(f.split("/")[:-1]) for f in HUB_FILES
                            if f.startswith(f"{rp}/checkpoints/step_") and not f.startswith(f"{rp}/{rel}")})
            for d in stale:
                upload_with_retry(lambda d=d: API.delete_folder(path_in_repo=d, repo_id=HF_REPO_ID, repo_type="dataset",
                                                                 commit_message=f"remove superseded checkpoint {d.rsplit('/', 1)[-1]}"))
                HUB_FILES.difference_update({f for f in list(HUB_FILES) if f.startswith(d + "/")})
        logger.debug(f"checkpoint {tag} saved (uploaded={upload_mid})")

    model.train()
    t0 = time.time()
    n = len(tr_y)
    try:
        while True:
            if few and st["step"] >= total_steps:
                break
            if not few and (st["epoch"] >= spec["epochs"] or st["stopped_early"]):
                break
            order = np.random.default_rng(spec["seed"] * 1000 + st["epoch"]).permutation(n)   # deterministic per epoch
            model.train()
            for bi, (idx, feats) in enumerate(batches(enc_tr, order, labels=True)):
                if bi < st["batch_in_epoch"]:
                    continue                                                   # resume without repeating batches
                b = tok.pad(feats, return_tensors="pt")
                b = {k: v.to(DEVICE) for k, v in b.items()}
                y = y_tr[idx].to(DEVICE)
                with autocast_ctx(mode):
                    loss = torch.nn.functional.cross_entropy(model(**b).logits.float(), y, weight=class_w)
                assert torch.isfinite(loss), f"non-finite loss at step {st['step']}"
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM); optimizer.step()
                scheduler.step()
                st["step"] += 1; st["batch_in_epoch"] = bi + 1
                if st["step"] % LOG_EVERY_N_STEPS == 0:
                    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
                    logger.info(f"[{spec['spec_id']}] step {st['step']}/{total_steps} loss {loss.item():.4f} "
                                f"lr {scheduler.get_last_lr()[0]:.2e} mem {mem:.2f} GB")
                if st["step"] % CHECKPOINT_EVERY_N_STEPS == 0:
                    checkpoint(f"step_{st['step']}")
                if few and st["step"] >= total_steps:
                    break
            st["epoch"] += 1; st["batch_in_epoch"] = 0
            if not few:                                                        # model selection on validation loss only
                lg = predict_logits(model, tok, enc_va, mode)
                vloss = float(torch.nn.functional.cross_entropy(torch.tensor(lg), torch.tensor(y_va)))
                vacc = float(np.mean(lg.argmax(1) == y_va))
                st["history"].append({"epoch": st["epoch"], "step": st["step"], "val_loss": vloss, "val_acc": vacc})
                logger.info(f"[{spec['spec_id']}] epoch {st['epoch']} val_loss {vloss:.4f} val_acc {vacc:.4f}")
                if vloss < st["best_val_loss"] - 1e-6:
                    st["best_val_loss"], st["bad_epochs"] = vloss, 0
                    st["best_state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    st["bad_epochs"] += 1
                    st["stopped_early"] = st["bad_epochs"] >= spec.get("patience", EARLY_STOP_PATIENCE)
            gc.collect()
    except torch.cuda.OutOfMemoryError as e:
        logger.error(f"CUDA OOM at step {st['step']}. Allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB. Error: {e}")
        try:
            checkpoint("oom_recovery"); logger.info("OOM recovery checkpoint saved.")
        except Exception as save_err:
            logger.error(f"Recovery checkpoint failed: {save_err}")
        torch.cuda.empty_cache()
        raise
    train_seconds = time.time() - t0
    if not few:
        assert st["best_state"] is not None, "no validation epoch completed"
        model.load_state_dict(st["best_state"])

    # ---- export logits for every evaluation input while the model is in memory ---------------------------------
    outs = {"val_logits": predict_logits(model, tok, enc_va, mode), "val_y": y_va}
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_inf = time.time()
    outs["test_logits"] = predict_logits(model, tok, enc_te, mode)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    test_items_per_second = len(enc_te["input_ids"]) / max(time.time() - t_inf, 1e-9)
    outs["test_y"] = np.asarray(data["test"][1])
    trunc = float(np.mean([len(x) >= MAX_SEQ_LEN for x in enc_te["input_ids"]]))
    if data["base_corpus"] == "setfit":                                         # CRI on original validation texts
        mask = tok.mask_token or tok.unk_token
        cue_t, rnd_t, orig_t = [], [], []
        for t in data["orig_validation_text"]:
            m, k = mask_cues(t, mask)
            if k:
                orig_t.append(t); cue_t.append(m); rnd_t.append(mask_random_noncue(t, mask, k, spec["seed"]))
        for key, texts in (("cri_orig", orig_t), ("cri_cue", cue_t), ("cri_random", rnd_t)):
            outs[f"{key}_logits"] = predict_logits(model, tok, encode(tok, texts, prefix, spec["placement"], spec["head"]), mode)

        if spec["transfer"]:
            for tgt in TRANSFER_TARGETS:
                d = DATA[tgt]["test"]
                outs[f"transfer_{tgt}_logits"] = predict_logits(model, tok, encode(tok, d["text"], prefix, spec["placement"], spec["head"]), mode)
                outs[f"transfer_{tgt}_y"] = np.asarray(d["label"])
    elif spec["stage"] == "cev_third" and spec["transfer"]:      # GoEmotions-trained: transfer to the other three targets
        for tgt in [t for t in TRANSFER_TARGETS if t != "goemotions"]:
            d = DATA[tgt]["test"]
            outs[f"transfer_{tgt}_logits"] = predict_logits(model, tok, encode(tok, d["text"], prefix, spec["placement"], spec["head"]), mode)
            outs[f"transfer_{tgt}_y"] = np.asarray(d["label"])

    if spec["stage"] in CEV_EXPORT_STAGES:             # crowd-enVENT: original and masked texts (2,750 each)

        for key, texts in (("cev_orig", CEV["text_orig"]), ("cev_masked", CEV["text_masked"])):
            outs[f"{key}_logits"] = predict_logits(model, tok, encode(tok, texts, prefix, spec["placement"], spec["head"]), mode)
    if spec["corpus"] == "cev_target":                 # the same 500 test texts with the writers' emotion words masked
        outs["cevtest_masked_logits"] = predict_logits(
            model, tok, encode(tok, [CEV["text_masked"][i] for i in CEV["reader_pos"]], prefix, spec["placement"], spec["head"]), mode)
    if spec["stage"] in ENI_EXPORT_STAGES:             # v8: enISEAR, the 572 four-class descriptions as the readers saw them
        outs["eni_logits"] = predict_logits(model, tok, encode(tok, ENI["text"], prefix, spec["placement"], spec["head"]), mode)
    np.savez_compressed(run_dir / "outputs" / "logits.npz", **outs)

    # ---- optional final model with reload gate --------------------------------------------------------------
    reload_agreement = None
    sel = SAVE_FINAL_MODEL_FOR
    if (spec["head"] == "cls" and spec["stage"] in sel["stages"] and spec["seed"] in sel["seeds"] and spec["condition"] in sel["conditions"]):
        final_dir = run_dir / "checkpoints" / "final"
        model.save_pretrained(final_dir, safe_serialization=True); tok.save_pretrained(final_dir)
        write_json(final_dir / "run_spec.json", {**spec, "prefix": prefix})
        # reloaded = AutoModelForSequenceClassification.from_pretrained(final_dir).to(DEVICE)
        reloaded = AutoModelForSequenceClassification.from_pretrained(final_dir).float().to(DEVICE)
        reload_agreement = float(np.mean(predict_logits(reloaded, tok, enc_va, mode).argmax(1) == outs["val_logits"].argmax(1)))
        del reloaded
        if reload_agreement < 0.999:
            raise RuntimeError(f"Reload gate failed: saved model agrees on {reload_agreement:.4f} of validation predictions")
    n_trainable_params = int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    n_total_params = int(sum(p.numel() for p in model.parameters()))
    del model, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return {"prefix": prefix, "train_seconds": train_seconds, "total_steps": total_steps, "steps_run": st["step"],
            "epochs_run": st["epoch"], "history": st["history"], "stopped_early": st["stopped_early"],
            "test_truncated_frac": trunc, "n_train": len(tr_y), "precision_used": mode, "reload_agreement": reload_agreement,
            "midrun_checkpoints_uploaded": upload_mid, "test_items_per_second": test_items_per_second,
            "trainable_parameters": n_trainable_params, "total_parameters": n_total_params}

def llm_demos(k: int) -> list:
    if k == 0:
        return []
    assert k == len(SETFIT), "demonstrations are one per class"
    rng = np.random.default_rng(LLM_DEMO_SEED)
    y = np.asarray(DATA["setfit"]["train"]["label"])
    demos = [(DATA["setfit"]["train"]["text"][int(rng.choice(np.flatnonzero(y == c)))], c) for c in range(len(SETFIT))]
    return [demos[i] for i in rng.permutation(len(demos))]

def llm_messages(text: str, demos: list) -> list:
    msgs = [{"role": "system", "content": LLM_SYSTEM}]
    for t, c in demos:
        msgs += [{"role": "user", "content": f"Text: {t}\nEmotion:"}, {"role": "assistant", "content": SETFIT[c]}]
    return msgs + [{"role": "user", "content": f"Text: {text}\nEmotion:"}]

def parse_label(raw: str) -> int:
    """Strict parser: exactly one of the six label names must occur as a word; otherwise -1 (format failure,
    scored as an error)."""
    found = [c for c, name in enumerate(SETFIT) if re.search(rf"\b{name}\b", raw.lower())]
    return found[0] if len(found) == 1 else -1

def run_llm(spec: dict, run_dir: Path, rp: str) -> dict:
    texts, labels = DATA["setfit"]["test"]["text"], DATA["setfit"]["test"]["label"]
    demos = llm_demos(spec["shots"])
    n_chunks = math.ceil(len(texts) / LLM_CHUNK)
    gen = None
    if spec["backend"] == "hf":
        from transformers import AutoModelForCausalLM
        tok = AutoTokenizer.from_pretrained(spec["hf_id"], token=SECRETS["HF_TOKEN"])
        kw = {"token": SECRETS["HF_TOKEN"]}
        if torch.cuda.is_available() and LLM_LOAD_4BIT:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
            kw["device_map"] = "auto"
        else:
            logger.warning("LLM loaded without 4-bit quantization (no GPU or LLM_LOAD_4BIT=False)")
        lm = AutoModelForCausalLM.from_pretrained(spec["hf_id"], **kw)
        if "device_map" not in kw:
            lm.to(DEVICE)
        lm.eval()
        def template(m):
            # return_dict=True gives the same type (BatchEncoding with input_ids and attention_mask) in Transformers 4 and 5;
            # without it, Transformers 5 returns a BatchEncoding where Transformers 4 returned a tensor
            return tok.apply_chat_template(m, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        def gen(msgs):
            try:
                enc = template(msgs)
            except Exception:
                # WARNING: chat template rejected the system role (or is missing); system text merged into the first user turn
                merged = [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}] + msgs[2:]
                if tok.chat_template:
                    enc = template(merged)
                else:
                    enc = tok("\n".join(m["content"] for m in merged) + "\n", return_tensors="pt")
            ids = enc["input_ids"].to(lm.device)
            mask = enc["attention_mask"].to(lm.device) if "attention_mask" in enc else torch.ones_like(ids)
            with torch.no_grad():
                out = lm.generate(input_ids=ids, attention_mask=mask, max_new_tokens=LLM_MAX_NEW_TOKENS, do_sample=False,
                                  pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
            return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True), spec["hf_id"]
    else:
        from openai import OpenAI
        client = OpenAI(api_key=SECRETS["OPENAI_API_KEY"])
        def gen(msgs):
            for attempt in range(6):
                try:
                    r = client.chat.completions.create(model=spec["api_model"], messages=msgs, temperature=0,
                                                       max_tokens=LLM_MAX_NEW_TOKENS, seed=LLM_DEMO_SEED)
                    return (r.choices[0].message.content or ""), r.model
                except Exception as e:
                    wait = 5 * 2 ** attempt
                    logger.warning(f"OpenAI call failed ({type(e).__name__}); retry in {wait}s"); time.sleep(wait)
            raise RuntimeError("OpenAI API failed after 6 attempts")
    reported = set()
    for ci in range(n_chunks):
        rel = f"outputs/llm_chunk_{ci:03d}.jsonl"
        if (run_dir / rel).exists() or hub_download(f"{rp}/{rel}") is not None:
            logger.info(f"chunk {ci + 1}/{n_chunks} already on the Hub"); continue
        rows = []
        for i in range(ci * LLM_CHUNK, min((ci + 1) * LLM_CHUNK, len(texts))):
            raw, model_id = gen(llm_messages(texts[i], demos))
            reported.add(model_id)
            rows.append({"i": i, "label": int(labels[i]), "raw": raw, "pred": parse_label(raw), "model_reported": model_id})
        with open(run_dir / rel, "w") as f:
            f.write("\n".join(json.dumps(r) for r in rows) + "\n")
        hub_upload_file(run_dir / rel, f"{rp}/{rel}")                # each chunk persisted before the next one starts
        logger.info(f"[{spec['spec_id']}] chunk {ci + 1}/{n_chunks} done")
    if spec["backend"] == "hf":
        del lm
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    gc.collect()
    return {"n_items": len(texts), "n_chunks": n_chunks, "demos": [[t, SETFIT[c]] for t, c in demos], "models_reported": sorted(reported)}

def run_audit(spec: dict, run_dir: Path, rp: str) -> dict:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.metrics import accuracy_score, f1_score
    tr, te = DATA["setfit"]["train"], DATA["setfit"]["test"]
    res = {"feel_rate": {k: float(np.mean([bool(FEEL_RE.search(t)) for t in d["text"]])) for k, d in
                         [("setfit_train", tr), ("setfit_test", te), ("goemotions_test", DATA["goemotions"]["test"]),
                          ("meld_test", DATA["meld"]["test"]), ("isear", DATA["isear"]["test"]), ("ed_test", DATA["ed"]["test"]),
                          ("tweeteval_test", DATA["tweeteval"]["test"])]}}
    raw_re = re.compile(r"\b(?:feel|feeling|feels|felt)\s+([a-z]+)", re.I)
    for skip in (True, False):                                   # one-token probe: majority training label of the word after "feel"
        nw = (lambda t: next_word(t)) if skip else (lambda t: (raw_re.search(t.lower()).group(1) if raw_re.search(t.lower()) else "<none>"))
        by = defaultdict(Counter)
        for t, y in zip(tr["text"], tr["label"]):
            by[nw(t)][y] += 1
        lut = {w: c.most_common(1)[0][0] for w, c in by.items()}
        fb = Counter(tr["label"]).most_common(1)[0][0]
        p = [lut.get(nw(t), fb) for t in te["text"]]
        res[f"probe_skip_intensifiers_{skip}"] = {"acc": float(accuracy_score(te["label"], p)), "macro_f1": float(f1_score(te["label"], p, average="macro"))}
    lab_by_text = defaultdict(set)
    for t, y in zip(tr["text"], tr["label"]):
        lab_by_text[t].add(y)
    res["train_texts_with_conflicting_labels"] = int(sum(len(v) > 1 for v in lab_by_text.values()))
    res["test_texts_also_in_train"] = int(sum(t in lab_by_text for t in te["text"]))
    vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
    Xtr, Xte = vec.fit_transform(tr["text"]), vec.transform(te["text"])
    for name, clf in [("tfidf_lr", LogisticRegression(C=1.0, max_iter=2000)), ("tfidf_svm", LinearSVC(C=1.0))]:
        t0 = time.time(); clf.fit(Xtr, tr["label"]); fit_s = time.time() - t0
        t0 = time.time(); p = clf.predict(Xte); ips = len(te["text"]) / max(time.time() - t0, 1e-9)
        res[name] = {"acc": float(accuracy_score(te["label"], p)), "macro_f1": float(f1_score(te["label"], p, average="macro")),
                     "fit_seconds": fit_s, "items_per_second_prevectorized": ips}
    res["cue_vocabulary_size"] = len(CUE_VOCAB)
    res["explicit_share"] = {k: float(v.mean()) for k, v in EXPLICIT.items()}
    write_json(run_dir / "outputs" / "audit.json", res)
    logger.info(f"Audit: TF-IDF LR {res['tfidf_lr']['acc']:.4f}, SVM {res['tfidf_svm']['acc']:.4f}; probe {res['probe_skip_intensifiers_True']['acc']:.4f}")
    return {"audit_file": "outputs/audit.json"}

def run_one(spec: dict) -> bool:
    rp = run_prefix(spec)
    run_dir = LOCAL_ROOT / rp
    for sub in SUBDIRS:
        (run_dir / sub).mkdir(parents=True, exist_ok=True); (run_dir / sub / ".gitkeep").touch()
    meta = {"run_id": RUN_ID, "spec": spec, "experiment_name": EXPERIMENT_NAME, "model_name": spec.get("hf_id") or spec.get("api_model") or spec["kind"],
            "timestamp_start": datetime.now().isoformat(), "timestamp_end": None, "status": "running", "completed": False,
            "library_versions": LIB_VERSIONS, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    write_json(run_dir / "metadata" / "run_metadata.json", meta)
    try:
        info = {"train": train_run, "llm": run_llm, "audit": run_audit}[spec["kind"]](spec, run_dir, rp)
    except Exception as e:
        logger.error(f"Run {spec['spec_id']} failed: {type(e).__name__}: {e}")
        meta.update(status="failed", error=f"{type(e).__name__}: {e}", timestamp_end=datetime.now().isoformat())
        write_json(run_dir / "metadata" / "run_metadata.json", meta)
        try:
            hub_upload_file(run_dir / "metadata" / "run_metadata.json", f"{rp}/metadata/run_metadata.json")
        except Exception as up_err:
            logger.error(f"Could not upload failure metadata: {up_err}")
        # the traceback keeps the failed run's frames (and its model) alive; clear them so the GPU memory is really freed
        traceback.clear_frames(e.__traceback__)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return False
    meta.update(status="completed", completed=True, timestamp_end=datetime.now().isoformat(), **info)
    write_json(run_dir / "metadata" / "run_metadata.json", meta)
    for p in (run_dir / "checkpoints").glob("step_*"):                  # mid-run checkpoints are not final artifacts
        shutil.rmtree(p, ignore_errors=True)
    (run_dir / "checkpoints" / "latest.json").unlink(missing_ok=True)
    if spec["kind"] == "train":
        assert (run_dir / "outputs" / "logits.npz").stat().st_size > 0
    hub_upload_folder(run_dir, rp, f"run {spec['spec_id']} completed")
    (run_dir / "DONE").write_text(datetime.now().isoformat())
    hub_upload_file(run_dir / "DONE", f"{rp}/DONE")                      # DONE only after the artifacts are on the Hub
    if info.get("midrun_checkpoints_uploaded"):                             # remove mid-run checkpoints only (never final/)
        step_dirs = sorted({"/".join(f.split("/")[:-1]) for f in HUB_FILES if f.startswith(f"{rp}/checkpoints/step_")
                            or f.startswith(f"{rp}/checkpoints/oom_recovery")})
        for d in step_dirs:
            upload_with_retry(lambda d=d: API.delete_folder(path_in_repo=d, repo_id=HF_REPO_ID, repo_type="dataset",
                                                             commit_message=f"remove mid-run checkpoint {d.rsplit('/', 1)[-1]}"))
        if f"{rp}/checkpoints/latest.json" in HUB_FILES:
            upload_with_retry(lambda: API.delete_file(path_in_repo=f"{rp}/checkpoints/latest.json", repo_id=HF_REPO_ID,
                                                       repo_type="dataset", commit_message="remove latest.json of completed run"))
        HUB_FILES.difference_update({f for f in list(HUB_FILES) if any(f.startswith(d + "/") for d in step_dirs)
                                     or f == f"{rp}/checkpoints/latest.json"})
    shutil.rmtree(run_dir / "checkpoints" / "final", ignore_errors=True)  # already on the Hub; free local disk
    RUN_STATUS[spec["spec_id"]] = "done"
    logger.info(f"Run {spec['spec_id']} completed and persisted ({spec.get('encoder', spec.get('model', spec['kind']))}, "
                f"{spec.get('condition', spec.get('shots', ''))}, seed {spec.get('seed', '-')})")
    return True

pending = [s for s in SPECS if RUN_STATUS[s["spec_id"]] != "done"]
logger.info(f"{len(pending)} pending runs in stage '{STAGE}'")
FAILED_RUNS = []
for i, spec in enumerate(pending, 1):
    logger.info("=" * 60)
    logger.info(f"Run {i}/{len(pending)}: {spec['spec_id']} " + " ".join(f"{k}={v}" for k, v in spec.items() if k not in ("spec_id", "stage")))
    if not run_one(spec):
        FAILED_RUNS.append(spec["spec_id"])
logger.info(f"Training finished for stage '{STAGE}': {len(pending) - len(FAILED_RUNS)} completed, {len(FAILED_RUNS)} failed")

[2026-10-07 12:07:26] [INFO] 0 pending runs in stage 'llm_transfer'
[2026-10-07 12:07:26] [INFO] Training finished for stage 'llm_transfer': 0 completed, 0 failed


In [33]:
# =====================================================================================
# CELL 8c — ZERO-SHOT LLM TRANSFER CONTROL (runs only when STAGE == "llm_transfer"; trains nothing)
# =====================================================================================
LLMT_MODEL = "gpt-4o-2024-08-06"
LLMT_PREFIX = f"{EXPERIMENT_NAME}/llm_transfer"
LLMT_CHUNK = 250

def llmt_tasks() -> dict:
    ch = np.asarray(CEV["changed"])
    return {
        "goemotions": (list(DATA["goemotions"]["test"]["text"]), list(DATA["goemotions"]["test"]["label"]), PRESENT_CLASSES["goemotions"]),
        "meld":       (list(DATA["meld"]["test"]["text"]), list(DATA["meld"]["test"]["label"]), PRESENT_CLASSES["meld"]),
        "isear":      (list(DATA["isear_ft"]["test"]["text"]), list(DATA["isear_ft"]["test"]["label"]), PRESENT_CLASSES["isear"]),
        "ed":         (list(DATA["ed"]["test"]["text"]), list(DATA["ed"]["test"]["label"]), PRESENT_CLASSES["ed"]),
        "cev_orig":   (list(CEV["text_orig"]), [int(v) for v in CEV["label"]], PRESENT_CLASSES["cev"]),
        "cev_masked": ([CEV["text_masked"][i] for i in np.flatnonzero(ch)], [int(v) for v in np.asarray(CEV["label"])[ch]], PRESENT_CLASSES["cev"]),
        # v8: enISEAR, the 572 four-class descriptions as the readers saw them (system message lists sadness, joy, anger, fear)
        "enisear":    (list(ENI["text"]), [int(v) for v in ENI["label"]], PRESENT_CLASSES["eni"]),
    }

def llmt_system(allowed: list) -> str:
    return ("You classify the emotion of a text. Answer with exactly one word from this list: "
            + ", ".join(SETFIT[c] for c in allowed) + ".")

def llmt_parse(raw: str, allowed: list) -> int:
    found = [c for c in allowed if re.search(rf"\b{SETFIT[c]}\b", (raw or "").lower())]
    return found[0] if len(found) == 1 else -1

def run_llm_transfer() -> dict:
    from openai import OpenAI
    client = OpenAI(api_key=SECRETS["OPENAI_API_KEY"])
    def gen(system: str, text: str):
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": f"Text: {text}\nEmotion:"}]
        for attempt in range(6):
            try:
                r = client.chat.completions.create(model=LLMT_MODEL, messages=msgs, temperature=0,
                                                   max_tokens=LLM_MAX_NEW_TOKENS, seed=LLM_DEMO_SEED)
                return (r.choices[0].message.content or ""), r.model
            except Exception as e:
                wait = 5 * 2 ** attempt
                logger.warning(f"OpenAI call failed ({type(e).__name__}); retry in {wait}s"); time.sleep(wait)
        raise RuntimeError("OpenAI API failed after 6 attempts")
    summary = {"model_requested": LLMT_MODEL, "tasks": {}}
    for task, (texts, labels, allowed) in llmt_tasks().items():
        system = llmt_system(allowed)
        n_chunks = math.ceil(len(texts) / LLMT_CHUNK)
        reported = set()
        for ci in range(n_chunks):
            rel = f"{LLMT_PREFIX}/{task}/chunk_{ci:03d}.jsonl"
            lp = local_path(rel)
            if lp.exists() or rel in HUB_FILES:
                continue
            rows = []
            for i in range(ci * LLMT_CHUNK, min((ci + 1) * LLMT_CHUNK, len(texts))):
                raw, model_id = gen(system, texts[i])
                reported.add(model_id)
                rows.append({"i": i, "label": int(labels[i]), "raw": raw, "pred": llmt_parse(raw, allowed), "model_reported": model_id})
            with open(lp, "w") as f:
                f.write("\n".join(json.dumps(r) for r in rows) + "\n")
            hub_upload_file(lp, rel)
            logger.info(f"llm_transfer {task}: chunk {ci + 1}/{n_chunks} done")
        summary["tasks"][task] = {"n_items": len(texts), "n_chunks": n_chunks, "allowed": [SETFIT[c] for c in allowed],
                                  "system": system, "models_reported_this_session": sorted(reported)}
    # v8: a rerun only sends the tasks whose chunks are missing; keep the earlier entries (models reported, timestamp) for
    # every task that sent nothing in this session, so the manifest still documents the earlier calls
    old = hub_download(f"{LLMT_PREFIX}/manifest.json") if f"{LLMT_PREFIX}/manifest.json" in HUB_FILES else None
    if old is not None:
        prev = json.load(open(old))
        for task, entry in summary["tasks"].items():
            if not entry["models_reported_this_session"] and task in prev.get("tasks", {}):
                summary["tasks"][task] = {**prev["tasks"][task],
                                          "written_in_session": prev["tasks"][task].get("written_in_session", prev.get("timestamp"))}
    mp = local_path(f"{LLMT_PREFIX}/manifest.json")
    write_json(mp, {**summary, "timestamp": datetime.now().isoformat(), "library_versions": LIB_VERSIONS})
    hub_upload_file(mp, f"{LLMT_PREFIX}/manifest.json")
    return summary

if STAGE == "llm_transfer":
    LLMT_SUMMARY = run_llm_transfer()
    logger.info(f"llm_transfer complete: { {k: v['n_items'] for k, v in LLMT_SUMMARY['tasks'].items()} }")

[2026-10-07 12:10:38] [INFO] llm_transfer goemotions: chunk 1/3 done
[2026-10-07 12:13:52] [INFO] llm_transfer goemotions: chunk 2/3 done
[2026-10-07 12:15:37] [INFO] llm_transfer goemotions: chunk 3/3 done
[2026-10-07 12:18:41] [INFO] llm_transfer meld: chunk 1/6 done
[2026-10-07 12:21:43] [INFO] llm_transfer meld: chunk 2/6 done
[2026-10-07 12:24:47] [INFO] llm_transfer meld: chunk 3/6 done
[2026-10-07 12:27:58] [INFO] llm_transfer meld: chunk 4/6 done
[2026-10-07 12:31:30] [INFO] llm_transfer meld: chunk 5/6 done
[2026-10-07 12:31:59] [INFO] llm_transfer meld: chunk 6/6 done
[2026-10-07 12:35:14] [INFO] llm_transfer isear: chunk 1/3 done
[2026-10-07 12:38:18] [INFO] llm_transfer isear: chunk 2/3 done
[2026-10-07 12:40:08] [INFO] llm_transfer isear: chunk 3/3 done
[2026-10-07 12:43:20] [INFO] llm_transfer ed: chunk 1/9 done
[2026-10-07 12:46:29] [INFO] llm_transfer ed: chunk 2/9 done
[2026-10-07 12:50:11] [INFO] llm_transfer ed: chunk 3/9 done
[2026-10-07 12:53:35] [INFO] llm_transfe

## Cell 9 — Evaluation

In [34]:
# =====================================================================================
# CELL 9 — EVALUATION
# Held-out evaluation from the logits exported in Cell 8. Model selection never used test data.
# Per run: in-domain metrics, explicit/implicit test accuracy, cue-reliance index (CRI), convergence flag,
# and for source runs with transfer: unrestricted / restricted / EM prior-corrected rules, explicit/implicit
# accuracy per rule, ATC predicted accuracy, and a degenerate-output flag.
# =====================================================================================
import tempfile
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

SOURCE_PRIOR = np.bincount(DATA["setfit"]["train"]["label"], minlength=6) / len(DATA["setfit"]["train"]["label"])
STRATA_KEY = {"setfit": "setfit_test", "ed": "ed_test", "tweeteval": "tweeteval_test", "meld_target": "meld",
              "goemotions_target": "goemotions", "isear_target": "isear_target_test"}

def ece(probs: np.ndarray, y: np.ndarray, bins: int = CALIBRATION_BINS) -> float:
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1); total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(np.mean(pred[m] == y[m]) - conf[m].mean())
    return float(total)

def classwise_ece(probs: np.ndarray, y: np.ndarray, c: int, bins: int = CALIBRATION_BINS) -> float:
    p, t = probs[:, c], (y == c).astype(float)
    edges = np.linspace(0, 1, bins + 1); total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p > lo) & (p <= hi) if lo > 0 else (p >= lo) & (p <= hi)
        if m.any():
            total += m.mean() * abs(t[m].mean() - p[m].mean())
    return float(total)

def calibration(val_logits, val_y, test_logits, test_y) -> dict:
    def nll(logits, y, T):
        z = logits / T; z = z - z.max(1, keepdims=True)
        return float(-np.mean(z[np.arange(len(y)), y] - np.log(np.exp(z).sum(1))))
    opt = minimize_scalar(lambda T: nll(val_logits, val_y, T), bounds=TEMPERATURE_BOUNDS, method="bounded")
    T = float(opt.x)
    at_bound = min(abs(T - TEMPERATURE_BOUNDS[0]), abs(T - TEMPERATURE_BOUNDS[1])) < 1e-3
    out = {"temperature": T, "temperature_at_bound": bool(at_bound)}
    k = test_logits.shape[1]; onehot = np.eye(k)[test_y]
    for tag, Tt in (("before", 1.0), ("after", T)):
        p = softmax(test_logits / Tt)
        out[tag] = {"ece": ece(p, test_y), "brier": float(np.mean(((p - onehot) ** 2).sum(1))), "nll": nll(test_logits, test_y, Tt),
                    "acc": float(np.mean(p.argmax(1) == test_y)),
                    "classwise_ece": [classwise_ece(p, test_y, c) for c in range(k)]}
        edges = np.linspace(0, 1, CALIBRATION_BINS + 1); conf, pred = p.max(1), p.argmax(1); rb = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            rb.append({"lo": float(lo), "hi": float(hi), "count": int(m.sum()),
                       "acc": float(np.mean(pred[m] == test_y[m])) if m.any() else None, "conf": float(conf[m].mean()) if m.any() else None})
        out[tag]["reliability_bins"] = rb
    return out

def softmax(x: np.ndarray) -> np.ndarray:
    z = x - x.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def strat_acc(y, pred, ex) -> dict:
    y, pred, ex = np.asarray(y), np.asarray(pred), np.asarray(ex, bool)
    assert len(y) == len(pred) == len(ex)
    return {"acc_explicit": float(np.mean(pred[ex] == y[ex])) if ex.any() else None,
            "acc_implicit": float(np.mean(pred[~ex] == y[~ex])) if (~ex).any() else None,
            "n_explicit": int(ex.sum()), "n_implicit": int((~ex).sum())}

def em_prior(probs: np.ndarray, allowed: list, iters: int = 200, tol: float = 1e-8):
    """Saerens et al. (2002) EM re-estimation of target priors over the allowed classes."""
    p = probs[:, allowed]; p = p / p.sum(1, keepdims=True)
    ps = SOURCE_PRIOR[allowed] / SOURCE_PRIOR[allowed].sum()
    pt = ps.copy()
    for _ in range(iters):
        w = p * (pt / ps); w = w / w.sum(1, keepdims=True)
        new = w.mean(0)
        done = np.abs(new - pt).max() < tol
        pt = new
        if done:
            break
    return np.asarray(allowed)[w.argmax(1)], pt

def cev_restrict(logits: np.ndarray) -> np.ndarray:
    """Predictions restricted to the five classes crowd-enVENT shares with SetFit."""
    r = np.full_like(logits, -np.inf)
    r[:, PRESENT_CLASSES["cev"]] = logits[:, PRESENT_CLASSES["cev"]]
    return r.argmax(1)

def restrict_to(logits: np.ndarray, allowed: list) -> np.ndarray:
    """Predictions restricted to the classes of a target (v8; the general form of cev_restrict)."""
    r = np.full_like(logits, -np.inf)
    r[:, allowed] = logits[:, allowed]
    return r.argmax(1)

def perspective_metrics(pred, y, maj) -> dict:
    """v8. pred: predictions (-1 = no valid answer, scored as an error); y: writer labels; maj: reader majority
    (-1 tie, -2 outside the shared classes). R: unique majority equal to the writer's label; Rbar: unique majority that
    differs from it. pi: share of the errors against the writer's label, over R and Rbar, that fall on Rbar."""
    pred, y, maj = np.asarray(pred), np.asarray(y), np.asarray(maj)
    ok = maj >= 0
    R, Rb, err = ok & (maj == y), ok & (maj != y), pred != y
    return {"acc_writer": float(np.mean(~err)), "agree_reader_majority": float(np.mean(pred[ok] == maj[ok])),
            "acc_R": float(np.mean(~err[R])), "acc_Rbar": float(np.mean(~err[Rb])) if Rb.any() else None,
            "pi_error_share_Rbar": float(err[Rb].sum() / max(int(err[ok].sum()), 1)),
            "share_Rbar_items": float(Rb.sum() / max(int(ok.sum()), 1)),
            "rbar_chose_readers": float(np.mean(pred[Rb] == maj[Rb])) if Rb.any() else None,
            "rbar_chose_writer": float(np.mean(pred[Rb] == y[Rb])) if Rb.any() else None,
            "pred_share": (np.bincount(pred[pred >= 0], minlength=6) / len(pred)).round(4).tolist(),
            "n": int(len(y)), "n_R": int(R.sum()), "n_Rbar": int(Rb.sum())}

def atc(val_logits, val_y, tgt_logits, allowed) -> float:
    pv = softmax(val_logits)
    acc_v = float(np.mean(pv.argmax(1) == val_y))
    t = float(np.quantile(pv.max(1), 1 - acc_v))
    # threshold and target confidences use the same full softmax; restricting the target softmax would inflate confidence
    return float(np.mean(softmax(tgt_logits).max(1) > t))

def target_metrics(y, pred, allowed, ex) -> dict:
    y, pred = np.asarray(y), np.asarray(pred)
    share = np.bincount(pred, minlength=6) / len(pred)
    return {"acc": float(accuracy_score(y, pred)), "macro_f1": float(f1_score(y, pred, labels=allowed, average="macro", zero_division=0)),
            "pred_share": share.round(4).tolist(), "degenerate": bool(share.max() >= 0.9), **strat_acc(y, pred, ex)}

def evaluate(spec: dict, z) -> dict:
    vy, ty = z["val_y"], z["test_y"]
    vp, tp = z["val_logits"].argmax(1), z["test_logits"].argmax(1)
    m = {"val_acc": float(np.mean(vp == vy)), "test_acc": float(np.mean(tp == ty)),
         "test_macro_f1": float(f1_score(ty, tp, average="macro", zero_division=0)),
         "test_weighted_f1": float(f1_score(ty, tp, average="weighted", zero_division=0)),
         "test_f1_per_class": f1_score(ty, tp, average=None, zero_division=0).round(4).tolist(),
         "converged": bool(np.mean(vp == vy) >= CONVERGED_VAL_ACC),
         "test_macro_precision": float(precision_score(ty, tp, average="macro", zero_division=0)),
         "test_macro_recall": float(recall_score(ty, tp, average="macro", zero_division=0)),
         "test_confusion": confusion_matrix(ty, tp, labels=list(range(z["test_logits"].shape[1]))).tolist(),
         "calibration": calibration(z["val_logits"], vy, z["test_logits"], ty)}
    base = "setfit" if spec["corpus"].startswith("setfit") else spec["corpus"]
    # if spec["corpus"] in ("meld_target", "goemotions_target", "isear_target", "ed"):   # macro-F1 over the classes present
    if spec["corpus"] in ("meld_target", "goemotions_target", "isear_target", "ed", "cev_target"):   # macro-F1 over present classes
        allowed = PRESENT_CLASSES[spec["corpus"].replace("_target", "")]
        m["test_macro_f1"] = float(f1_score(ty, tp, labels=allowed, average="macro", zero_division=0))
    if base in STRATA_KEY:
        m.update({f"test_{k}": v for k, v in strat_acc(ty, tp, EXPLICIT[STRATA_KEY[base]]).items()})
    if "cri_orig_logits" in z.files:
        o, c, r = (z[f"cri_{k}_logits"].argmax(1) for k in ("orig", "cue", "random"))
        m["cri_flip_cue"], m["cri_flip_random"] = float(np.mean(c != o)), float(np.mean(r != o))
        m["cri"] = m["cri_flip_cue"] - m["cri_flip_random"]; m["cri_n"] = int(len(o))
    for tgt in TRANSFER_TARGETS:
        if f"transfer_{tgt}_logits" not in z.files:
            continue
        lg, y = z[f"transfer_{tgt}_logits"], z[f"transfer_{tgt}_y"]
        allowed, ex = PRESENT_CLASSES[tgt], EXPLICIT[tgt if tgt != "ed" else "ed_test"]
        restricted = np.full_like(lg, -np.inf); restricted[:, allowed] = lg[:, allowed]
        em_pred, pt = em_prior(softmax(lg), allowed)
        m[f"transfer_{tgt}"] = {"unrestricted": target_metrics(y, lg.argmax(1), allowed, ex),
                                "restricted": target_metrics(y, restricted.argmax(1), allowed, ex),
                                "prior_em": {**target_metrics(y, em_pred, allowed, ex), "estimated_prior": pt.round(4).tolist()},
                                "true_prior": [float(np.mean(y == c)) for c in allowed],
                                "atc_predicted_acc": atc(z["val_logits"], vy, lg, allowed)}
    if "eni_logits" in z.files:                        # v8: enISEAR (four classes), scored against the writer's label
        m["enisear"] = perspective_metrics(restrict_to(z["eni_logits"], PRESENT_CLASSES["eni"]), ENI["label"], ENI["reader_majority"])
    return m

def evaluate_llm(spec: dict, rp: str) -> dict:
    rows = []
    for ci in range(math.ceil(len(DATA["setfit"]["test"]["text"]) / LLM_CHUNK)):
        rel = f"{rp}/outputs/llm_chunk_{ci:03d}.jsonl"
        lp = LOCAL_ROOT / rel
        path = lp if lp.exists() else hub_download(rel)
        assert path is not None, f"missing LLM chunk {rel}"
        rows += [json.loads(l) for l in open(path) if l.strip()]
    assert len(rows) == len(DATA["setfit"]["test"]["text"]) and len({r["i"] for r in rows}) == len(rows), "LLM outputs incomplete"
    y = np.array([r["label"] for r in rows]); p = np.array([r["pred"] for r in rows])
    return {"n": int(len(rows)), "acc": float(np.mean(p == y)), "format_failure_rate": float(np.mean(p == -1)),
            "macro_f1": float(f1_score(y, p, labels=list(range(6)), average="macro", zero_division=0)),
            "acc_on_parsed_only": float(np.mean(p[p >= 0] == y[p >= 0])) if (p >= 0).any() else None,
            "pred_share": (np.bincount(p[p >= 0], minlength=6) / len(p)).round(4).tolist(),
            "models_reported": sorted({r["model_reported"] for r in rows}), **strat_acc(y, p, EXPLICIT["setfit_test"])}

def evaluate_stage(specs: list) -> int:
    todo = [s for s in specs if RUN_STATUS.get(s["spec_id"]) == "done" and f"{run_prefix(s)}/metrics/metrics.json" not in HUB_FILES]
    logger.info(f"Evaluating {len(todo)} runs without metrics")
    batch_root = Path(tempfile.mkdtemp(prefix="metrics_batch_"))
    n_done = 0
    for i, s in enumerate(todo, 1):
        rp = run_prefix(s)
        if s["kind"] == "llm":
            metrics = {"spec": s, **evaluate_llm(s, rp), "evaluated": datetime.now().isoformat()}
        elif s["kind"] == "audit":
            lp = LOCAL_ROOT / rp / "outputs" / "audit.json"
            path = lp if lp.exists() else hub_download(f"{rp}/outputs/audit.json")
            with open(path) as fh:
                metrics = {"spec": s, "audit": json.load(fh), "evaluated": datetime.now().isoformat()}
        else:
            lp = LOCAL_ROOT / rp / "outputs" / "logits.npz"
            path = lp if lp.exists() else hub_download(f"{rp}/outputs/logits.npz")
            assert path is not None and Path(path).exists(), f"logits missing for completed run {s['spec_id']}"
            with np.load(path) as z:
                metrics = {"spec": s, **evaluate(s, z), "evaluated": datetime.now().isoformat()}
        out = batch_root / s["spec_id"] / "metrics" / "metrics.json"; out.parent.mkdir(parents=True, exist_ok=True)
        write_json(out, metrics)
        local = LOCAL_ROOT / rp / "metrics" / "metrics.json"; local.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(out, local)
        n_done += 1
        if n_done % 100 == 0 or i == len(todo):                            # one Hub commit per 100 runs
            hub_upload_folder(batch_root, f"{EXPERIMENT_NAME}/{STAGE}/runs", f"metrics for {n_done} runs ({STAGE})")
            shutil.rmtree(batch_root, ignore_errors=True); batch_root.mkdir(parents=True)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return n_done

N_EVALUATED = evaluate_stage(SPECS) if STAGE != "analysis" else 0
logger.info(f"Evaluation complete: {N_EVALUATED} runs evaluated in stage '{STAGE}'")


[2026-10-07 14:01:18] [INFO] Evaluating 0 runs without metrics
[2026-10-07 14:01:18] [INFO] Evaluation complete: 0 runs evaluated in stage 'llm_transfer'


## Cell 9b — Aggregation and pre-registered hypothesis tests

In [35]:
# =====================================================================================
# CELL 9b — AGGREGATION AND PRE-REGISTERED HYPOTHESIS TESTS (multiseed aggregation cell)
# Reads the metrics of every evaluated run of every stage from the Hub and computes, where the data exist:
#   tuning selection -> shared/tuned_recipes.json | seed tables (mean +- sample SD, n) | item-paired and seed-paired
#   prompt contrasts (exact McNemar, Wald CI, MDES, equivalence margin, t / sign / permutation, random-effects)
#   H1 explicit share | H2 explicit vs implicit transfer (GEE) | H3 CRI vs transfer (OLS, MixedLM, cluster bootstrap,
#   ATC comparison) | H4 prompt effect by stratum | H5 cue masking vs random masking | convergence (Fisher).
# Every analysis is skipped with a log message when its inputs are not yet available.
# =====================================================================================
import warnings
from scipy import stats as sps
import statsmodels.api as sm
import statsmodels.formula.api as smf

ANALYSIS_DIR = LOCAL_ROOT / ANALYSIS_PREFIX
(ANALYSIS_DIR / "metrics").mkdir(parents=True, exist_ok=True)
(ANALYSIS_DIR / "figures").mkdir(parents=True, exist_ok=True)
ANALYSIS = {}

def load_all_metrics() -> pd.DataFrame:
    rows = []
    for f in sorted(HUB_FILES):
        if not (f.endswith("/metrics/metrics.json") and "/runs/" in f):
            continue
        lp = LOCAL_ROOT / f
        path = lp if lp.exists() else hub_download(f)
        with open(path) as fh:
            m = json.load(fh)
        meta_path = f.replace("/metrics/metrics.json", "/metadata/run_metadata.json")
        lm = LOCAL_ROOT / meta_path
        mp = lm if lm.exists() else hub_download(meta_path)
        meta = json.load(open(mp)) if mp is not None else {}
        extra = {k: meta.get(k) for k in ("train_seconds", "test_items_per_second", "trainable_parameters", "total_parameters",
                                          "epochs_run", "steps_run", "precision_used", "prefix", "gpu")}
        rows.append({**m["spec"], **extra, **{k: v for k, v in m.items() if k != "spec"}})
    df = pd.DataFrame(rows)
    logger.info(f"Loaded metrics for {len(df)} evaluated runs across stages")
    return df

def load_test_correct(spec_row) -> np.ndarray:
    rp = f"{EXPERIMENT_NAME}/{spec_row['stage']}/runs/{spec_row['spec_id']}"
    lp = LOCAL_ROOT / rp / "outputs" / "logits.npz"
    path = lp if lp.exists() else hub_download(f"{rp}/outputs/logits.npz")
    with np.load(path) as z:
        return z["test_logits"].argmax(1) == z["test_y"]

def wilson(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def paired_item_stats(ca: np.ndarray, cb: np.ndarray) -> dict:
    ca, cb = np.asarray(ca, bool), np.asarray(cb, bool)
    n = len(ca); b = int(np.sum(ca & ~cb)); c = int(np.sum(~ca & cb))
    d = (b - c) / n; psi = (b + c) / n
    se = math.sqrt(max(psi - d * d, 0.0) / n)
    p = float(sps.binomtest(b, b + c, 0.5).pvalue) if b + c else 1.0
    return {"n": n, "b": b, "c": c, "delta_pp": 100 * d, "ci95_pp": [100 * (d - 1.96 * se), 100 * (d + 1.96 * se)],
            "ci90_pp": [100 * (d - 1.645 * se), 100 * (d + 1.645 * se)], "p_exact": p,
            "mdes80_pp": 100 * (1.959964 + 0.841621) * math.sqrt(psi / n),
            "equivalence_margin_pp": 100 * max(abs(d - 1.645 * se), abs(d + 1.645 * se))}

def sign_flip_p(d: np.ndarray, seed: int = 0) -> float:
    d = np.asarray(d, float); obs = abs(d.mean())
    if len(d) <= N_PERMUTATIONS_MAX_EXACT:
        signs = np.array(list(itertools.product([-1, 1], repeat=len(d))))
    else:
        signs = np.random.default_rng(seed).choice([-1, 1], size=(100000, len(d)))
    return float(np.mean(np.abs((signs * d).mean(1)) >= obs - 1e-12))

def seed_paired(a: np.ndarray, b: np.ndarray) -> dict:
    d = np.asarray(a, float) - np.asarray(b, float); n = len(d)
    mean, sd = float(d.mean()), float(d.std(ddof=1)) if n > 1 else float("nan")
    t = sps.t.ppf(0.975, n - 1) if n > 1 else float("nan")
    nz = d[d != 0]
    return {"n": n, "mean": mean, "sd": sd, "ci95": [mean - t * sd / math.sqrt(n), mean + t * sd / math.sqrt(n)],
            "p_t": float(sps.ttest_1samp(d, 0).pvalue) if n > 1 and sd > 0 else float("nan"),
            "p_sign": float(sps.binomtest(int((nz > 0).sum()), len(nz), 0.5).pvalue) if len(nz) else 1.0,
            "p_perm": sign_flip_p(d), "var_mean": sd * sd / n if n > 1 else float("nan")}

def random_effects(effects, variances) -> dict:
    y, v = np.asarray(effects, float), np.asarray(variances, float)
    w = 1 / v; fixed = np.sum(w * y) / np.sum(w); q = float(np.sum(w * (y - fixed) ** 2)); k = len(y)
    cc = np.sum(w) - np.sum(w ** 2) / np.sum(w)
    tau2 = max(0.0, (q - (k - 1)) / cc) if cc > 0 else 0.0
    wr = 1 / (v + tau2); mu = float(np.sum(wr * y) / np.sum(wr)); se = float(math.sqrt(1 / np.sum(wr)))
    return {"k": k, "mu": mu, "se": se, "ci95": [mu - 1.96 * se, mu + 1.96 * se], "ci90": [mu - 1.645 * se, mu + 1.645 * se],
            "tau2": tau2, "Q": q, "I2": max(0.0, (q - (k - 1)) / q) if q > 0 else 0.0}

def holm(pvals: dict) -> dict:
    items = sorted(pvals.items(), key=lambda kv: kv[1]); m = len(items); out, running = {}, 0.0
    for i, (k, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p)); out[k] = running
    return out

RECIPE_KEYS = ["stage", "corpus", "encoder", "placement", "head", "text_transform", "transform_eval", "fewshot_k",
               "peft_method", "rebalance", "lr", "epochs"]

def tex_table(df: pd.DataFrame, cols: list, path: Path, fmt: dict) -> None:
    """Write booktabs body rows (no environment) so the paper can \\input them; values formatted per column."""
    lines = []
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            cells.append(fmt[c](v) if c in fmt and v is not None and not (isinstance(v, float) and np.isnan(v)) else
                         ("--" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v).replace("_", "\\_")))
        lines.append(" & ".join(cells) + " \\\\")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n")

pct = lambda v: f"{100 * v:.2f}"
f4 = lambda v: f"{v:.4f}"
pm_pct = lambda m, s: f"{100 * m:.2f} $\\pm$ {100 * s:.2f}" if s == s else f"{100 * m:.2f}"

def nontrain_tables(A: dict, D_LLM: pd.DataFrame, D_AUDIT: pd.DataFrame, TEX: Path) -> None:
    if len(D_LLM):
        lt = D_LLM[["model", "shots", "acc", "macro_f1", "format_failure_rate", "acc_on_parsed_only", "models_reported"]].sort_values(["model", "shots"])
        tex_table(lt, ["model", "shots", "acc", "macro_f1", "format_failure_rate"], TEX / "llm.tex", {"acc": pct, "macro_f1": f4, "format_failure_rate": pct})
        A["table_llm"] = lt.to_dict("records")
    if len(D_AUDIT):
        A["audit"] = D_AUDIT.iloc[-1]["audit"]

def cev_analyses(df: pd.DataFrame) -> dict:
    """crowd-enVENT analyses (pre-registration amendment).
    Confirmatory: E1 author-masked minimal pairs with a target-trained control (difference in differences on the same items);
    E2 CRI versus the minimal-pair drop; E3 writer, reader, and model on identical texts.
    Exploratory: E4 appraisal dimensions of model errors; E5 same-configuration reruns (cev stages versus main/shortcut)."""
    out = {}
    src, tgt = df[df.stage.isin(CEV_STAGES)], df[df.stage == "cev_target"]
    if not len(src) and not len(tgt):
        return out
    y, ch, rp, mj = CEV["label"], CEV["changed"], CEV["reader_pos"], CEV["reader_majority"]
    ch_r, ok = ch[rp], mj >= 0
    yr = y[rp]
    def npz(r):
        p = f"{EXPERIMENT_NAME}/{r.stage}/runs/{r.spec_id}"
        lp = LOCAL_ROOT / p / "outputs" / "logits.npz"
        return np.load(lp if lp.exists() else hub_download(f"{p}/outputs/logits.npz"))
    S, T = [], []
    for _, r in src.iterrows():
        with npz(r) as z:
            S.append((r, cev_restrict(z["cev_orig_logits"]), cev_restrict(z["cev_masked_logits"])))
    for _, r in tgt.iterrows():
        with npz(r) as z:
            T.append((r, cev_restrict(z["test_logits"]), cev_restrict(z["cevtest_masked_logits"])))
    TEX = ANALYSIS_DIR / "tables"
    # ---- E1: minimal pairs (863 author-masked texts); control = target-trained models on the 163 masked reader-set items
    if S:
        d = pd.DataFrame([{"stage": r.stage, "encoder": r.encoder, "condition": r.condition, "seed": int(r.seed),
                           "text_transform": r.text_transform, "cri": r.get("cri"), "in_acc": float(r.test_acc),
                           "drop863": 100 * (np.mean(po[ch] == y[ch]) - np.mean(pm[ch] == y[ch])),
                           "drop163": 100 * (np.mean(po[rp][ch_r] == yr[ch_r]) - np.mean(pm[rp][ch_r] == yr[ch_r]))} for r, po, pm in S])
        base = d[d.stage == "cev_source"]
        e1, per = {"n_pairs": int(ch.sum())}, []
        for enc, g in base.groupby("encoder"):
            if len(g) >= 2:
                per.append((enc, float(g.drop863.mean()), float(g.drop863.var(ddof=1) / len(g)), len(g)))
        e1["source_drop_pp_by_encoder"] = {e: {"mean": m_, "n_runs": n} for e, m_, _, n in per}
        if len(per) >= 2:
            e1["source_drop_pp_pooled"] = random_effects([p[1] for p in per], [max(p[2], 1e-9) for p in per])
        if T:
            td = pd.DataFrame([{"encoder": r.encoder, "seed": int(r.seed), "acc500": float(np.mean(po == yr)),
                                "drop163": 100 * (np.mean(po[ch_r] == yr[ch_r]) - np.mean(pm[ch_r] == yr[ch_r]))} for r, po, pm in T])
            did = []
            for enc in sorted(set(base.encoder) & set(td.encoder)):
                a, b = base[base.encoder == enc].drop163.to_numpy(), td[td.encoder == enc].drop163.to_numpy()
                if len(a) >= 2 and len(b) >= 2:
                    did.append((enc, float(a.mean() - b.mean()), float(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))))
            e1["target_drop163_pp_by_encoder"] = td.groupby("encoder").drop163.mean().to_dict()
            e1["target_acc500_by_encoder"] = td.groupby("encoder").acc500.mean().to_dict()
            e1["did_source_minus_target_pp"] = {e: v for e, v, _ in did}
            if len(did) >= 2:
                e1["did_pooled"] = random_effects([x[1] for x in did], [max(x[2], 1e-9) for x in did])
        out["cev_E1_minimal_pairs"] = e1
        # ---- E2: does the label-free CRI (our cue vocabulary) predict the drop on the authors' independent masking?
        d2 = d.dropna(subset=["cri"])
        e2 = {"n_runs": int(len(d2)), "n_encoders": int(d2.encoder.nunique())}
        if len(d2) >= 10 and d2.encoder.nunique() >= 2 and d2.cri.std() > 0:
            zf = lambda s: (s - s.mean()) / (s.std() + 1e-12)
            d2 = d2.assign(z_cri=zf(d2.cri), z_in=zf(d2.in_acc))
            ols = smf.ols("drop863 ~ z_in + z_cri", data=d2).fit()
            b = float(ols.params["z_cri"])
            e2.update(spearman_cri_drop=float(sps.spearmanr(d2.cri, d2.drop863)[0]), beta_cri_ols_pp_per_sd=b,
                      p_ols_one_sided=float(ols.pvalues["z_cri"] / 2 if b > 0 else 1 - ols.pvalues["z_cri"] / 2))
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    mix = smf.mixedlm("drop863 ~ z_in + z_cri", data=d2, groups=d2["encoder"]).fit(reml=True)
                ci = mix.conf_int().loc["z_cri"].tolist()
                if np.all(np.isfinite(ci)):
                    e2.update(beta_cri_mixed=float(mix.params["z_cri"]), ci95_mixed=ci,
                              p_mixed_one_sided=float(sps.norm.sf(mix.tvalues["z_cri"])))
                else:
                    e2["mixed_model_note"] = "singular fit; not used for inference"
            except Exception as e:
                logger.error(f"E2 mixed model failed: {e}")
        out["cev_E2_cri_validation"] = e2
        # ---- E3: writer, reader, and model on the same 500 texts (source models without masked training)
        g_ = CEV["reader_guesses"]; inset = g_ >= 0
        e3 = {"n_items": int(len(rp)), "n_items_with_reader_majority": int(ok.sum()), "n_masked_items": int(ch_r.sum()),
              "human_individual_acc_13way": float(CEV["reader_item_acc"].mean()),
              "human_individual_acc_inset_guesses": float((g_ == yr[:, None])[inset].mean()),
              "human_majority_acc": float(np.mean(mj[ok] == yr[ok])),
              "human_individual_acc_masked_items": float(CEV["reader_item_acc"][ch_r].mean()),
              "human_individual_acc_unmasked_items": float(CEV["reader_item_acc"][~ch_r].mean()),
              "note": ("Readers chose among 13 emotions; models are restricted to the five shared classes. "
                       "Human and model accuracies are therefore not directly comparable; the in-set figure conditions "
                       "readers on choosing one of the five classes.")}
        rows, conf_m = [], np.zeros((6, 6))
        for r, po, pm in S:
            if r.stage != "cev_source":
                continue
            p = po[rp]
            rows.append({"encoder": r.encoder, "condition": r.condition, "seed": int(r.seed),
                         "acc_writer": float(np.mean(p == yr)), "acc_writer_masked_items": float(np.mean(p[ch_r] == yr[ch_r])),
                         "acc_writer_unmasked_items": float(np.mean(p[~ch_r] == yr[~ch_r])),
                         "agree_reader_majority": float(np.mean(p[ok] == mj[ok]))})
            for a_, b_ in zip(yr, p):
                conf_m[a_, b_] += 1
        if rows:
            mt = pd.DataFrame(rows).groupby(["encoder", "condition"]).mean(numeric_only=True).reset_index()
            e3["models"] = mt.drop(columns=["seed"]).to_dict("records")
            tex_table(mt, ["encoder", "condition", "acc_writer", "acc_writer_masked_items", "acc_writer_unmasked_items", "agree_reader_majority"],
                      TEX / "cev_perspective.tex", {c: pct for c in ["acc_writer", "acc_writer_masked_items", "acc_writer_unmasked_items", "agree_reader_majority"]})
            # do models fail where readers fail? item-level GEE clustered by item (source runs without masked training)
            parts = [pd.DataFrame({"correct": (po[rp] == yr).astype(int), "item": np.arange(len(rp)), "masked": ch_r.astype(int),
                                   "reader_acc": CEV["reader_item_acc"]}) for r, po, pm in S if r.stage == "cev_source"]
            dd = pd.concat(parts, ignore_index=True)
            dd["z_reader_acc"] = (dd.reader_acc - dd.reader_acc.mean()) / (dd.reader_acc.std() + 1e-12)
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    fit = smf.gee("correct ~ z_reader_acc + masked", groups="item", data=dd, family=sm.families.Binomial(),
                                  cov_struct=sm.cov_struct.Exchangeable()).fit()
                e3["gee_model_correct_on_reader_accuracy"] = {k: {"log_or": float(fit.params[k]), "ci95": fit.conf_int().loc[k].tolist(),
                                                                  "p": float(fit.pvalues[k])} for k in ("z_reader_acc", "masked")}
            except Exception as e:
                logger.error(f"E3 GEE failed: {e}")
            # confusion similarity over the 20 off-diagonal cells of the five shared classes (writer label x predicted/guessed)
            cls = PRESENT_CLASSES["cev"]
            conf_h = np.zeros((6, 6))
            for a_, gl in zip(yr, g_):
                for gg in gl:
                    if gg >= 0:
                        conf_h[a_, gg] += 1
            norm = lambda M: (M / np.maximum(M.sum(1, keepdims=True), 1))[np.ix_(cls, cls)]
            off = ~np.eye(len(cls), dtype=bool)
            e3["confusion_similarity_spearman_offdiag"] = float(sps.spearmanr(norm(conf_m)[off], norm(conf_h)[off])[0])
        out["cev_E3_perspective"] = e3
        # ---- E4 (exploratory): which writer appraisals characterize model errors? item-clustered GEE with class fixed effects
        e4 = {}
        none_runs = [(r, po) for r, po, pm in S if r.stage == "cev_source" and r.condition == "none"]
        if none_runs:
            A_ = CEV["appraisal"]; Z = (A_ - A_.mean(0)) / (A_.std(0) + 1e-12)
            parts = []
            for r, po in none_runs:
                f_ = pd.DataFrame(Z, columns=CEV_APPRAISALS)
                f_["correct"] = (po == y).astype(int); f_["item"] = np.arange(len(y)); f_["label"] = y; f_["masked"] = ch.astype(int)
                parts.append(f_)
            dd = pd.concat(parts, ignore_index=True)
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    fit = smf.gee("correct ~ " + " + ".join(CEV_APPRAISALS) + " + C(label) + masked", groups="item", data=dd,
                                  family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()).fit()
                ci = fit.conf_int()
                pv = {k: float(fit.pvalues[k]) for k in CEV_APPRAISALS}
                ph = holm(pv)
                e4 = {k: {"log_or_per_sd": float(fit.params[k]), "ci95": ci.loc[k].tolist(), "p": pv[k], "p_holm": ph[k]} for k in CEV_APPRAISALS}
                e4["_masked"] = {"log_or": float(fit.params["masked"]), "ci95": ci.loc["masked"].tolist(), "p": float(fit.pvalues["masked"])}
                e4["_note"] = "exploratory; adjusted for emotion class; Holm over the 21 appraisal dimensions"
            except Exception as e:
                logger.error(f"E4 GEE failed: {e}")
        out["cev_E4_appraisals_exploratory"] = e4
    # ---- E5 (exploratory): same configuration trained twice (cev stages vs main/shortcut), same-GPU pairs separately
    keys = ["corpus", "encoder", "condition", "seed", "lr", "epochs", "placement", "head", "text_transform", "transform_eval"]
    ref = df[df.stage.isin(["main", "shortcut"])]
    pairs = []
    for _, r in src.iterrows():
        mrow = ref
        for k in keys:
            mrow = mrow[mrow[k] == r[k]]
        if len(mrow) == 1:
            m0 = mrow.iloc[0]
            c1, c0 = load_test_correct(r), load_test_correct(m0)
            pairs.append({"encoder": r.encoder, "same_gpu": bool(r.get("gpu") == m0.get("gpu")),
                          "abs_diff_pp": float(100 * abs(r.test_acc - m0.test_acc)), "item_disagreement": float(np.mean(c1 != c0))})
    if pairs:
        pp = pd.DataFrame(pairs)
        out["cev_E5_rerun_variation_exploratory"] = {
            "n_pairs": int(len(pp)), "n_same_gpu": int(pp.same_gpu.sum()),
            "same_gpu": pp[pp.same_gpu][["abs_diff_pp", "item_disagreement"]].describe().to_dict() if pp.same_gpu.any() else None,
            "all": pp[["abs_diff_pp", "item_disagreement"]].describe().to_dict()}
    return out

def aggregate(df: pd.DataFrame) -> dict:
    A = {}
    if df.empty:
        logger.warning("No evaluated runs yet; nothing to aggregate.")
        return A
    df = df.copy()
    df["kind"] = df["kind"].fillna("train") if "kind" in df else "train"
    D_LLM, D_AUDIT = df[df.kind == "llm"], df[df.kind == "audit"]
    df = df[df.kind == "train"].copy()
    for k in ["text_transform", "fewshot_k", "peft_method", "rebalance"]:
        if k not in df:
            df[k] = None
        df[k] = df[k].astype(object).where(df[k].notna(), "none")
    TEX = ANALYSIS_DIR / "tables"
    if not len(df):                                              # no training runs yet (e.g. after the audit stage)
        nontrain_tables(A, D_LLM, D_AUDIT, TEX)
        A["H1_explicit_share"] = {k: {"share": float(v.mean()), "n": int(len(v)), "wilson95": wilson(int(v.sum()), len(v))}
                                  for k, v in EXPLICIT.items()}
        logger.info("No training runs evaluated yet; only audit/LLM tables and H1 computed.")
        return A
    # ---- recipe selection (tuning: main encoders; lr_sweep: DeBERTa/ModernBERT), mean over both conditions ------
    selected = dict(TUNED_RECIPES or {})
    for st, encs in RECIPE_SELECTION_STAGES.items():
        tun = df[df.stage == st]
        if not len(tun):
            continue
        exp_n = len(expand_stage(st, None))
        if len(tun) < exp_n:
            logger.info(f"{st} incomplete ({len(tun)}/{exp_n} runs evaluated); recipes not selected yet.")
            continue
        g = tun.groupby(["encoder", "lr", "epochs"]).val_acc.agg(["mean", "std", "count"]).reset_index()
        A[f"{st}_table"] = g.to_dict("records")
        for enc, sub in g.groupby("encoder"):
            r = sub.sort_values(["mean", "lr"], ascending=[False, True]).iloc[0]      # tie -> smaller learning rate
            rec = {"lr": float(r.lr), "epochs": int(r.epochs), "val_acc_mean": float(r["mean"]), "val_acc_sd": float(r["std"]),
                   "n_runs": int(r["count"]), "selected_in": st}
            if enc in selected and (selected[enc]["lr"] != rec["lr"] or selected[enc]["epochs"] != rec["epochs"]):
                raise RuntimeError(f"Tuned recipe for {enc} on the Hub differs from the recomputed selection. Investigate.")
            selected[enc] = rec
    if selected and selected != (TUNED_RECIPES or {}):
        lp = local_path(TUNED_RECIPES_PATH); write_json(lp, selected); hub_upload_file(lp, TUNED_RECIPES_PATH)
        logger.info(f"Tuned recipes written to the Hub: { {k: (v['lr'], v['epochs']) for k, v in selected.items()} }")
    A["tuned_recipes"] = selected
    # ---- seed tables -------------------------------------------------------------------------------------------
    keys = ["stage", "corpus", "encoder", "condition", "placement", "head", "text_transform", "transform_eval", "fewshot_k",
            "peft_method", "rebalance"]
    tab = df.groupby(keys).agg(n=("test_acc", "size"), acc_mean=("test_acc", "mean"), acc_sd=("test_acc", "std"),
                               mf1_mean=("test_macro_f1", "mean"), mf1_sd=("test_macro_f1", "std"),
                               converged=("converged", "sum")).reset_index()
    A["seed_table"] = tab.to_dict("records")
    tab.to_csv(ANALYSIS_DIR / "metrics" / "seed_table.csv", index=False)
    # ---- prompt contrasts (condition vs none, same recipe and seed) ------------------------------------------------
    item_rows, seed_rows = [], []
    base = df[df.condition == "none"].set_index(RECIPE_KEYS + ["seed"])
    # the cev and eni stages repeat main/shortcut configurations: they never enter the prompt-effect statistics
    for key, grp in df[(df.condition != "none") & ~df.stage.isin(CEV_STAGES + ["cev_target", "cev_third"] + ENI_EXPORT_STAGES)].groupby(RECIPE_KEYS + ["condition"]):
        rk = key[:-1]; cond = key[-1]
        pa, pb = [], []
        for _, r in grp.iterrows():
            k0 = tuple(rk) + (r.seed,)
            if k0 not in base.index:
                continue
            r0 = base.loc[k0]
            pa.append(r.test_acc); pb.append(r0.test_acc)
            if r.stage in ("main", "intra", "shortcut", "scaling", "target"):
                ca, cb = load_test_correct(r), load_test_correct(r0.to_dict() | {"stage": k0[0], "spec_id": r0.spec_id})
                row = {**dict(zip(RECIPE_KEYS, rk)), "condition": cond, "seed": int(r.seed), **paired_item_stats(ca, cb)}
                sk = STRATA_KEY.get("setfit" if str(r.corpus).startswith("setfit") else r.corpus)
                if sk and len(EXPLICIT[sk]) == len(ca) and not r.transform_eval:
                    ex = EXPLICIT[sk]
                    row["explicit"] = paired_item_stats(ca[ex], cb[ex]) if ex.any() else None
                    row["implicit"] = paired_item_stats(ca[~ex], cb[~ex]) if (~ex).any() else None
                item_rows.append(row)
        if len(pa) >= 3:
            seed_rows.append({**dict(zip(RECIPE_KEYS, rk)), "condition": cond,
                              **seed_paired(100 * np.array(pa), 100 * np.array(pb))})
    A["item_paired"] = item_rows; A["seed_paired"] = seed_rows
    pooled = {}
    sp = pd.DataFrame(seed_rows)
    if len(sp):
        for key, g in sp.groupby(["stage", "corpus", "condition", "placement", "head", "text_transform", "transform_eval", "fewshot_k",
                                  "peft_method", "rebalance"]):
            g = g[g.var_mean > 0]
            if len(g) >= 2:
                pooled["|".join(map(str, key))] = random_effects(g["mean"], g["var_mean"])
    A["random_effects"] = pooled
    # ---- H1: explicit share -------------------------------------------------------------------------------------
    A["H1_explicit_share"] = {k: {"share": float(v.mean()), "n": int(len(v)), "wilson95": wilson(int(v.sum()), len(v))}
                              for k, v in EXPLICIT.items()}
    # ---- H2: explicit vs implicit transfer accuracy (GEE over item correctness, clustered by run) ----------------
    src = df[df.stage.isin(["main"]) & df.condition.isin(["none", "A"])]
    h2, h2p = {}, {}
    for tgt in TRANSFER_TARGETS:
        col = f"transfer_{tgt}"
        if col not in src or src[col].isna().all():
            continue
        ys, xs, gs = [], [], []
        for _, r in src[src[col].notna()].iterrows():
            rp = f"{EXPERIMENT_NAME}/{r.stage}/runs/{r.spec_id}"
            lp = LOCAL_ROOT / rp / "outputs" / "logits.npz"
            with np.load(lp if lp.exists() else hub_download(f"{rp}/outputs/logits.npz")) as z:
                lg, y = z[f"transfer_{tgt}_logits"], z[f"transfer_{tgt}_y"]
            allowed = PRESENT_CLASSES[tgt]
            rl = np.full_like(lg, -np.inf); rl[:, allowed] = lg[:, allowed]
            ex = EXPLICIT[tgt if tgt != "ed" else "ed_test"]
            # cluster by ITEM: explicitness is an item-level property and every item recurs once per run, so clustering by
            # run would treat the repeated items as independent and make the interval far too narrow
            ys.append((rl.argmax(1) == y).astype(int)); xs.append(ex.astype(int)); gs.append(np.arange(len(y)))
        y_all, x_all, g_all = np.concatenate(ys), np.concatenate(xs), np.concatenate(gs)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fit = sm.GEE(y_all, sm.add_constant(x_all), groups=g_all, family=sm.families.Binomial(),
                             cov_struct=sm.cov_struct.Exchangeable()).fit()
            est, lo, hi = float(fit.params[1]), float(fit.conf_int()[1][0]), float(fit.conf_int()[1][1])
            p_one = float(sps.norm.sf(fit.params[1] / fit.bse[1]))          # H2 predicts explicit > implicit
        except Exception as e:
            logger.error(f"H2 GEE failed for {tgt}: {e}"); continue
        h2[tgt] = {"n_runs": len(ys), "acc_explicit": float(y_all[x_all == 1].mean()), "acc_implicit": float(y_all[x_all == 0].mean()),
                   "log_or_explicit": est, "ci95": [lo, hi], "p_one_sided": p_one}
        h2p[tgt] = p_one
    for t, p in holm(h2p).items():
        h2[t]["p_holm"] = p
    A["H2_stratified_transfer"] = h2
    # ---- H3: CRI predicts transfer beyond in-domain accuracy ---------------------------------------------------------
    h3, h3p = {}, {}
    # in-domain accuracy must be measured on the original test set, so runs evaluated on masked test text are excluded
    cand = (df[df.stage.isin(["main", "shortcut"]) & df["cri"].notna() & (df.transform_eval != True)] if "cri" in df else df.iloc[0:0])
    for tgt in TRANSFER_TARGETS:
        col = f"transfer_{tgt}"
        if col not in cand or cand[col].isna().all():
            continue
        c = cand[cand[col].notna()].copy()
        c["transfer_acc"] = c[col].map(lambda d: d["restricted"]["acc"]); c["atc"] = c[col].map(lambda d: d["atc_predicted_acc"])
        if len(c) < 10 or c.encoder.nunique() < 2:
            logger.info(f"H3 {tgt}: {len(c)} runs from {c.encoder.nunique()} encoders; waiting for more runs")
            continue
        z = lambda s: (s - s.mean()) / (s.std() + 1e-12)
        c["z_in"], c["z_cri"] = z(c.test_acc), z(c.cri)
        ols = smf.ols("transfer_acc ~ z_in + z_cri", data=c).fit()
        r2_in = smf.ols("transfer_acc ~ z_in", data=c).fit().rsquared
        res = {"n_runs": int(len(c)), "n_encoders": int(c.encoder.nunique()), "beta_cri_ols": float(ols.params["z_cri"]),
               "beta_in_ols": float(ols.params["z_in"]), "r2_in_only": float(r2_in), "r2_in_plus_cri": float(ols.rsquared),
               "spearman_cri": float(sps.spearmanr(c.cri, c.transfer_acc)[0]),
               "spearman_in_domain": float(sps.spearmanr(c.test_acc, c.transfer_acc)[0]),
               "spearman_atc": float(sps.spearmanr(c.atc, c.transfer_acc)[0])}
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                mix = smf.mixedlm("transfer_acc ~ z_in + z_cri", data=c, groups=c["encoder"], re_formula="1",
                                  vc_formula={"seed": "0 + C(seed)"}).fit(reml=True)
            ci = mix.conf_int().loc["z_cri"].tolist()
            p1 = float(sps.norm.cdf(mix.tvalues["z_cri"]))                  # H3 predicts a negative CRI coefficient
            if np.all(np.isfinite(ci)) and np.isfinite(p1):
                res.update(beta_cri_mixed=float(mix.params["z_cri"]), ci95_mixed=ci, p_mixed_one_sided=p1)
                h3p[tgt] = p1
            else:
                # WARNING: singular random-effects fit (e.g. no between-encoder variance); the cluster bootstrap is reported instead
                logger.warning(f"H3 {tgt}: mixed model gave a non-finite interval; reporting OLS and cluster bootstrap only")
                res["mixed_model_note"] = "singular fit; not used for inference"
        except Exception as e:
            logger.error(f"H3 mixed model failed for {tgt}: {e}")
        rng = np.random.default_rng(0); encs = c.encoder.unique(); boots = []
        for _ in range(2000):
            pick = rng.choice(encs, len(encs))
            bs = pd.concat([c[c.encoder == e] for e in pick])
            if bs.cri.nunique() > 2 and bs.test_acc.nunique() > 2:
                bs = bs.assign(z_in=z(bs.test_acc), z_cri=z(bs.cri))
                boots.append(smf.ols("transfer_acc ~ z_in + z_cri", data=bs).fit().params["z_cri"])
        res["ci95_cluster_bootstrap"] = [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))] if boots else None
        h3[tgt] = res
    for t, p in holm(h3p).items():
        h3[t]["p_holm"] = p
    A["H3_cri_vs_transfer"] = h3
    # ---- H4: prompt effect by stratum (A vs none, seed-paired, per encoder) ------------------------------------------
    h4 = {}
    ip = [r for r in item_rows if r["condition"] == "A" and r.get("explicit") and r.get("implicit")]
    for corpus in sorted({r["corpus"] for r in ip}):
        rr = [r for r in ip if r["corpus"] == corpus and r["stage"] in ("main", "intra")]
        if len(rr) >= 3:
            diff = np.array([abs(r["implicit"]["delta_pp"]) - abs(r["explicit"]["delta_pp"]) for r in rr])
            h4[corpus] = {"n_pairs": len(rr), "mean_abs_implicit_minus_abs_explicit_pp": float(diff.mean()),
                          "p_perm": sign_flip_p(diff), "n_implicit_items": rr[0]["implicit"]["n"], "n_explicit_items": rr[0]["explicit"]["n"]}
    A["H4_prompt_by_stratum"] = h4
    # ---- H5: cue masking vs random masking -----------------------------------------------------------------------------
    sc = df[df.stage == "shortcut"]
    h5 = {}
    if len(sc) and "cri" in sc:
        piv = sc.pivot_table(index=["encoder", "condition", "transform_eval", "seed"], columns="text_transform", values="cri")
        if {"mask_feel", "mask_random"} <= set(piv.columns):
            piv = piv.dropna()
            if len(piv) >= 3:
                h5["cri_mask_feel_minus_mask_random"] = seed_paired(piv["mask_feel"].values, piv["mask_random"].values)
        for tgt in TRANSFER_TARGETS:
            col = f"transfer_{tgt}"
            if col in sc and sc[col].notna().any():
                s2 = sc[sc[col].notna()].assign(acc=lambda d: d[col].map(lambda x: x["restricted"]["acc"]))
                pv = s2.pivot_table(index=["encoder", "condition", "transform_eval", "seed"], columns="text_transform", values="acc").dropna()
                if {"mask_feel", "mask_random"} <= set(pv.columns) and len(pv) >= 3:
                    h5[f"transfer_{tgt}_mask_feel_minus_mask_random"] = seed_paired(100 * pv["mask_feel"].values, 100 * pv["mask_random"].values)
    A["H5_shortcut"] = h5
    # ---- H2 control: the same explicit/implicit gap in target-trained models would indicate item difficulty -----
    ctrl = {}
    for tgt, tcorp in [("goemotions", "goemotions_target"), ("meld", "meld_target"), ("isear", "isear_target"), ("ed", "ed")]:
        tr_runs = df[(df.corpus == tcorp) & (df.condition == "none") & df.stage.isin(["target", "intra"])]
        if tgt not in h2 or not len(tr_runs):
            continue
        g_t = float((tr_runs.test_acc_explicit - tr_runs.test_acc_implicit).mean())
        g_s = h2[tgt]["acc_explicit"] - h2[tgt]["acc_implicit"]
        if tgt == "isear":                                         # recompute the source gap on the 650 target-test items
            pos = np.asarray(DATA["isear_ft"]["test"]["pos_in_full"]); ex = EXPLICIT["isear"][pos]; accs = []
            for _, r in src[src["transfer_isear"].notna()].iterrows():
                rp = f"{EXPERIMENT_NAME}/{r.stage}/runs/{r.spec_id}"; lp = LOCAL_ROOT / rp / "outputs" / "logits.npz"
                with np.load(lp if lp.exists() else hub_download(f"{rp}/outputs/logits.npz")) as z:
                    lg, y = z["transfer_isear_logits"][pos], z["transfer_isear_y"][pos]
                rl = np.full_like(lg, -np.inf); rl[:, PRESENT_CLASSES["isear"]] = lg[:, PRESENT_CLASSES["isear"]]
                c = rl.argmax(1) == y; accs.append((c[ex].mean(), c[~ex].mean()))
            g_s = float(np.mean([a - b for a, b in accs]))
        ctrl[tgt] = {"gap_source_trained": g_s, "gap_target_trained": g_t, "difference_in_differences": g_s - g_t,
                     "n_target_runs": int(len(tr_runs)), "note": "source and target models scored on the same test items"}
    A["H2_control_target_trained"] = ctrl
    # ---- encoder differences: MixedLM test_acc ~ encoder + (1 | seed), main stage, per condition ----------------
    enc_mm = {}
    mn = df[(df.stage == "main") & df.condition.isin(["none", "A"])]
    for cond, g in mn.groupby("condition"):
        if g.encoder.nunique() >= 2 and g.seed.nunique() >= 3:
            ref = "roberta" if "roberta" in set(g.encoder) else sorted(g.encoder)[0]
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    fit = smf.mixedlm(f"test_acc ~ C(encoder, Treatment('{ref}'))", g.assign(test_acc=100 * g.test_acc), groups=g["seed"]).fit(reml=True)
                ci = fit.conf_int()
                enc_mm[cond] = {"reference": ref, "effects_pp": {k.split("T.")[-1].rstrip("]"): {"est": float(fit.params[k]), "ci95": ci.loc[k].tolist(),
                                "p": float(fit.pvalues[k])} for k in fit.params.index if "T." in k}}
            except Exception as e:
                logger.error(f"encoder mixed model failed ({cond}): {e}")
    A["encoder_differences_mixed"] = enc_mm
    # ---- item-level GEE for the prompt effect (A vs none), clustered by test item, encoder fixed effects -----------
    gee = {}
    for (stage, corpus), g in df[df.condition.isin(["none", "A"]) & df.stage.isin(["main", "intra"]) & (df.text_transform == "none")].groupby(["stage", "corpus"]):
        if g.condition.nunique() < 2:
            continue
        parts = []
        for _, r in g.iterrows():
            c = load_test_correct(r)
            parts.append(pd.DataFrame({"y": c.astype(int), "prompt": int(r.condition == "A"), "encoder": r.encoder, "item": np.arange(len(c))}))
        d = pd.concat(parts, ignore_index=True)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fit = smf.gee("y ~ prompt + C(encoder)", groups="item", data=d, family=sm.families.Binomial(),
                              cov_struct=sm.cov_struct.Exchangeable()).fit()
            ci = fit.conf_int().loc["prompt"].tolist()
            gee[f"{stage}|{corpus}"] = {"log_or_prompt": float(fit.params["prompt"]), "ci95": ci, "p": float(fit.pvalues["prompt"]),
                                         "n_rows": int(len(d)), "n_runs": int(len(g))}
        except Exception as e:
            logger.error(f"item-level GEE failed for {corpus}: {e}")
    A["prompt_effect_item_gee"] = gee
    # ---- tables for the paper ----------------------------------------------------------------------------------------
    main_none = df[(df.stage == "main")]
    if len(main_none):
        t = main_none.groupby(["encoder", "condition"]).agg(acc=("test_acc", "mean"), acc_sd=("test_acc", "std"), mf1=("test_macro_f1", "mean"),
                                                           mp=("test_macro_precision", "mean"), mr=("test_macro_recall", "mean"), n=("seed", "nunique"),
                                                           epochs=("epochs_run", "mean")).reset_index()
        t["acc_str"] = [pm_pct(a, b) for a, b in zip(t.acc, t.acc_sd)]
        tex_table(t, ["encoder", "condition", "acc_str", "mf1", "mp", "mr", "n"], TEX / "main_by_condition.tex", {"mf1": f4, "mp": f4, "mr": f4})
        A["table_main_by_condition"] = t.drop(columns=["acc_str"]).to_dict("records")
        cal = main_none[main_none.condition.isin(["none", "A"])].copy()
        cal["ece_before"] = cal.calibration.map(lambda c: c["before"]["ece"]); cal["ece_after"] = cal.calibration.map(lambda c: c["after"]["ece"])
        cal["brier_before"] = cal.calibration.map(lambda c: c["before"]["brier"]); cal["brier_after"] = cal.calibration.map(lambda c: c["after"]["brier"])
        cal["T"] = cal.calibration.map(lambda c: c["temperature"]); cal["T_bound"] = cal.calibration.map(lambda c: c["temperature_at_bound"])
        ct = cal.groupby(["encoder", "condition"])[["ece_before", "ece_after", "brier_before", "brier_after", "T", "T_bound"]].mean().reset_index()
        tex_table(ct, ["encoder", "condition", "ece_before", "ece_after", "brier_before", "brier_after", "T"], TEX / "calibration.tex",
                  {"ece_before": pct, "ece_after": pct, "brier_before": f4, "brier_after": f4, "T": lambda v: f"{v:.2f}"})
        A["table_calibration"] = ct.to_dict("records")
        eff = main_none[main_none.condition == "A"].groupby("encoder").agg(params=("total_parameters", "first"), train_min=("train_seconds", lambda x: x.mean() / 60),
                                                                           items_per_s=("test_items_per_second", "mean"), acc=("test_acc", "mean")).reset_index()
        tex_table(eff, ["encoder", "params", "train_min", "items_per_s", "acc"], TEX / "efficiency.tex",
                  {"params": lambda v: f"{int(v):,}", "train_min": lambda v: f"{v:.1f}", "items_per_s": lambda v: f"{v:.0f}", "acc": pct})
        A["table_efficiency"] = eff.to_dict("records")
    rows_t = []
    for tgt in TRANSFER_TARGETS:
        col = f"transfer_{tgt}"
        if col not in df:
            continue
        s_ = df[df.stage.isin(["main"]) & df[col].notna() & df.condition.isin(["none", "A"])]
        for (enc, cond), g in s_.groupby(["encoder", "condition"]):
            row = {"target": tgt, "encoder": enc, "condition": cond, "n": len(g)}
            for rule in ("unrestricted", "restricted", "prior_em"):
                v = g[col].map(lambda d: d[rule]["acc"]); row[f"{rule}_acc"], row[f"{rule}_sd"] = v.mean(), v.std()
            row["atc_pred"] = g[col].map(lambda d: d["atc_predicted_acc"]).mean()
            row["degenerate_runs"] = int(g[col].map(lambda d: d["unrestricted"]["degenerate"]).sum())
            # v8: cev_third and eni_third repeat the GoEmotions configurations of the target stage; keep the upper bound on the
            # target-stage runs only (cev_third gave the same test accuracies, so the published values do not change)
            up = df[(df.corpus == (tgt + "_target" if tgt != "ed" else "ed")) & (df.encoder == enc) & (df.condition == cond)
                    & ~df.stage.isin(["cev_third"] + ENI_EXPORT_STAGES)]
            row["target_trained_acc"] = up.test_acc.mean() if len(up) else None
            if tgt == "isear":                   # source models scored on exactly the 650 items of the target-trained test split
                pos = np.asarray(DATA["isear_ft"]["test"]["pos_in_full"]); accs = []
                for _, r in g.iterrows():
                    rp = f"{EXPERIMENT_NAME}/{r.stage}/runs/{r.spec_id}"; lp = LOCAL_ROOT / rp / "outputs" / "logits.npz"
                    with np.load(lp if lp.exists() else hub_download(f"{rp}/outputs/logits.npz")) as z:
                        lg, y = z["transfer_isear_logits"][pos], z["transfer_isear_y"][pos]
                    rl = np.full_like(lg, -np.inf); rl[:, PRESENT_CLASSES["isear"]] = lg[:, PRESENT_CLASSES["isear"]]
                    accs.append(float(np.mean(rl.argmax(1) == y)))
                row["restricted_acc_same_items_as_target"] = float(np.mean(accs))
            else:
                row["restricted_acc_same_items_as_target"] = row["restricted_acc"]
            rows_t.append(row)
    if rows_t:
        tt = pd.DataFrame(rows_t)
        tt["unrestricted"] = [pm_pct(a, b) for a, b in zip(tt.unrestricted_acc, tt.unrestricted_sd)]
        tt["restricted"] = [pm_pct(a, b) for a, b in zip(tt.restricted_acc, tt.restricted_sd)]
        tt["prior_em"] = [pm_pct(a, b) for a, b in zip(tt.prior_em_acc, tt.prior_em_sd)]
        tex_table(tt, ["target", "encoder", "condition", "unrestricted", "restricted", "prior_em", "atc_pred",
                       "restricted_acc_same_items_as_target", "target_trained_acc"],
                  TEX / "transfer.tex", {"atc_pred": pct, "restricted_acc_same_items_as_target": pct, "target_trained_acc": pct})
        A["table_transfer"] = tt.drop(columns=["unrestricted", "restricted", "prior_em"]).to_dict("records")
    for stage_name, keycol in (("peft", "peft_method"), ("rebalance", "rebalance")):
        g = df[df.stage == stage_name]
        if len(g):
            per = g.groupby(["encoder", keycol]).agg(acc=("test_acc", "mean"), acc_sd=("test_acc", "std"), mf1=("test_macro_f1", "mean"),
                                                     wf1=("test_weighted_f1", "mean"), trainable=("trainable_parameters", "first"),
                                                     f1_love=("test_f1_per_class", lambda x: np.mean([v[2] for v in x])),
                                                     f1_surprise=("test_f1_per_class", lambda x: np.mean([v[5] for v in x])),
                                                     n=("seed", "nunique")).reset_index()
            per["acc_str"] = [pm_pct(a, b) for a, b in zip(per.acc, per.acc_sd)]
            tex_table(per, ["encoder", keycol, "trainable", "acc_str", "mf1", "wf1", "f1_surprise", "f1_love", "n"], TEX / f"{stage_name}.tex",
                      {"trainable": lambda v: f"{int(v):,}", "mf1": f4, "wf1": f4, "f1_surprise": f4, "f1_love": f4})
            A[f"table_{stage_name}"] = per.drop(columns=["acc_str"]).to_dict("records")
    nontrain_tables(A, D_LLM, D_AUDIT, TEX)
    # ---- same-seed rerun variation -----------------------------------------------------------------------------------
    rr = df[df.stage == "rerun"]
    rerun = {}
    if len(rr) and "repeat" in rr:
        for enc, g in rr.groupby("encoder"):
            if g["repeat"].nunique() == 2:
                a, b = (g[g["repeat"] == k].iloc[0] for k in sorted(g["repeat"].unique()))
                ca, cb = load_test_correct(a), load_test_correct(b)
                rerun[enc] = {"acc_1": float(a.test_acc), "acc_2": float(b.test_acc), "abs_diff_pp": float(100 * abs(a.test_acc - b.test_acc)),
                              "item_disagreement": float(np.mean(ca != cb)), **paired_item_stats(ca, cb)}
    A["rerun_variation"] = rerun
    A.update(cev_analyses(df))
    # ---- convergence ---------------------------------------------------------------------------------------------------
    conv = {}
    cv = df[df.stage == "convergence"]
    for enc, g in cv.groupby("encoder"):
        a, b = g[g.condition == "A"].converged, g[g.condition == "none"].converged
        if len(a) and len(b):
            conv[enc] = {"converged_A": [int(a.sum()), int(len(a))], "converged_none": [int(b.sum()), int(len(b))],
                         "p_fisher": float(sps.fisher_exact([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])[1])}
    A["convergence"] = conv
    return A

METRICS_DF = load_all_metrics()
ANALYSIS = aggregate(METRICS_DF)
for name, obj in ANALYSIS.items():
    write_json(ANALYSIS_DIR / "metrics" / f"{name}.json", obj)
if len(METRICS_DF):
    METRICS_DF.drop(columns=[c for c in METRICS_DF.columns if c.startswith("transfer_")]).to_csv(ANALYSIS_DIR / "metrics" / "all_runs.csv", index=False)
logger.info(f"Aggregation complete: {sorted(ANALYSIS)}")

[2026-10-07 14:01:20] [INFO] Loaded metrics for 1672 evaluated runs across stages
[2026-10-07 14:03:14] [INFO] Aggregation complete: ['H1_explicit_share', 'H2_control_target_trained', 'H2_stratified_transfer', 'H3_cri_vs_transfer', 'H4_prompt_by_stratum', 'H5_shortcut', 'audit', 'cev_E1_minimal_pairs', 'cev_E2_cri_validation', 'cev_E3_perspective', 'cev_E4_appraisals_exploratory', 'cev_E5_rerun_variation_exploratory', 'convergence', 'encoder_differences_mixed', 'item_paired', 'lr_sweep_table', 'prompt_effect_item_gee', 'random_effects', 'rerun_variation', 'seed_paired', 'seed_table', 'table_calibration', 'table_efficiency', 'table_llm', 'table_main_by_condition', 'table_peft', 'table_rebalance', 'table_transfer', 'tuned_recipes', 'tuning_table']


## CELL 9c — ROBUSTNESS RECOMPUTATIONS

In [36]:
# =====================================================================================
# CELL 9c — ROBUSTNESS RECOMPUTATIONS (analysis only: reads stored logits and metrics, trains nothing)
# R1  H4 with a size-calibrated permutation null. The original H4 statistic, mean(|effect on implicit items| -
#     |effect on explicit items|), is biased upward when the implicit stratum is smaller (sampling noise inflates
#     absolute effects on fewer items). The null here permutes the explicit/implicit labels across items while keeping
#     both stratum sizes, applying the same permutation to every encoder x seed pair (items are shared across runs).
# R2  H2 control with uncertainty: difference in differences (source-trained gap minus target-trained gap) with a joint
#     bootstrap over test items and runs. Both model families are scored with the restricted rule on identical items.
# R3  H3 restricted to main-stage runs, i.e. without the masked-training runs of the shortcut stage, which inject
#     cue-reliance variation by design.
# Results: analysis/metrics/robustness_R1_h4.json, robustness_R2_h2_control.json, robustness_R3_h3_main_only.json
# (uploaded to the Hub by Cell 11 together with the rest of the analysis folder).
# =====================================================================================
ROBUST_N_PERM = 5000
ROBUST_N_BOOT = 2000
ROBUST_SEED = 20261004

def robust_train_df() -> pd.DataFrame:
    df = METRICS_DF.copy()
    df["kind"] = df["kind"].fillna("train") if "kind" in df else "train"
    df = df[df.kind == "train"].copy()
    for k in ["text_transform", "fewshot_k", "peft_method", "rebalance"]:
        if k not in df:
            df[k] = None
        df[k] = df[k].astype(object).where(df[k].notna(), "none")
    return df

def robust_npz(r):
    p = f"{EXPERIMENT_NAME}/{r['stage']}/runs/{r['spec_id']}"
    lp = LOCAL_ROOT / p / "outputs" / "logits.npz"
    return np.load(lp if lp.exists() else hub_download(f"{p}/outputs/logits.npz"))

def robust_restricted_correct(lg: np.ndarray, y: np.ndarray, allowed: list) -> np.ndarray:
    r = np.full_like(lg, -np.inf)
    r[:, allowed] = lg[:, allowed]
    return r.argmax(1) == y

def robust_r1_h4(df: pd.DataFrame) -> dict:
    out, pv = {}, {}
    rng = np.random.default_rng(ROBUST_SEED)
    keys = ["stage", "corpus", "encoder", "lr", "epochs", "placement", "head", "seed"]
    pool = df[df.stage.isin(["main", "intra"]) & (df.text_transform == "none") & df.condition.isin(["none", "A"])]
    for corpus, g in pool.groupby("corpus"):
        sk = STRATA_KEY.get("setfit" if str(corpus).startswith("setfit") else corpus)
        if sk is None:
            continue
        ex = np.asarray(EXPLICIT[sk], bool)
        a, b = g[g.condition == "A"].set_index(keys), g[g.condition == "none"].set_index(keys)
        common = a.index.intersection(b.index)
        if len(common) < 3:
            logger.info(f"R1 {corpus}: {len(common)} prompt/no-prompt pairs; at least 3 needed")
            continue
        D = np.stack([load_test_correct({"stage": kk[0], "spec_id": a.loc[kk, "spec_id"]}).astype(float)
                      - load_test_correct({"stage": kk[0], "spec_id": b.loc[kk, "spec_id"]}).astype(float) for kk in common])
        assert D.shape[1] == len(ex), f"R1 {corpus}: {D.shape[1]} test items but {len(ex)} explicitness flags"
        n, k = len(ex), int(ex.sum())
        assert 0 < k < n, f"R1 {corpus}: one stratum is empty"
        def stat(masks: np.ndarray) -> np.ndarray:            # masks: (P, n) explicit flags -> (P,) statistic
            M = masks.astype(float)
            imp, exp_ = D @ (1.0 - M).T / (n - k), D @ M.T / k
            return (np.abs(imp) - np.abs(exp_)).mean(0)
        obs = float(stat(ex[None, :])[0])
        null = np.concatenate([stat(np.stack([rng.permutation(ex) for _ in range(min(1000, ROBUST_N_PERM - s))]))
                               for s in range(0, ROBUST_N_PERM, 1000)])
        assert len(null) == ROBUST_N_PERM
        p = float((1 + np.sum(null >= obs - 1e-12)) / (ROBUST_N_PERM + 1))
        out[corpus] = {"n_pairs": int(len(common)), "n_implicit_items": int(n - k), "n_explicit_items": k,
                       "observed_pp": 100 * obs, "null_mean_pp": 100 * float(null.mean()),
                       "null_95pct_pp": [100 * float(np.percentile(null, 2.5)), 100 * float(np.percentile(null, 97.5))],
                       "excess_over_null_pp": 100 * (obs - float(null.mean())), "p_perm_size_calibrated": p}
        pv[corpus] = p
    for c, ph in holm(pv).items():
        out[c]["p_holm"] = ph
    out["_note"] = ("statistic = mean over encoder x seed pairs of |prompt effect on implicit items| - |on explicit items| (pp); "
                    "null = explicit/implicit labels permuted across items with stratum sizes fixed; one-sided")
    return out

def robust_r2_h2_control(df: pd.DataFrame) -> dict:
    out, pv = {}, {}
    rng = np.random.default_rng(ROBUST_SEED + 1)
    targets = [("goemotions", "goemotions_target", "goemotions", None),
               ("meld", "meld_target", "meld", None),
               ("isear", "isear_target", "isear_target_test", np.asarray(DATA["isear_ft"]["test"]["pos_in_full"])),
               ("ed", "ed", "ed_test", None)]
    src_all = df[(df.stage == "main") & df.condition.isin(["none", "A"])]
    for tgt, tcorp, exkey, pos in targets:
        col = f"transfer_{tgt}"
        tr_runs = df[(df.corpus == tcorp) & (df.condition == "none") & df.stage.isin(["target", "intra"])]
        if col not in src_all or src_all[col].isna().all() or not len(tr_runs):
            continue
        src = src_all[src_all[col].notna()]
        allowed = PRESENT_CLASSES[tgt]
        S, T, y_src, y_tgt = [], [], None, None
        for _, r in src.iterrows():
            with robust_npz(r) as z:
                lg, y = z[f"transfer_{tgt}_logits"], z[f"transfer_{tgt}_y"]
            if pos is not None:
                lg, y = lg[pos], y[pos]
            y_src = y if y_src is None else y_src
            assert np.array_equal(y, y_src), f"R2 {tgt}: source runs disagree on test labels"
            S.append(robust_restricted_correct(lg, y, allowed))
        for _, r in tr_runs.iterrows():
            with robust_npz(r) as z:
                lg, y = z["test_logits"], z["test_y"]
            y_tgt = y if y_tgt is None else y_tgt
            T.append(robust_restricted_correct(lg, y, allowed))
        # identical items and labels for source-trained and target-trained models are a precondition of this analysis
        assert np.array_equal(y_src, y_tgt), f"R2 {tgt}: source and target models were not scored on identical items"
        ex = np.asarray(EXPLICIT[exkey], bool)
        if pos is not None:
            assert np.array_equal(ex, np.asarray(EXPLICIT["isear"], bool)[pos]), "R2 isear: explicitness flags misaligned"
        S, T = np.asarray(S, float), np.asarray(T, float)
        assert S.shape[1] == T.shape[1] == len(ex)
        gap = lambda C, e: float(np.mean(C[:, e].mean(1) - C[:, ~e].mean(1)))
        g_s, g_t = gap(S, ex), gap(T, ex)
        n, rs, rt, boots = len(ex), len(S), len(T), []
        while len(boots) < ROBUST_N_BOOT:
            ii = rng.integers(0, n, n); e = ex[ii]
            if e.all() or not e.any():
                continue
            bs, bt = S[rng.integers(0, rs, rs)][:, ii], T[rng.integers(0, rt, rt)][:, ii]
            boots.append((gap(bs, e), gap(bt, e)))
        bo = np.asarray(boots); did_b = bo[:, 0] - bo[:, 1]
        # a bootstrap with B resamples cannot resolve p below 1/B; report that floor instead of 0
        p = float(max(1.0 / ROBUST_N_BOOT, min(1.0, 2 * min(np.mean(did_b <= 0), np.mean(did_b >= 0)))))
        ci = lambda v: [100 * float(np.percentile(v, 2.5)), 100 * float(np.percentile(v, 97.5))]
        out[tgt] = {"n_items": int(n), "n_explicit": int(ex.sum()), "n_source_runs": int(rs), "n_target_runs": int(rt),
                    "gap_source_pp": 100 * g_s, "gap_source_ci95_pp": ci(bo[:, 0]),
                    "gap_target_pp": 100 * g_t, "gap_target_ci95_pp": ci(bo[:, 1]),
                    "did_pp": 100 * (g_s - g_t), "did_ci95_pp": ci(did_b), "p_two_sided_bootstrap": p}
        pv[tgt] = p
    for t, ph in holm(pv).items():
        out[t]["p_holm"] = ph
    out["_note"] = ("gap = accuracy on explicit minus implicit items (restricted rule); source = main-stage runs with no prompt "
                    "or template A; target = models trained on the target corpus without a prompt; joint bootstrap over items and runs")
    return out

def robust_r3_h3_main_only(df: pd.DataFrame) -> dict:
    out, pv = {}, {}
    rng = np.random.default_rng(ROBUST_SEED + 2)
    cand = df[(df.stage == "main") & df["cri"].notna()] if "cri" in df else df.iloc[0:0]
    for tgt in TRANSFER_TARGETS:
        col = f"transfer_{tgt}"
        if col not in cand or cand[col].isna().all():
            continue
        c = cand[cand[col].notna()].copy()
        c["transfer_acc"] = c[col].map(lambda d: d["restricted"]["acc"])
        if len(c) < 10 or c.encoder.nunique() < 2:
            continue
        zf = lambda s: (s - s.mean()) / (s.std() + 1e-12)
        c["z_in"], c["z_cri"] = zf(c.test_acc), zf(c.cri)
        ols = smf.ols("transfer_acc ~ z_in + z_cri", data=c).fit()
        res = {"n_runs": int(len(c)), "n_encoders": int(c.encoder.nunique()), "beta_cri_ols": float(ols.params["z_cri"]),
               "spearman_cri": float(sps.spearmanr(c.cri, c.transfer_acc)[0]),
               "r2_in_only": float(smf.ols("transfer_acc ~ z_in", data=c).fit().rsquared), "r2_in_plus_cri": float(ols.rsquared)}
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                mix = smf.mixedlm("transfer_acc ~ z_in + z_cri", data=c, groups=c["encoder"], re_formula="1",
                                  vc_formula={"seed": "0 + C(seed)"}).fit(reml=True)
            ci_m = mix.conf_int().loc["z_cri"].tolist()
            p1 = float(sps.norm.cdf(mix.tvalues["z_cri"]))            # H3 predicts a negative coefficient
            if np.all(np.isfinite(ci_m)) and np.isfinite(p1):
                res.update(beta_cri_mixed=float(mix.params["z_cri"]), ci95_mixed=ci_m, p_mixed_one_sided=p1)
                pv[tgt] = p1
            else:
                res["mixed_model_note"] = "singular fit; not used for inference"
        except Exception as e:
            logger.error(f"R3 mixed model failed for {tgt}: {e}")
        encs, boots = c.encoder.unique(), []
        for _ in range(ROBUST_N_BOOT):
            bs = pd.concat([c[c.encoder == e] for e in rng.choice(encs, len(encs))])
            if bs.cri.nunique() > 2 and bs.test_acc.nunique() > 2:
                bs = bs.assign(z_in=zf(bs.test_acc), z_cri=zf(bs.cri))
                boots.append(smf.ols("transfer_acc ~ z_in + z_cri", data=bs).fit().params["z_cri"])
        res["ci95_cluster_bootstrap"] = [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))] if boots else None
        res["n_bootstrap_valid"] = int(len(boots))     # resamples with enough distinct encoders to fit the model
        res["n_distinct_bootstrap_estimates"] = int(len(np.unique(np.round(boots, 12)))) if boots else 0
        out[tgt] = res
    for t, ph in holm(pv).items():
        out[t]["p_holm"] = ph
    out["_note"] = "H3 model refitted on main-stage runs only (no masked-training runs); coefficients are accuracy per SD of CRI"
    return out

def robust_r1b_placebo(df: pd.DataFrame) -> dict:
    """R1b (SetFit only, where R1 was significant). Question: is the larger prediction change on implicit items specific to
    the instruction's meaning, or do harder items change more under ANY prefix? Placebo = the random-token prefix of the
    same token length as template A. For every encoder x seed, template A and the random prefix are both compared with the
    SAME no-prompt run (same initialization and data order), so a correctness change occurs exactly where a prefix changed
    the prediction. Primary statistic: log concentration ratio, log(change rate on implicit items / change rate on explicit
    items), template A minus random prefix, paired bootstrap over encoder x seed. In this paired design the ratio does not
    depend on how strongly each prefix perturbs the model (validated by simulation: false-positive rate 0.07-0.09 at a
    nominal 0.05 with perturbation sizes differing up to 2x; slightly liberal). Also reported: signed effects per stratum."""
    rng = np.random.default_rng(ROBUST_SEED + 3)
    g = df[(df.stage == "main") & (df.corpus == "setfit") & (df.text_transform == "none")
           & df.condition.isin(["none", "A", "random"])]
    if not len(g):
        return {}
    assert not g.duplicated(["encoder", "condition", "seed"]).any(), "R1b: duplicate main runs for one encoder/condition/seed"
    ex = np.asarray(EXPLICIT["setfit_test"], bool)
    corr = {(r.encoder, r.condition, int(r.seed)): load_test_correct(r).astype(float) for _, r in g.iterrows()}
    assert all(len(v) == len(ex) for v in corr.values()), "R1b: test size does not match explicitness flags"
    keys = sorted((e, s) for (e, c, s) in corr if c == "none" and (e, "A", s) in corr and (e, "random", s) in corr)
    if len(keys) < 3:
        logger.info(f"R1b: {len(keys)} complete encoder x seed triples (none, A, random); at least 3 needed")
        return {}
    DA = np.stack([corr[(e, "A", s)] - corr[(e, "none", s)] for e, s in keys])
    DR = np.stack([corr[(e, "random", s)] - corr[(e, "none", s)] for e, s in keys])
    encs = np.array([e for e, _ in keys])
    def logratio(D):
        fi, fe = np.abs(D[:, ~ex]).mean(), np.abs(D[:, ex]).mean()
        return float(np.log(fi / fe)) if fi > 0 and fe > 0 else float("nan")
    est = logratio(DA) - logratio(DR)
    boots = []
    while len(boots) < ROBUST_N_BOOT:
        idx = rng.integers(0, len(keys), len(keys))                  # paired: A and random resampled together
        v = logratio(DA[idx]) - logratio(DR[idx])
        if np.isfinite(v):
            boots.append(v)
    boots = np.asarray(boots)
    p = float(max(1.0 / ROBUST_N_BOOT, min(1.0, 2 * min(np.mean(boots <= 0), np.mean(boots >= 0)))))
    per_enc = {str(e): logratio(DA[encs == e]) - logratio(DR[encs == e]) for e in sorted(set(encs))}
    def rates(D):
        return {"change_rate_implicit_pct": 100 * float(np.abs(D[:, ~ex]).mean()),
                "change_rate_explicit_pct": 100 * float(np.abs(D[:, ex]).mean()),
                "concentration_ratio": float(np.exp(logratio(D)))}
    def signed(D, m):
        v = 100 * D[:, m].mean(1); t = sps.t.ppf(0.975, len(v) - 1); se = v.std(ddof=1) / np.sqrt(len(v))
        return {"mean_pp": float(v.mean()), "ci95_pp": [float(v.mean() - t * se), float(v.mean() + t * se)]}
    return {"corpus": "setfit", "n_pairs": int(len(keys)), "n_implicit_items": int((~ex).sum()), "n_explicit_items": int(ex.sum()),
            "template_A_vs_none": rates(DA), "random_prefix_vs_none": rates(DR),
            "log_concentration_ratio_A_minus_random": {"est": float(est), "ci95": [float(np.percentile(boots, 2.5)),
                                                       float(np.percentile(boots, 97.5))], "p_two_sided_bootstrap": p},
            "per_encoder_log_ratio_difference": per_enc,
            "signed_effect_A_implicit": signed(DA, ~ex), "signed_effect_A_explicit": signed(DA, ex),
            "signed_effect_random_implicit": signed(DR, ~ex), "signed_effect_random_explicit": signed(DR, ex),
            "_note": ("Positive log-ratio difference: template A's changes concentrate on implicit items more than a meaningless "
                      "prefix's changes do (instruction-specific). Near 0: implicit items change more under any prefix "
                      "(generic volatility of harder items). Signed effects show whether a prefix helps (positive) or only "
                      "moves predictions. The bootstrap treats encoder x seed pairs as independent; per-encoder differences "
                      "show sign consistency. Simulation calibration: slightly liberal (0.07-0.09 at nominal 0.05).")}

ROBUSTNESS = {}
_rdf = robust_train_df()
if len(_rdf):
    for name, fn in [("robustness_R1_h4", robust_r1_h4), ("robustness_R1b_h4_placebo", robust_r1b_placebo),
                     ("robustness_R2_h2_control", robust_r2_h2_control),
                     ("robustness_R3_h3_main_only", robust_r3_h3_main_only)]:
        try:
            ROBUSTNESS[name] = fn(_rdf)
            write_json(ANALYSIS_DIR / "metrics" / f"{name}.json", ROBUSTNESS[name])
        except Exception as e:
            logger.error(f"{name} failed: {type(e).__name__}: {e}")
            raise
for name, res in ROBUSTNESS.items():
    if name.endswith("R1b_h4_placebo"):
        if res:
            lr = res["log_concentration_ratio_A_minus_random"]
            logger.info(f"R1b setfit: concentration ratio template A {res['template_A_vs_none']['concentration_ratio']:.2f} vs random "
                        f"prefix {res['random_prefix_vs_none']['concentration_ratio']:.2f}; log difference {lr['est']:+.3f} "
                        f"[{lr['ci95'][0]:+.3f}, {lr['ci95'][1]:+.3f}], p = {lr['p_two_sided_bootstrap']:.4f}; signed effect on implicit "
                        f"items: A {res['signed_effect_A_implicit']['mean_pp']:+.2f} pp "
                        f"{[round(x, 2) for x in res['signed_effect_A_implicit']['ci95_pp']]}, random "
                        f"{res['signed_effect_random_implicit']['mean_pp']:+.2f} pp")
        continue
    for key, v in res.items():
        if key.startswith("_"):
            continue
        if name.endswith("R1_h4"):
            logger.info(f"R1 {key}: observed {v['observed_pp']:+.2f} pp vs size-matched null {v['null_mean_pp']:+.2f} pp "
                        f"-> excess {v['excess_over_null_pp']:+.2f} pp, p = {v['p_perm_size_calibrated']:.4f}")
        elif name.endswith("R2_h2_control"):
            logger.info(f"R2 {key}: DiD {v['did_pp']:+.2f} pp, 95% CI [{v['did_ci95_pp'][0]:+.2f}, {v['did_ci95_pp'][1]:+.2f}], "
                        f"p {'<' if v['p_two_sided_bootstrap'] <= 1.0 / ROBUST_N_BOOT else '='} {v['p_two_sided_bootstrap']:.4f}")
        else:
            logger.info(f"R3 {key}: beta_CRI mixed {v.get('beta_cri_mixed', float('nan')):+.4f}, bootstrap CI "
                        f"{v.get('ci95_cluster_bootstrap')} ({v.get('n_distinct_bootstrap_estimates', 0)} distinct estimates), n = {v['n_runs']}")
logger.info(f"Robustness recomputations complete: {sorted(ROBUSTNESS)}")

[2026-10-07 14:05:01] [INFO] R1 ed: observed +0.48 pp vs size-matched null +0.27 pp -> excess +0.21 pp, p = 0.1574
[2026-10-07 14:05:01] [INFO] R1 setfit: observed +2.58 pp vs size-matched null +1.19 pp -> excess +1.38 pp, p = 0.0010
[2026-10-07 14:05:01] [INFO] R1 tweeteval: observed +0.29 pp vs size-matched null +0.21 pp -> excess +0.07 pp, p = 0.3961
[2026-10-07 14:05:01] [INFO] R1b setfit: concentration ratio template A 5.97 vs random prefix 6.09; log difference -0.019 [-0.093, +0.054], p = 0.6050; signed effect on implicit items: A +0.02 pp [-0.99, 1.02], random +0.05 pp
[2026-10-07 14:05:01] [INFO] R2 goemotions: DiD +5.97 pp, 95% CI [-0.62, +12.75], p = 0.0820
[2026-10-07 14:05:01] [INFO] R2 meld: DiD +0.10 pp, 95% CI [-5.67, +5.34], p = 0.9770
[2026-10-07 14:05:01] [INFO] R2 isear: DiD -3.65 pp, 95% CI [-8.92, +1.55], p = 0.1540
[2026-10-07 14:05:01] [INFO] R2 ed: DiD +3.77 pp, 95% CI [+0.08, +7.48], p = 0.0450
[2026-10-07 14:05:01] [INFO] R3 goemotions: beta_CRI mixed +0.0010,

In [37]:
# =====================================================================================
# CELL 9d — NEW CONTROLS (analysis only). X1 third-corpus E1 | X2 second source | X3 zero-shot LLM | X4 E3 on reader-version texts
# =====================================================================================
X_N_BOOT = 2000
X_SEED = 20261008

def x_runs(df, stage, **eq):
    d = df[df.stage == stage]
    for k, v in eq.items():
        d = d[d[k] == v]
    return d

def x_cev_reader():
    rp = np.asarray(CEV["reader_pos"]); ch = np.asarray(CEV["changed"]); y = np.asarray(CEV["label"])
    return rp, ch, y, ch[rp], y[rp], np.asarray(CEV["reader_majority"])

def x_drops(rows, kind: str) -> pd.DataFrame:
    rp, ch, y, w, yr, _ = x_cev_reader()
    out = []
    for _, r in rows.iterrows():
        with robust_npz(r) as z:
            if kind == "target":
                po, pm = cev_restrict(z["test_logits"]), cev_restrict(z["cevtest_masked_logits"])
                assert len(po) == len(rp), "crowd-enVENT test set is not the reader set"
                d163, d863 = 100 * (np.mean(po[w] == yr[w]) - np.mean(pm[w] == yr[w])), np.nan
            else:
                po, pm = cev_restrict(z["cev_orig_logits"]), cev_restrict(z["cev_masked_logits"])
                d863 = 100 * (np.mean(po[ch] == y[ch]) - np.mean(pm[ch] == y[ch]))
                d163 = 100 * (np.mean(po[rp][w] == yr[w]) - np.mean(pm[rp][w] == yr[w]))
        out.append({"encoder": r.encoder, "condition": r.condition, "seed": int(r.seed), "drop863": d863, "drop163": d163})
    return pd.DataFrame(out)

def x_pooled_diff(a, b, col) -> dict:
    ys, vs, per = [], [], {}
    for e in sorted(set(a.encoder) & set(b.encoder)):
        u, v = a[a.encoder == e][col].dropna(), b[b.encoder == e][col].dropna()
        if len(u) < 2 or len(v) < 2:
            continue
        ys.append(u.mean() - v.mean()); vs.append(u.var(ddof=1) / len(u) + v.var(ddof=1) / len(v)); per[e] = float(ys[-1])
    return {**random_effects(ys, vs), "per_encoder": per} if len(ys) >= 2 else {}

def x_e1_third(df) -> dict:
    third = x_runs(df, "cev_third")
    if not len(third):
        return {}
    D3 = x_drops(third, "source")
    DS = x_drops(x_runs(df, "cev_source"), "source"); DSn = DS[DS.condition == "none"]
    DT = x_drops(x_runs(df, "cev_target"), "target")
    pool = lambda D, c: {**random_effects([D[D.encoder == e][c].mean() for e in sorted(D.encoder.unique())],
                                          [D[D.encoder == e][c].var(ddof=1) / max(1, (D.encoder == e).sum()) for e in sorted(D.encoder.unique())])}
    tgt = x_runs(df, "target", corpus="goemotions_target", condition="none")
    m = third.merge(tgt, on=["encoder", "seed"], suffixes=("_x", "_t"))
    return {"n_runs": {"goemotions_trained": len(D3), "setfit_trained_none": len(DSn), "cev_trained": len(DT)},
            "drop863_pooled": {"goemotions_trained": pool(D3, "drop863"), "setfit_trained_none": pool(DSn, "drop863")},
            "drop163_pooled": {"goemotions_trained": pool(D3, "drop163"), "setfit_trained_none": pool(DSn, "drop163"),
                               "cev_trained": pool(DT, "drop163")},
            "diff_setfit_minus_goemotions_863": x_pooled_diff(DSn, D3, "drop863"),
            "diff_setfit_minus_goemotions_163": x_pooled_diff(DSn, D3, "drop163"),
            "diff_goemotions_minus_cev_163": x_pooled_diff(D3, DT, "drop163"),
            "diff_setfit_minus_cev_163_none_only": x_pooled_diff(DSn, DT, "drop163"),
            "goemotions_test_acc_by_encoder": third.groupby("encoder").test_acc.mean().to_dict(),
            "rerun_check_vs_target_stage": {"pairs": int(len(m)), "equal_test_acc": int((m.test_acc_x == m.test_acc_t).sum())},
            "_note": "Shortcut account predicts SetFit drop > GoEmotions drop. GoEmotions ~ crowd-enVENT drop => E1 reflects the target's own cue. Condition none."}

def x_correct_matrix(rows, key, allowed, pos=None):
    C = []
    for _, r in rows.iterrows():
        with robust_npz(r) as z:
            lg, yy = z[key], z[key.replace("_logits", "_y")]
        if pos is not None:
            lg, yy = lg[pos], np.asarray(yy)[pos]
        C.append(robust_restricted_correct(lg, np.asarray(yy), allowed))
    return np.array(C)

def x_gap_did(CS, CT, ex, rng) -> dict:
    def gap(C, idx):
        Ci = C[:, idx]; e = ex[idx]
        return 100 * (Ci[:, e].mean() - Ci[:, ~e].mean())
    n = CS.shape[1]; all_i = np.arange(n)
    gS, gT = gap(CS, all_i), gap(CT, all_i)
    boots = []
    for _ in range(X_N_BOOT):
        i = rng.integers(0, n, n)
        if ex[i].all() or (~ex[i]).all():
            continue
        s = CS[rng.integers(0, len(CS), len(CS))]; t = CT[rng.integers(0, len(CT), len(CT))]
        boots.append(gap(s, i) - gap(t, i))
    boots = np.array(boots)
    p = float(min(1.0, 2 * min(np.mean(boots <= 0), np.mean(boots >= 0))))
    return {"gap_source_pp": float(gS), "gap_target_pp": float(gT), "did_pp": float(gS - gT),
            "did_ci95_pp": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
            "did_ci90_pp": [float(np.percentile(boots, 5)), float(np.percentile(boots, 95))], "p_two_sided_bootstrap": p,
            "source_acc_pct": float(100 * CS.mean()), "target_acc_pct": float(100 * CT.mean())}

def x_second_source(df) -> dict:
    third = x_runs(df, "cev_third")
    if not len(third) or "transfer_meld" not in third or third["transfer_meld"].isna().all():
        return {}
    rng = np.random.default_rng(X_SEED)
    isear_pos = np.asarray(DATA["isear_ft"]["test"]["pos_in_full"])
    spec = [("meld", "meld_target", EXPLICIT["meld"], None), ("isear", "isear_target", EXPLICIT["isear_target_test"], isear_pos),
            ("ed", "ed", EXPLICIT["ed_test"], None)]
    out, pv = {}, {}
    for tgt, tcorp, ex, pos in spec:
        tr = df[(df.corpus == tcorp) & (df.condition == "none") & df.stage.isin(["target", "intra"])]
        if not len(tr):
            continue
        CS = x_correct_matrix(third, f"transfer_{tgt}_logits", PRESENT_CLASSES[tgt], pos)
        CT = x_correct_matrix(tr, "test_logits", PRESENT_CLASSES[tgt])
        assert CS.shape[1] == CT.shape[1] == len(ex), f"{tgt}: item sets differ"
        out[tgt] = x_gap_did(CS, CT, np.asarray(ex), rng); pv[tgt] = out[tgt]["p_two_sided_bootstrap"]
        out[tgt].update(n_items=int(len(ex)), n_source_runs=int(len(CS)), n_target_runs=int(len(CT)))
    for t, ph in holm(pv).items():
        out[t]["p_holm"] = ph
    out["_note"] = "Source = GoEmotions-trained (cev_third, none). Restricted rule. ISEAR on the 650-item test split."
    return out

def x_llm_load(task):
    rows, ci = [], 0
    while True:
        rel = f"{EXPERIMENT_NAME}/llm_transfer/{task}/chunk_{ci:03d}.jsonl"
        lp = LOCAL_ROOT / rel
        if not lp.exists():
            if rel not in HUB_FILES:
                break
            lp = hub_download(rel)
        rows += [json.loads(l) for l in open(lp) if l.strip()]
        ci += 1
    return pd.DataFrame(rows)

def x_boot_gap(correct, ex, rng):
    g = lambda c, e: 100 * (c[e].mean() - c[~e].mean())
    b = []
    for _ in range(X_N_BOOT):
        i = rng.integers(0, len(correct), len(correct))
        if ex[i].all() or (~ex[i]).all():
            continue
        b.append(g(correct[i], ex[i]))
    return {"gap_pp": float(g(correct, ex)), "ci95_pp": [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]}

def x_e3_gee(correct_runs: list, extra: dict) -> dict:
    rp, ch, y, w, yr, mj = x_cev_reader()
    racc = np.asarray(CEV["reader_item_acc"]); cons = np.asarray(CEV["reader_consensus"])
    z = lambda v: (v - v.mean()) / (v.std() + 1e-12)
    dd = pd.concat([pd.DataFrame({"correct": c.astype(int), "item": np.arange(len(rp)), "z_racc": z(racc), "z_cons": z(cons),
                                  "changed": w.astype(int)}) for c in correct_runs], ignore_index=True)
    res = {}
    for name, f in [("reader_acc", "correct ~ z_racc + changed"), ("reader_acc_and_consensus", "correct ~ z_racc + z_cons + changed")]:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = smf.gee(f, groups="item", data=dd, family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()).fit()
        res[name] = {k: {"log_or": float(fit.params[k]), "ci95": fit.conf_int().loc[k].tolist(), "p": float(fit.pvalues[k])}
                     for k in fit.params.index if k != "Intercept"}
    return {**res, **extra}

def x_mismatch_agreement(preds_runs: list) -> dict:
    rp, ch, y, w, yr, mj = x_cev_reader()
    mm = (mj >= 0) & (mj != yr)
    if not mm.any():
        return {}
    aw = [float(np.mean(p[mm] == yr[mm])) for p in preds_runs]; am = [float(np.mean(p[mm] == mj[mm])) for p in preds_runs]
    return {"n_texts": int(mm.sum()), "agree_writer_mean": float(np.mean(aw)), "agree_reader_majority_mean": float(np.mean(am)),
            "agree_writer_range": [min(aw), max(aw)], "agree_reader_majority_range": [min(am), max(am)]}

def x_e3_reader_version(df) -> dict:
    src = x_runs(df, "cev_source")
    if not len(src) or "reader_consensus" not in CEV:
        return {}
    rp, ch, y, w, yr, mj = x_cev_reader()
    orig, masked = [], []
    for _, r in src.iterrows():
        with robust_npz(r) as z:
            orig.append(cev_restrict(z["cev_orig_logits"])[rp]); masked.append(cev_restrict(z["cev_masked_logits"])[rp])
    return {"n_runs": len(src),
            "original_text": x_e3_gee([p == yr for p in orig], {"acc_pct": float(100 * np.mean([np.mean(p == yr) for p in orig]))}),
            "masked_text_reader_version": x_e3_gee([p == yr for p in masked], {"acc_pct": float(100 * np.mean([np.mean(p == yr) for p in masked]))}),
            "perspective_mismatch_masked_text": x_mismatch_agreement(masked),
            "corr_reader_acc_consensus": float(np.corrcoef(CEV["reader_item_acc"], CEV["reader_consensus"])[0, 1]),
            "_note": "Reader acc = share of 5 readers choosing writer's label; consensus = share choosing modal label (13 classes). Exploratory."}

def x_llm_transfer(df) -> dict:
    if f"{EXPERIMENT_NAME}/llm_transfer/manifest.json" not in HUB_FILES and not (LOCAL_ROOT / f"{EXPERIMENT_NAME}/llm_transfer/manifest.json").exists():
        return {}
    rng = np.random.default_rng(X_SEED + 1)
    exkey = {"goemotions": "goemotions", "meld": "meld", "isear": "isear_target_test", "ed": "ed_test"}
    out = {}
    for tgt in ["goemotions", "meld", "isear", "ed"]:
        d = x_llm_load(tgt).sort_values("i")
        ex = np.asarray(EXPLICIT[exkey[tgt]]); assert len(d) == len(ex), f"llm_transfer {tgt}: {len(d)} rows, expected {len(ex)}"
        c = (d.pred.values == d.label.values)
        out[tgt] = {"n": int(len(d)), "acc_pct": float(100 * c.mean()), "format_failure_pct": float(100 * (d.pred.values < 0).mean()),
                    "implicit_gap": x_boot_gap(c, ex, rng), "models_reported": sorted(d.model_reported.unique().tolist())}
    o = x_llm_load("cev_orig").sort_values("i"); m = x_llm_load("cev_masked").sort_values("i")
    rp, ch, y, w, yr, mj = x_cev_reader()
    po = o.pred.values.copy(); pm = po.copy(); pm[np.flatnonzero(ch)] = m.pred.values
    assert len(po) == len(y) and len(m) == int(ch.sum())
    drop = lambda idx: 100 * (np.mean(po[idx] == y[idx]) - np.mean(pm[idx] == y[idx]))
    ci_idx = np.flatnonzero(ch); b = [drop(rng.choice(ci_idx, len(ci_idx))) for _ in range(X_N_BOOT)]
    out["crowd_envent"] = {"acc_orig_all_pct": float(100 * np.mean(po == y)), "drop863_pp": float(drop(ci_idx)),
                           "drop863_ci95_pp": [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))],
                           "drop163_pp": float(drop(rp[w])),
                           "reader_set_acc_orig_pct": float(100 * np.mean(po[rp] == yr)),
                           "reader_set_acc_masked_pct": float(100 * np.mean(pm[rp] == yr)),
                           "reader_set_agree_majority_masked_pct": float(100 * np.mean(pm[rp][mj >= 0] == mj[mj >= 0])),
                           "e3_masked_text": x_e3_gee([pm[rp] == yr], {}), "perspective_mismatch": x_mismatch_agreement([pm[rp]])}
    out["_note"] = "Zero-shot gpt-4o-2024-08-06, temp 0, target-restricted label list, strict parser (failures = errors). ISEAR 650 items."
    return out

X_RESULTS = {}
_xdf = robust_train_df()
for name, fn in [("x_e1_third", x_e1_third), ("x_second_source", x_second_source),
                 ("x_e3_reader_version", x_e3_reader_version), ("x_llm_transfer", x_llm_transfer)]:
    # v8: x5_setfit_vs_goemotions_gap, x6_reader_baselines, x7_prefix_transfer, x8_explicit_validation and x9_relative_drops
    # were listed here without definitions (NameError) and were never run; no number in the paper comes from them.
    try:
        res = fn(_xdf)
    except Exception as e:
        logger.error(f"{name} failed: {type(e).__name__}: {e}")
        raise
    if res:
        X_RESULTS[name] = res
        write_json(ANALYSIS_DIR / "metrics" / f"{name}.json", res)
        logger.info(f"{name}: written")
    else:
        logger.info(f"{name}: inputs not available yet, skipped")
if "x_e1_third" in X_RESULTS:
    r = X_RESULTS["x_e1_third"]
    for k in ["diff_setfit_minus_goemotions_163", "diff_goemotions_minus_cev_163"]:
        if r.get(k):
            logger.info(f"X1 {k}: {r[k]['mu']:+.2f} pp [{r[k]['ci95'][0]:+.2f}, {r[k]['ci95'][1]:+.2f}]")
    logger.info(f"X1 rerun check vs target stage: {r['rerun_check_vs_target_stage']}")

[2026-10-07 14:05:01] [INFO] x_e1_third: written
[2026-10-07 14:05:04] [INFO] x_second_source: written
[2026-10-07 14:05:05] [INFO] x_e3_reader_version: written
[2026-10-07 14:05:05] [INFO] x_llm_transfer: written
[2026-10-07 14:05:05] [INFO] X1 diff_setfit_minus_goemotions_163: -34.79 pp [-38.36, -31.21]
[2026-10-07 14:05:05] [INFO] X1 diff_goemotions_minus_cev_163: +22.73 pp [+20.00, +25.45]
[2026-10-07 14:05:05] [INFO] X1 rerun check vs target stage: {'pairs': 25, 'equal_test_acc': 22}


## Cell 9e — Reader perspective: crowd-enVENT and enISEAR (v8)

In [ ]:
# =====================================================================================
# CELL 9e — READER PERSPECTIVE: crowd-enVENT AND enISEAR (v8, reader-perspective revision; analysis only, trains nothing)
# Quantities (paper, Section 3.2): among texts whose reader majority is unique and lies within the classes of the corpus,
# R = majority equals the writer's label and Rbar = majority differs from it. Per run, on the version the readers saw:
# accuracy against the writer's label on R and on Rbar, the share pi of its errors (over R and Rbar) that fall on Rbar,
# and on Rbar the shares of predictions equal to the reader majority and to the writer's label.
# P0 (exploratory) crowd-enVENT: SetFit-trained (cev_source), GoEmotions-trained (cev_third), crowd-enVENT-trained
#    (cev_target), GPT-4o (llm_transfer chunks), masked version, five shared classes. Complements X4 of Cell 9d.
# enISEAR inputs: eni_source and eni_third runs (eni_logits) and the GPT-4o task llm_transfer/enisear (Cell 8c).
# P1 (confirmatory, Amendment 2) enISEAR: classifier correctness increases with reader item accuracy
#    (GEE, binomial, exchangeable, clustered by item; one-sided p).
# P2 (confirmatory, Amendment 2) enISEAR: on Rbar, classifiers choose the reader majority more often than the writer's
#    label (difference of shares; joint bootstrap over Rbar items and runs; one-sided p).
# Outputs: analysis/metrics/perspective_crowd_envent.json, perspective_enisear.json, perspective_numbers.json (values for
# the paper's \NN keys) and analysis/tables/perspective.tex, supp_enisear.tex (complete tabulars). Every part is skipped
# with a log message when its inputs are not yet on the Hub.
# =====================================================================================
def p_bootstrap(P: np.ndarray, y: np.ndarray, maj: np.ndarray, rng) -> dict:
    """P: (runs, items) predictions. Joint bootstrap over runs and items, items resampled within R and within Rbar so
    that both set sizes stay fixed. With one run, only items are resampled."""
    ok = maj >= 0
    iR, iB = np.flatnonzero(ok & (maj == y)), np.flatnonzero(ok & (maj != y))
    if not len(iR) or not len(iB):
        return {}
    boots = []
    for _ in range(P_N_BOOT):
        rr = P[rng.integers(0, len(P), len(P))] if len(P) > 1 else P
        bR, bB = rng.choice(iR, len(iR)), rng.choice(iB, len(iB))
        eR, eB = rr[:, bR] != y[bR], rr[:, bB] != y[bB]
        boots.append([1 - eR.mean(), 1 - eB.mean(), eB.sum(1).mean() / max((eR.sum(1) + eB.sum(1)).mean(), 1e-12),
                      np.mean(rr[:, bB] == maj[bB]), np.mean(rr[:, bB] == y[bB])])
    b = np.asarray(boots)
    ci = lambda j: [float(np.percentile(b[:, j], 2.5)), float(np.percentile(b[:, j], 97.5))]
    diff = b[:, 3] - b[:, 4]                                       # P2 statistic: chose readers minus chose writer on Rbar
    return {"acc_R_ci95": ci(0), "acc_Rbar_ci95": ci(1), "pi_ci95": ci(2), "rbar_chose_readers_ci95": ci(3),
            "rbar_chose_writer_ci95": ci(4), "readers_minus_writer_ci95": [float(np.percentile(diff, 2.5)), float(np.percentile(diff, 97.5))],
            "p_readers_gt_writer_one_sided": float(max(1.0 / P_N_BOOT, np.mean(diff <= 0)))}

def p_family(rows: list, y: np.ndarray, maj: np.ndarray, rng) -> dict:
    """rows: list of (spec-like dict, predictions on the reader texts)."""
    if not rows:
        return {}
    P = np.stack([p for _, p in rows])
    per = pd.DataFrame([{"encoder": r.get("encoder"), "condition": r.get("condition"), "seed": r.get("seed"),
                         **{k: v for k, v in perspective_metrics(p, y, maj).items() if not isinstance(v, list)}} for r, p in rows])
    keys = ["acc_writer", "agree_reader_majority", "acc_R", "acc_Rbar", "pi_error_share_Rbar", "rbar_chose_readers", "rbar_chose_writer"]
    cell = per.groupby(["encoder", "condition"])[keys].mean()
    return {"n_runs": int(len(P)), "mean": per[keys].mean().to_dict(),
            "range_of_encoder_condition_means": {k: [float(cell[k].min()), float(cell[k].max())] for k in keys},
            "by_encoder_condition": cell.reset_index().to_dict("records"), **p_bootstrap(P, y, maj, rng)}

def p_gee(rows: list, y: np.ndarray, item_acc: np.ndarray, extra: dict = None) -> dict:
    """P1: correct ~ z(reader item accuracy) [+ extra item covariates], GEE clustered by item."""
    if not rows:
        return {}
    z_acc = (item_acc - item_acc.mean()) / (item_acc.std(ddof=1) + 1e-12)
    parts = [pd.DataFrame({"correct": (p == y).astype(int), "item": np.arange(len(y)), "z_reader_acc": z_acc,
                           **(extra or {})}) for _, p in rows]
    d = pd.concat(parts, ignore_index=True)
    formula = "correct ~ z_reader_acc" + "".join(f" + {k}" for k in (extra or {}))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = smf.gee(formula, groups="item", data=d, family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()).fit()
    b, ci = float(fit.params["z_reader_acc"]), fit.conf_int().loc["z_reader_acc"].tolist()
    return {"log_or_per_sd": b, "ci95": [float(ci[0]), float(ci[1])], "p_two_sided": float(fit.pvalues["z_reader_acc"]),
            "p_one_sided": float(sps.norm.sf(fit.tvalues["z_reader_acc"])), "n_rows": int(len(d)), "n_runs": int(len(rows))}

def p_llm_rows(task: str):
    files = sorted(f for f in HUB_FILES if f.startswith(f"{LLMT_PREFIX}/{task}/") and f.endswith(".jsonl"))
    rows = []
    for f in files:
        lp = LOCAL_ROOT / f
        rows += [json.loads(l) for l in open(lp if lp.exists() else hub_download(f)) if l.strip()]
    rows.sort(key=lambda r: r["i"])
    return rows

def p_runs(df: pd.DataFrame, stage: str, key: str, select=None) -> list:
    out = []
    for _, r in df[df.stage == stage].iterrows():
        with robust_npz(r) as z:
            if key not in z.files:
                continue
            lg = z[key]
        out.append((r.to_dict(), lg if select is None else select(lg)))
    return out

def perspective_crowd_envent(df: pd.DataFrame) -> dict:
    rng = np.random.default_rng(P_SEED)
    rp, yr, mj = CEV["reader_pos"], CEV["label"][CEV["reader_pos"]], CEV["reader_majority"]
    cls = PRESENT_CLASSES["cev"]
    fam = {"setfit_trained": [(r, restrict_to(lg, cls)[rp]) for r, lg in p_runs(df, "cev_source", "cev_masked_logits")],
           "goemotions_trained": [(r, restrict_to(lg, cls)[rp]) for r, lg in p_runs(df, "cev_third", "cev_masked_logits")],
           "crowd_envent_trained": [(r, restrict_to(lg, cls)) for r, lg in p_runs(df, "cev_target", "cevtest_masked_logits")]}
    lo, lm = p_llm_rows("cev_orig"), p_llm_rows("cev_masked")
    if len(lo) == len(CEV["label"]) and len(lm) == int(CEV["changed"].sum()):
        assert np.array_equal([r["label"] for r in lo], CEV["label"]), "llm_transfer cev_orig rows misaligned"
        assert np.array_equal([r["label"] for r in lm], CEV["label"][CEV["changed"]]), "llm_transfer cev_masked rows misaligned"
        p = np.array([r["pred"] for r in lo]); p[np.flatnonzero(CEV["changed"])] = [r["pred"] for r in lm]
        fam["gpt4o"] = [({"encoder": "gpt-4o-2024-08-06", "condition": "zero-shot", "seed": 0}, p[rp])]
    else:
        logger.info("P0: GPT-4o crowd-enVENT chunks (llm_transfer) not found; GPT-4o row skipped")
    ok = mj >= 0
    out = {"n_reader_texts": int(len(rp)), "n_unique_majority": int(ok.sum()), "n_R": int((ok & (mj == yr)).sum()),
           "n_Rbar": int((ok & (mj != yr)).sum()), "share_Rbar": float((ok & (mj != yr)).sum() / ok.sum()),
           "reader_majority_recovers_writer": float((ok & (mj == yr)).sum() / ok.sum()),
           "families": {k: p_family(v, yr, mj, rng) for k, v in fam.items() if v},
           "_note": "exploratory (P0); masked version, the version the readers saw; five shared classes"}
    if fam["setfit_trained"]:                              # check against the analysis already in the paper (masked text, 77 texts)
        s = out["families"]["setfit_trained"]["mean"]
        logger.info(f"P0 check: SetFit-trained masked accuracy {100 * s['acc_writer']:.2f}% (paper 46.44%), on Rbar chose readers "
                    f"{100 * s['rbar_chose_readers']:.1f}% and writer {100 * s['rbar_chose_writer']:.1f}% (paper 48.2% and 22.7%)")
    return out

def perspective_enisear(df: pd.DataFrame) -> dict:
    rng = np.random.default_rng(P_SEED + 1)
    y, mj, ia = ENI["label"], ENI["reader_majority"], ENI["reader_item_acc"]
    cls = PRESENT_CLASSES["eni"]
    ok = mj >= 0
    out = {"n_texts": int(len(y)), "n_unicode_marker_normalized": int(ENI["n_unicode_marker"]),
           "n_tied_majority": int((mj == -1).sum()), "n_majority_outside_four": int((mj == -2).sum()),
           "n_unique_majority": int(ok.sum()), "n_R": int((ok & (mj == y)).sum()), "n_Rbar": int((ok & (mj != y)).sum()),
           "share_Rbar": float((ok & (mj != y)).sum() / ok.sum()),
           "reader_majority_recovers_writer": float((ok & (mj == y)).sum() / ok.sum()),
           "individual_reader_acc_7way": float(ia.mean()), "families": {}, "P1": {}, "P2": {}}
    # in-set reader accuracy: votes for the writer's label over all votes that fall within the four shared classes
    col = {e: j for j, e in enumerate(ENI_EMOTIONS)}
    writer_votes = ENI["reader_votes"][np.arange(len(y)), [col[SETFIT[c].capitalize()] for c in y]]
    out["individual_reader_acc_inset"] = float(writer_votes.sum() / ENI["reader_votes"][:, [col[SETFIT[c].capitalize()] for c in cls]].sum())
    fam = {"setfit_trained": [(r, restrict_to(lg, cls)) for r, lg in p_runs(df, "eni_source", "eni_logits")],
           "goemotions_trained": [(r, restrict_to(lg, cls)) for r, lg in p_runs(df, "eni_third", "eni_logits")]}
    rows = p_llm_rows("enisear")                       # GPT-4o, zero-shot, Cell 8c (task "enisear")
    if len(rows) == len(y) and [q["i"] for q in rows] == list(range(len(y))):
        assert np.array_equal([q["label"] for q in rows], y), "llm_transfer enisear rows misaligned with enISEAR labels"
        fam["gpt4o"] = [({"encoder": LLMT_MODEL, "condition": "zero-shot", "seed": 0}, np.array([q["pred"] for q in rows]))]
        out["gpt4o_format_failure_rate"] = float(np.mean(fam["gpt4o"][0][1] == -1))
        out["gpt4o_models_reported"] = sorted({q["model_reported"] for q in rows})
    fam = {k: v for k, v in fam.items() if v}
    if not fam:
        logger.info("enISEAR: no eni_* runs evaluated yet; descriptive reader statistics only")
        return out
    for k, rows in fam.items():
        out["families"][k] = p_family(rows, y, mj, rng)
        out["P1"][k] = p_gee(rows, y, ia)
        f = out["families"][k]
        out["P2"][k] = {"n_Rbar": out["n_Rbar"], "chose_readers": f["mean"]["rbar_chose_readers"], "chose_writer": f["mean"]["rbar_chose_writer"],
                        "difference_ci95": f["readers_minus_writer_ci95"], "p_one_sided": f["p_readers_gt_writer_one_sided"]}
        out["families"][k]["sadness_prediction_share"] = float(np.mean(np.stack([p for _, p in rows]) == 0))
    out["true_sadness_share"] = float(np.mean(y == 0))
    out["_note"] = ("P1 and P2 are confirmatory (Amendment 2; directions: P1 positive, P2 readers > writer). The release contains "
                    "only the versions the readers saw; '…' was normalized to '...'. GoEmotions-trained models are known to "
                    "treat the three-dot marker as a sadness cue on crowd-enVENT: compare sadness_prediction_share with true_sadness_share.")
    return out

def p_fmt(v, nd=1, pct=True):
    return "--" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{(100 * v if pct else v):.{nd}f}"

def perspective_tables(pc: dict, pe: dict, TEX: Path) -> None:
    labels = [("setfit_trained", "SetFit-trained"), ("goemotions_trained", "GoEmotions-trained"),
              ("crowd_envent_trained", "crowd-enVENT-trained"), ("gpt4o", "GPT-4o, zero-shot")]
    lines = [r"\begin{tabular}{lcccccc}", r"\toprule",
             r" & \multicolumn{3}{c}{crowd-enVENT} & \multicolumn{3}{c}{enISEAR} \\",
             r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
             r"Classifier & $\mathcal{R}$ & $\bar{\mathcal{R}}$ & $\pi$ & $\mathcal{R}$ & $\bar{\mathcal{R}}$ & $\pi$ \\", r"\midrule"]
    for key, name in labels:
        cells = []
        for res in (pc, pe):
            m = res.get("families", {}).get(key, {}).get("mean")
            cells += [p_fmt(m["acc_R"]), p_fmt(m["acc_Rbar"]), p_fmt(m["pi_error_share_Rbar"])] if m else ["--", "--", "--"]
        lines.append(f"{name} & " + " & ".join(cells) + r" \\")
    lines += [r"\midrule", r"Share of $\bar{\mathcal{R}}$ texts & & & " + p_fmt(pc.get("share_Rbar")) + " & & & " + p_fmt(pe.get("share_Rbar")) + r" \\",
              r"\bottomrule", r"\end{tabular}"]
    (TEX / "perspective.tex").write_text("\n".join(lines) + "\n")
    rows = [r"\begin{tabular}{lcc}", r"\toprule", r"Classifier or reader & Writer & Majority \\", r"\midrule"]
    st = pe.get("families", {}).get("setfit_trained")
    names = {"albert": "ALBERT", "distilbert": "DistilBERT", "distilroberta": "DistilRoBERTa", "electra": "ELECTRA", "roberta": "RoBERTa"}
    rng_ = lambda vals: f"{p_fmt(min(vals))}--{p_fmt(max(vals))}"
    if st:
        cell = pd.DataFrame(st["by_encoder_condition"])
        for enc, g in cell.groupby("encoder"):
            rows.append(f"{names.get(enc, enc)} & {rng_(g.acc_writer)} & {rng_(g.agree_reader_majority)}" + r" \\")
    gt = pe.get("families", {}).get("goemotions_trained")
    if gt:
        cell = pd.DataFrame(gt["by_encoder_condition"])
        rows.append(f"GoEmotions-trained & {rng_(cell.acc_writer)} & {rng_(cell.agree_reader_majority)}" + r" \\")
    g4 = pe.get("families", {}).get("gpt4o")
    if g4:
        rows.append(f"GPT-4o, zero-shot & {p_fmt(g4['mean']['acc_writer'])} & {p_fmt(g4['mean']['agree_reader_majority'])}" + r" \\")
    rows += [f"Individual reader, seven classes & {p_fmt(pe['individual_reader_acc_7way'])} & n/a" + r" \\",
             f"Individual reader, four classes$^{{a}}$ & {p_fmt(pe['individual_reader_acc_inset'])} & n/a" + r" \\",
             f"Reader majority & {p_fmt(pe['reader_majority_recovers_writer'])} & n/a" + r" \\", r"\bottomrule", r"\end{tabular}"]
    (TEX / "supp_enisear.tex").write_text("\n".join(rows) + "\n")

def perspective_numbers(pc: dict, pe: dict) -> dict:
    """Values for the \\NN{key}{value} placeholders of the paper; add them to the numbers script under the same keys."""
    N, ci = {}, lambda v, nd=1, pct=True: f"[{p_fmt(v[0], nd, pct)}, {p_fmt(v[1], nd, pct)}]"
    N["e3-majrec"], N["e3-share-rbar"] = p_fmt(pc["reader_majority_recovers_writer"]), p_fmt(pc["share_Rbar"])
    s = pc["families"].get("setfit_trained")
    if s:
        N.update({"p0-cev-src-r": p_fmt(s["mean"]["acc_R"]), "p0-cev-src-r-ci": ci(s["acc_R_ci95"]),
                  "p0-cev-src-rbar": p_fmt(s["mean"]["acc_Rbar"]), "p0-cev-src-rbar-ci": ci(s["acc_Rbar_ci95"]),
                  "p0-cev-src-pi": p_fmt(s["mean"]["pi_error_share_Rbar"]), "p0-cev-src-pi-ci": ci(s["pi_ci95"])})
    for k, tag in (("goemotions_trained", "goe"), ("crowd_envent_trained", "cev"), ("gpt4o", "llm")):
        f = pc["families"].get(k)
        if f:
            N[f"p0-cev-{tag}-pi"], N[f"p0-cev-{tag}-pi-ci"] = p_fmt(f["mean"]["pi_error_share_Rbar"]), ci(f["pi_ci95"])
    N.update({"size-enisear-all": "1{,}001", "size-enisear-per-class": "143", "size-enisear-four": str(pe["n_texts"]),
              "size-enisear-ties": str(pe["n_tied_majority"]), "size-enisear-outside": str(pe["n_majority_outside_four"]),
              "enisear-nmaj": str(pe["n_unique_majority"]), "enisear-majrec": p_fmt(pe["reader_majority_recovers_writer"]),
              "enisear-share-rbar": p_fmt(pe["share_Rbar"]), "p2-n": str(pe["n_Rbar"]),
              "enisear-hind": p_fmt(pe["individual_reader_acc_7way"]), "enisear-hinset": p_fmt(pe["individual_reader_acc_inset"]),
              "enisear-marker": str(pe["n_unicode_marker_normalized"])})
    gs = pe.get("families", {}).get("goemotions_trained")
    if gs:                                                  # three-dot marker check, as for crowd-enVENT (true share: 25%)
        N["x1-enisear-goe-sad"] = p_fmt(gs["sadness_prediction_share"])
    s = pe.get("families", {}).get("setfit_trained")
    if s:
        r = s["range_of_encoder_condition_means"]["acc_writer"]
        N.update({"enisear-src-min": p_fmt(r[0]), "enisear-src-max": p_fmt(r[1]), "p0-enisear-src-pi": p_fmt(s["mean"]["pi_error_share_Rbar"]),
                  "p1-gee": p_fmt(pe["P1"]["setfit_trained"]["log_or_per_sd"], 2, False), "p1-gee-ci": ci(pe["P1"]["setfit_trained"]["ci95"], 2, False),
                  "p1-gee-p": f"{pe['P1']['setfit_trained']['p_one_sided']:.2g}",
                  "p2-r": p_fmt(pe["P2"]["setfit_trained"]["chose_readers"]), "p2-w": p_fmt(pe["P2"]["setfit_trained"]["chose_writer"]),
                  "p2-p": f"{pe['P2']['setfit_trained']['p_one_sided']:.2g}"})
    g = pe.get("families", {}).get("gpt4o")
    if g:
        N.update({"enisear-gpt4o": p_fmt(g["mean"]["acc_writer"]), "p1-gee-gpt4o": p_fmt(pe["P1"]["gpt4o"]["log_or_per_sd"], 2, False),
                  "p1-gee-gpt4o-ci": ci(pe["P1"]["gpt4o"]["ci95"], 2, False),
                  "p2-gpt4o-r": p_fmt(pe["P2"]["gpt4o"]["chose_readers"]), "p2-gpt4o-w": p_fmt(pe["P2"]["gpt4o"]["chose_writer"])})
    return N

PERSPECTIVE = {}
_pdf = robust_train_df()
if len(_pdf):
    PERSPECTIVE["perspective_crowd_envent"] = perspective_crowd_envent(_pdf)
    PERSPECTIVE["perspective_enisear"] = perspective_enisear(_pdf)
    PERSPECTIVE["perspective_numbers"] = perspective_numbers(PERSPECTIVE["perspective_crowd_envent"], PERSPECTIVE["perspective_enisear"])
    (ANALYSIS_DIR / "tables").mkdir(parents=True, exist_ok=True)
    perspective_tables(PERSPECTIVE["perspective_crowd_envent"], PERSPECTIVE["perspective_enisear"], ANALYSIS_DIR / "tables")
    for name, obj in PERSPECTIVE.items():
        write_json(ANALYSIS_DIR / "metrics" / f"{name}.json", obj)
    for k, f in PERSPECTIVE["perspective_crowd_envent"]["families"].items():
        logger.info(f"P0 crowd-enVENT {k}: acc R {100 * f['mean']['acc_R']:.1f}%, Rbar {100 * f['mean']['acc_Rbar']:.1f}%, "
                    f"pi {100 * f['mean']['pi_error_share_Rbar']:.1f}% {[round(100 * x, 1) for x in f['pi_ci95']]} "
                    f"(share of Rbar texts {100 * PERSPECTIVE['perspective_crowd_envent']['share_Rbar']:.1f}%)")
    for k, v in PERSPECTIVE["perspective_enisear"].get("P2", {}).items():
        p1 = PERSPECTIVE["perspective_enisear"]["P1"][k]
        logger.info(f"enISEAR {k}: P1 log OR {p1['log_or_per_sd']:+.2f} {[round(x, 2) for x in p1['ci95']]} (one-sided p {p1['p_one_sided']:.2g}); "
                    f"P2 on {v['n_Rbar']} texts readers {100 * v['chose_readers']:.1f}% vs writer {100 * v['chose_writer']:.1f}% "
                    f"(one-sided p {v['p_one_sided']:.2g})")
logger.info(f"Reader-perspective analyses complete: {sorted(PERSPECTIVE)}")

## Cell 10 — Visualization

In [38]:
# =====================================================================================
# CELL 10 — VISUALIZATION (GodMode visualization standard)
# DejaVu Serif, no titles, no bold, full border, no grid, tab10, axis labels 13 pt, ticks 11 pt, PNG at dpi=600.
# Each figure is produced only when its analysis exists; missing inputs are logged and skipped.
# =====================================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"font.family": "DejaVu Serif", "font.weight": "normal", "axes.labelweight": "normal",
                            "axes.titleweight": "normal"})
COLORS = plt.cm.tab10.colors
FIG_DIR = ANALYSIS_DIR / "figures"
FIGURES = []

def style(ax, xlabel: str, ylabel: str):
    ax.set_xlabel(xlabel, fontsize=13, fontweight="normal"); ax.set_ylabel(ylabel, fontsize=13, fontweight="normal")
    ax.tick_params(labelsize=11, pad=4)
    for sp in ax.spines.values():
        sp.set_visible(True)
    ax.grid(False)

def panel(ax, label: str):
    ax.text(-0.08, 1.02, label, transform=ax.transAxes, fontsize=12, fontweight="normal", va="bottom", ha="right")

def save(fig, name: str):
    path = FIG_DIR / name
    plt.tight_layout(pad=1.5)
    fig.savefig(path, dpi=600, bbox_inches="tight", format="png")
    plt.close(fig)
    assert path.exists() and path.stat().st_size > 0, f"Export failed: {path}"
    FIGURES.append(path); logger.info(f"Figure saved: {path.name}")

def generate_figures(A: dict, df: pd.DataFrame):
    if "condition" not in df or "stage" not in df:              # no training runs evaluated yet
        df = pd.DataFrame(columns=["stage", "condition", "encoder", "seed", "test_acc", "cri", "calibration", "test_confusion"])
    df = df[df["kind"].fillna("train") == "train"] if "kind" in df else df
    # (1) H1 explicit share per corpus
    h1 = A.get("H1_explicit_share")
    if h1:
        names = ["setfit_test", "isear", "goemotions", "ed_test", "tweeteval_test", "meld"]
        names = [n for n in names if n in h1]
        fig, ax = plt.subplots(figsize=(6, 5))
        v = [100 * h1[n]["share"] for n in names]
        err = np.array([[100 * (h1[n]["share"] - h1[n]["wilson95"][0]) for n in names],
                        [100 * (h1[n]["wilson95"][1] - h1[n]["share"]) for n in names]])
        ax.bar(range(len(names)), v, yerr=err, color=[COLORS[i] for i in range(len(names))], alpha=0.85, capsize=3)
        ax.set_xticks(range(len(names))); ax.set_xticklabels([n.replace("_test", "") for n in names], rotation=30, ha="right")
        ax.set_ylim(0, 100); style(ax, "Corpus", "Explicit items (%)")
        save(fig, "fig_explicit_share.png")
    # (2) H2 explicit vs implicit transfer accuracy
    h2 = A.get("H2_stratified_transfer")
    if h2:
        t = list(h2)
        fig, ax = plt.subplots(figsize=(6, 5)); x = np.arange(len(t)); w = 0.38
        ax.bar(x - w / 2, [100 * h2[k]["acc_explicit"] for k in t], w, color=COLORS[0], label="Explicit items")
        ax.bar(x + w / 2, [100 * h2[k]["acc_implicit"] for k in t], w, color=COLORS[1], label="Implicit items")
        ax.set_xticks(x); ax.set_xticklabels(t); ax.legend(fontsize=11); style(ax, "Target corpus", "Transfer accuracy (%)")
        save(fig, "fig_stratified_transfer.png")
    # (3) H3 CRI vs transfer accuracy, one panel per target
    if len(df) and "cri" in df and A.get("H3_cri_vs_transfer"):
        tg = [t for t in TRANSFER_TARGETS if t in A["H3_cri_vs_transfer"]]
        fig, axes = plt.subplots(1, len(tg), figsize=(5 * len(tg), 5), squeeze=False)
        encs = sorted(df.encoder.unique())
        for j, (ax, t) in enumerate(zip(axes[0], tg)):
            c = df[df.stage.isin(["main", "shortcut"]) & df.cri.notna() & df[f"transfer_{t}"].notna()]
            for i, e in enumerate(encs):
                s = c[c.encoder == e]
                if len(s):
                    ax.scatter(s.cri, [100 * d["restricted"]["acc"] for d in s[f"transfer_{t}"]], s=14, color=COLORS[i % 10],
                               alpha=0.75, label=e, marker="o")
            style(ax, "Cue-reliance index", f"{t} accuracy (%)"); ax.xaxis.set_major_locator(plt.MaxNLocator(6))
            panel(ax, f"({chr(97 + j)})")
        axes[0][0].legend(fontsize=9)
        save(fig, "fig_cri_vs_transfer.png")
    # (4) seed-paired prompt effect (A vs none) per encoder with random-effects pooled estimate
    sp = pd.DataFrame(A.get("seed_paired", []))
    if len(sp):
        sp = sp[(sp.condition == "A") & sp.stage.isin(["main", "intra"]) & (sp.text_transform == "none")]
    if len(sp):
        corp = sorted(sp.corpus.unique())
        fig, axes = plt.subplots(1, len(corp), figsize=(5 * len(corp), 5), squeeze=False)
        for j, (ax, cp) in enumerate(zip(axes[0], corp)):
            s = sp[sp.corpus == cp].sort_values("encoder").reset_index(drop=True)
            for i, r in s.iterrows():
                ax.plot(r.ci95, [i, i], color=COLORS[0], lw=1.5); ax.plot(r["mean"], i, "o", color=COLORS[0], ms=5)
            key = next((k for k in A.get("random_effects", {}) if k.startswith(f"{s.stage.iloc[0]}|{cp}|A|")), None)
            labels = list(s.encoder)
            if key:
                re_ = A["random_effects"][key]; i = len(s)
                ax.plot(re_["ci95"], [i, i], color=COLORS[3], lw=1.5); ax.plot(re_["mu"], i, "o", color=COLORS[3], ms=6)
                labels.append("Random-effects pooled")
            ax.axvline(0, color="0.3", lw=0.8, ls="--")
            ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels); ax.invert_yaxis()
            style(ax, "Accuracy difference, A minus none (pp)", ""); panel(ax, f"({chr(97 + j)})")
        save(fig, "fig_prompt_effects.png")
    # (5) multiseed summary: main-grid accuracy per encoder without prompt (mean +- SD)
    if len(df):
        m = df[(df.stage == "main") & (df.condition == "none")]
        if len(m):
            g = m.groupby("encoder").test_acc.agg(["mean", "std", "count"]).reset_index()
            fig, ax = plt.subplots(figsize=(6, 5))
            ax.bar(range(len(g)), 100 * g["mean"], yerr=100 * g["std"], color=[COLORS[i] for i in range(len(g))], alpha=0.85, capsize=3, width=0.6)
            ax.set_xticks(range(len(g))); ax.set_xticklabels(g.encoder, rotation=30, ha="right")
            lo = 100 * (g["mean"] - g["std"]).min(); ax.set_ylim(max(0, lo - 2), min(100, 100 * (g["mean"] + g["std"]).max() + 1))
            style(ax, "Encoder", "Test accuracy (%)")
            save(fig, "multiseed_summary.png")
    # (6) convergence rate
    cv = A.get("convergence")
    if cv:
        fig, ax = plt.subplots(figsize=(6, 5)); x = np.arange(len(cv)); w = 0.38
        for j, (key, lab) in enumerate([("converged_none", "No prompt"), ("converged_A", "Template A")]):
            rates = [cv[e][key][0] / cv[e][key][1] for e in cv]
            ci = [wilson(cv[e][key][0], cv[e][key][1]) for e in cv]
            err = np.array([[100 * (r - c[0]) for r, c in zip(rates, ci)], [100 * (c[1] - r) for r, c in zip(rates, ci)]])
            ax.bar(x + (j - 0.5) * w, [100 * r for r in rates], w, yerr=err, color=COLORS[j], capsize=3, label=lab)
        ax.set_xticks(x); ax.set_xticklabels(list(cv)); ax.set_ylim(0, 105); ax.legend(fontsize=11)
        style(ax, "Encoder", "Converged runs (%)")
        save(fig, "fig_convergence.png")

def calibration_and_confusion(df: pd.DataFrame):
    """Reliability diagram and row-normalized confusion matrix for RoBERTa (or the first encoder), template A, lowest seed."""
    if "condition" not in df or "stage" not in df:              # no training runs evaluated yet
        df = pd.DataFrame(columns=["stage", "condition", "encoder", "seed", "test_acc", "cri", "calibration", "test_confusion"])
    df = df[df["kind"].fillna("train") == "train"] if "kind" in df else df
    if not len(df) or "calibration" not in df:
        return
    m = df[(df.stage == "main") & (df.condition == "A") & df.calibration.notna()]
    if not len(m):
        return
    enc = "roberta" if "roberta" in set(m.encoder) else sorted(m.encoder)[0]
    r = m[m.encoder == enc].sort_values("seed").iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for j, (tag, lab) in enumerate([("before", "Before temperature scaling"), ("after", "After temperature scaling")]):
        ax = axes[j]
        bins = [b for b in r.calibration[tag]["reliability_bins"] if b["count"]]
        ax.bar([(b["lo"] + b["hi"]) / 2 for b in bins], [b["acc"] for b in bins], width=0.9 / CALIBRATION_BINS, color=COLORS[0], alpha=0.8, label="Accuracy")
        ax.plot([(b["lo"] + b["hi"]) / 2 for b in bins], [b["conf"] for b in bins], "o", color=COLORS[3], ms=4, label="Mean confidence")
        ax.plot([0, 1], [0, 1], "--", color="0.4", lw=1)
        ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.05); ax.legend(fontsize=10, loc="upper left")
        style(ax, "Confidence", "Accuracy"); panel(ax, f"({chr(97 + j)})")
        ax.text(0.98, 0.04, f"ECE {100 * r.calibration[tag]['ece']:.2f}%", transform=ax.transAxes, ha="right", fontsize=11)
    save(fig, "fig_reliability.png")
    cm = np.asarray(r.test_confusion, float); cm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(100 * cm, cmap="viridis", vmin=0, vmax=100)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{100 * cm[i, j]:.0f}", ha="center", va="center", fontsize=9, color="white" if cm[i, j] < 0.6 else "black")
    names = SETFIT[:cm.shape[0]]
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha="right"); ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    cbar = fig.colorbar(im, ax=ax, shrink=0.8, extend="neither"); cbar.set_label("Row share (%)", fontsize=11)
    style(ax, "Predicted class", "True class")
    save(fig, "fig_confusion.png")

generate_figures(ANALYSIS, METRICS_DF)
calibration_and_confusion(METRICS_DF)
logger.info(f"{len(FIGURES)} figures generated")


[2026-10-07 14:05:06] [INFO] Figure saved: fig_explicit_share.png
[2026-10-07 14:05:07] [INFO] Figure saved: fig_stratified_transfer.png
[2026-10-07 14:05:10] [INFO] Figure saved: fig_cri_vs_transfer.png
[2026-10-07 14:05:11] [INFO] Figure saved: fig_prompt_effects.png
[2026-10-07 14:05:12] [INFO] Figure saved: multiseed_summary.png
[2026-10-07 14:05:13] [INFO] Figure saved: fig_convergence.png
[2026-10-07 14:05:14] [INFO] Figure saved: fig_reliability.png
[2026-10-07 14:05:15] [INFO] Figure saved: fig_confusion.png
[2026-10-07 14:05:15] [INFO] 8 figures generated


## Cell 11 — Final artifact upload and completion

In [39]:
# =====================================================================================
# CELL 11 — FINAL ARTIFACT UPLOAD AND COMPLETION
# Uploads the aggregated analysis and figures, then closes the session. The completion signal is emitted only if
# every run of the stage is DONE and evaluated and every upload succeeded; otherwise the session is marked
# incomplete and rerunning the notebook with the same STAGE continues where it stopped.
# RESOLUTION: per-run final models are uploaded in Cell 8 (checkpoints/final/) for the runs selected in
# SAVE_FINAL_MODEL_FOR; storing a model for every run would exceed practical Hub storage for hundreds of runs,
# and every analysis uses the exported logits instead.
# =====================================================================================
def finalize_run() -> bool:
    hub_upload_folder(ANALYSIS_DIR, ANALYSIS_PREFIX, f"analysis after stage {STAGE} ({SESSION_TIMESTAMP}_{RUN_ID})")
    for f in FIGURES:
        assert f"{ANALYSIS_PREFIX}/figures/{f.name}" in HUB_FILES, f"figure not confirmed on the Hub: {f.name}"
    not_done = [s["spec_id"] for s in SPECS if RUN_STATUS.get(s["spec_id"]) != "done"]
    not_eval = [s["spec_id"] for s in SPECS if f"{run_prefix(s)}/metrics/metrics.json" not in HUB_FILES]
    complete = not not_done and not not_eval and not FAILED_RUNS
    SESSION_METADATA.update(timestamp_end=datetime.now().isoformat(), status="completed" if complete else "incomplete",
                            completed=complete, runs_total=len(SPECS), runs_not_done=not_done, runs_not_evaluated=not_eval,
                            failed_runs=FAILED_RUNS, figures=[f.name for f in FIGURES])
    write_json(session_dir / "metadata" / "run_metadata.json", SESSION_METADATA)
    hub_upload_file(session_dir / "metadata" / "run_metadata.json", f"{SESSION_PREFIX}/metadata/run_metadata.json")
    if not complete:
        logger.warning(f"Stage '{STAGE}' is not complete: {len(not_done)} runs not done, {len(not_eval)} not evaluated, "
                       f"{len(FAILED_RUNS)} failed this session. Rerun the notebook with the same STAGE to continue.")
        return False
    (session_dir / "DONE").write_text(datetime.now().isoformat())
    hub_upload_file(session_dir / "DONE", f"{SESSION_PREFIX}/DONE")
    logger.info(f"Run {RUN_ID} completed successfully. All artifacts persisted.")
    return True

STAGE_COMPLETE = finalize_run()
print(f"Stage '{STAGE}': {'COMPLETE' if STAGE_COMPLETE else 'INCOMPLETE - rerun to continue'} | runs {len(SPECS)} | "
      f"failed this session {len(FAILED_RUNS)} | figures {len(FIGURES)} | Hub repo {HF_REPO_ID}")
if STAGE == "tuning" and STAGE_COMPLETE:
    print("Tuned recipes:", json.dumps(ANALYSIS.get("tuned_recipes"), indent=1))


[2026-10-07 14:05:24] [INFO] Run 588316 completed successfully. All artifacts persisted.
Stage 'llm_transfer': COMPLETE | runs 0 | failed this session 0 | figures 8 | Hub repo hacky1997/emotion_explicit_cue
